# FastCCC on PFC data by case and age

This notebook runs FastCCC on one integrated `.h5ad` object by splitting cells by individual `case`, then summarizes age-associated communication changes across cases.

In [ ]:
import os
from pathlib import Path
import inspect
import warnings

import numpy as np
import pandas as pd
import scanpy as sc
import matplotlib.pyplot as plt
import seaborn as sns
import scipy.stats as st

import fastccc

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 200)
pd.set_option('display.width', 160)
sns.set(style='whitegrid', context='notebook')

print('Python executable:', os.sys.executable)
print('fastccc path:', fastccc.__file__)
print('FastCCC signature:')
print(inspect.signature(fastccc.statistical_analysis_method))


In [ ]:
# Paths
DATA = Path('/bigdata/disk1/HDD1-Data/aya/pfc.clean.h5ad')
DB_PATH = Path('/bigdata/disk1/HDD1-Data/aya/FastCCC-main/db/CPDBv5.0.0')
OUT = Path('/bigdata/disk1/HDD1-Data/aya/fastccc_pfc_jeffries_cpdbv5_newclusters_age')
TMP = OUT / 'tmp_inputs'
RESULTS = OUT / 'per_donor_results'

for p in [OUT, TMP, RESULTS]:
    p.mkdir(parents=True, exist_ok=True)

print('DATA   =', DATA)
print('DB     =', DB_PATH)
print('OUT    =', OUT)
print('RESULTS=', RESULTS)

assert DATA.exists(), f'Missing input: {DATA}'
assert DB_PATH.exists(), f'Missing FastCCC DB dir: {DB_PATH}'



In [ ]:
adata = sc.read_h5ad(DATA)
adata.obs_names_make_unique()
adata.var_names_make_unique()

print(adata)
print('\nobs columns:')
print(list(adata.obs.columns))
print('\nfirst 20 genes:')
print(adata.var_names[:20].tolist())


In [ ]:
# Check Jeffries annotation hierarchy.
for col, label in [
    ('new_clusters', 'finest annotation'),
    ('new_clusters2', 'intermediate annotation'),
    ('new_clusters3', 'broad annotation'),
]:
    assert col in adata.obs.columns, f'Missing {col}'
    vals = sorted(adata.obs[col].astype(str).dropna().unique())
    print(f'{col} ({label}): n={len(vals)}')
    print(vals)
    print()



In [ ]:
# Core metadata config
# Use donor-level id rather than sample-level case.
CASE_KEY = 'id'
AGE_KEY = 'age'
META_KEY = 'new_clusters'      # fine annotation for FastCCC and continuous analysis
BROAD_META_KEY = 'new_clusters3' # broad annotation from Jeffries h5ad
GROUPED_DISPLAY_META_KEY = 'new_clusters2' # display-only annotation for grouped FastCCC Fig.4-style LRIs
CONVERT_TYPE = 'hgnc_symbol'

for key in [CASE_KEY, AGE_KEY, META_KEY, BROAD_META_KEY, GROUPED_DISPLAY_META_KEY]:
    assert key in adata.obs.columns, f'Missing obs column: {key}'

print('CASE_KEY     =', CASE_KEY)
print('AGE_KEY      =', AGE_KEY)
print('META_KEY     =', META_KEY)
print('BROAD_META_KEY =', BROAD_META_KEY)
print('GROUPED_DISPLAY_META_KEY =', GROUPED_DISPLAY_META_KEY)
print('CONVERT_TYPE =', CONVERT_TYPE)
print()
print('unique id        =', adata.obs['id'].astype(str).nunique())
print('unique case      =', adata.obs['case'].astype(str).nunique())
print('unique orig.ident=', adata.obs['orig.ident'].astype(str).nunique())




In [ ]:
# Donor-age summary
AGE_MIN = 25

case_age_df = adata.obs[[CASE_KEY, AGE_KEY, 'case']].drop_duplicates().copy()
case_age_df[AGE_KEY] = pd.to_numeric(case_age_df[AGE_KEY], errors='coerce')
case_age_df = case_age_df.sort_values([AGE_KEY, CASE_KEY, 'case']).reset_index(drop=True)

case_count_df = (
    adata.obs.groupby([CASE_KEY, AGE_KEY])
    .size()
    .reset_index(name='n_cells')
)
case_count_df[AGE_KEY] = pd.to_numeric(case_count_df[AGE_KEY], errors='coerce')
case_count_df = case_count_df.sort_values([AGE_KEY, 'n_cells'], ascending=[True, False]).reset_index(drop=True)

eligible_case_info = case_count_df[case_count_df[AGE_KEY] >= AGE_MIN].copy()

print('AGE_MIN =', AGE_MIN)
print('Unique donor-age pairs:', len(case_count_df))
print(f'Donor-age pairs with {AGE_KEY} >= {AGE_MIN}:', len(eligible_case_info))
print()
print('Example donor -> sample(case) mapping:')
display(case_age_df.head(20))
print('Eligible donor table:')
display(eligible_case_info.head(20))


In [ ]:
# FastCCC expects normalized + log1p input.
# The original matrix is NOT log-transformed, so each donor-specific subset
# will be normalized and log1p-transformed before running FastCCC.
MIN_CELLS_PER_CASE = 500
MAX_CASES = None   # full run; set an integer for a pilot run
USE_CASE_LIST = None  # optional donor id list for pilot/full rerun

eligible_cases = eligible_case_info.loc[
    eligible_case_info['n_cells'] >= MIN_CELLS_PER_CASE,
    CASE_KEY,
].astype(str).tolist()

if USE_CASE_LIST is not None:
    eligible_cases = [c for c in eligible_cases if c in set(USE_CASE_LIST)]
if MAX_CASES is not None:
    eligible_cases = eligible_cases[:int(MAX_CASES)]

print('Eligible donor ids after age and cell-count filtering:', len(eligible_cases))
print(eligible_cases[:20])


In [ ]:
# Quick input range check on the original matrix
# This confirms the source matrix is not yet log-transformed.
X = adata.X
try:
    x_min = X.min()
    x_max = X.max()
except Exception:
    x_min = np.min(X)
    x_max = np.max(X)

print('Original X min =', x_min)
print('Original X max =', x_max)
print('If Original X max is much larger than ~14, FastCCC cannot use it directly.')
print('We therefore normalize_total + log1p inside each eligible case before analysis.')


In [ ]:
def normalize_for_fastccc(sub):
    sub = sub.copy()
    sc.pp.normalize_total(sub, target_sum=1e4)
    sc.pp.log1p(sub)
    return sub


def _safe_case_age(sub_obs, age_key=AGE_KEY):
    vals = pd.to_numeric(sub_obs[age_key], errors='coerce').dropna().unique()
    if len(vals) == 0:
        return np.nan
    return float(vals[0])


FASTCCC_FORCE_RERUN = False
AUTO_RERUN_MISSING_CASE_RESULTS = False
RUN_FASTCCC_IF_MISSING = True


def case_result_paths(case_id):
    case_dir = RESULTS / str(case_id)
    return {
        'case_dir': case_dir,
        'strength_file': case_dir / 'interactions_strength.tsv',
        'pval_file': case_dir / 'pvals.tsv',
        'percent_file': case_dir / 'percents_analysis.tsv',
        'input_path': TMP / f'{case_id}.h5ad',
    }


def case_result_files_exist(case_id):
    paths = case_result_paths(case_id)
    return (
        paths['strength_file'].exists()
        and paths['pval_file'].exists()
        and paths['percent_file'].exists()
    )


def run_fastccc_for_case(case_id, adata, case_key=CASE_KEY, age_key=AGE_KEY, meta_key=META_KEY, force_rerun=FASTCCC_FORCE_RERUN):
    sub = adata[adata.obs[case_key].astype(str) == str(case_id)].copy()
    n_cells = int(sub.n_obs)
    age_val = _safe_case_age(sub.obs, age_key=age_key)

    if pd.isna(age_val) or age_val < AGE_MIN:
        return {'case': str(case_id), 'age': age_val, 'n_cells': n_cells, 'status': 'skip_age_filter'}

    if n_cells < MIN_CELLS_PER_CASE:
        return {'case': str(case_id), 'age': age_val, 'n_cells': n_cells, 'status': 'skip_too_few_cells'}

    paths = case_result_paths(case_id)
    paths['case_dir'].mkdir(parents=True, exist_ok=True)

    if case_result_files_exist(case_id) and not force_rerun:
        interactions_strength = pd.read_csv(paths['strength_file'], sep='\t', index_col=0)
        return {
            'case': str(case_id),
            'age': age_val,
            'n_cells': n_cells,
            'n_celltypes': int(sub.obs[meta_key].nunique()),
            'x_max_after_log1p': np.nan,
            'n_cellpairs': int(interactions_strength.shape[0]),
            'n_lr_pairs': int(interactions_strength.shape[1]),
            'status': 'exists_skip',
            'result_dir': str(paths['case_dir']),
            'input_path': str(paths['input_path']),
        }

    if not force_rerun and not RUN_FASTCCC_IF_MISSING:
        return {
            'case': str(case_id),
            'age': age_val,
            'n_cells': n_cells,
            'n_celltypes': int(sub.obs[meta_key].nunique()),
            'status': 'missing_cache_skip',
            'result_dir': str(paths['case_dir']),
            'input_path': str(paths['input_path']),
        }

    # FastCCC needs normalized + log1p values, so we transform each donor here.
    sub = normalize_for_fastccc(sub)

    X = sub.X
    try:
        x_max = float(X.max())
    except Exception:
        x_max = float(np.max(X))

    sub.write_h5ad(paths['input_path'])

    interactions_strength, pvals, percents_analysis = fastccc.statistical_analysis_method(
        str(DB_PATH),
        None,
        str(paths['input_path']),
        CONVERT_TYPE,
        meta_key=meta_key,
        save_path=str(paths['case_dir']),
    )

    interactions_strength.to_csv(paths['strength_file'], sep='\t')
    pvals.to_csv(paths['pval_file'], sep='\t')
    percents_analysis.to_csv(paths['percent_file'], sep='\t')

    return {
        'case': str(case_id),
        'age': age_val,
        'n_cells': n_cells,
        'n_celltypes': int(sub.obs[meta_key].nunique()),
        'x_max_after_log1p': x_max,
        'n_cellpairs': int(interactions_strength.shape[0]),
        'n_lr_pairs': int(interactions_strength.shape[1]),
        'status': 'ok',
        'result_dir': str(paths['case_dir']),
        'input_path': str(paths['input_path']),
    }




In [ ]:
# Run FastCCC for each donor id
RUN_SUMMARY_FILE = OUT / 'run_summary.tsv'
REUSE_RUN_SUMMARY = False


def _run_summary_is_usable(df):
    if df is None or df.empty or 'case' not in df.columns:
        return False
    summary_cases = set(df['case'].astype(str))
    needed_cases = set(map(str, eligible_cases))
    if not needed_cases.issubset(summary_cases):
        return False
    ok_like = df[df['status'].astype(str).isin(['ok', 'exists_skip'])]['case'].astype(str).tolist()
    return all(case_result_files_exist(case_id) for case_id in ok_like)


if RUN_SUMMARY_FILE.exists() and REUSE_RUN_SUMMARY and not FASTCCC_FORCE_RERUN:
    cached_run_summary = pd.read_csv(RUN_SUMMARY_FILE, sep='\t')
    if _run_summary_is_usable(cached_run_summary):
        run_summary = cached_run_summary.copy()
        print('Loaded cached run summary:', RUN_SUMMARY_FILE)
    else:
        print('Cached run summary exists but some FastCCC result files are missing. Rebuilding run summary.')
        run_records = []
        for i, case_id in enumerate(eligible_cases, 1):
            print(f'[{i}/{len(eligible_cases)}] donor_id={case_id}')
            try:
                rec = run_fastccc_for_case(case_id, adata)
            except Exception as e:
                rec = {'case': str(case_id), 'age': np.nan, 'n_cells': np.nan, 'status': f'error: {repr(e)}'}
            run_records.append(rec)
            print(rec)
        run_summary = pd.DataFrame(run_records)
        run_summary.to_csv(RUN_SUMMARY_FILE, sep='\t', index=False)
        print('Saved:', RUN_SUMMARY_FILE)
else:
    run_records = []

    for i, case_id in enumerate(eligible_cases, 1):
        print(f'[{i}/{len(eligible_cases)}] donor_id={case_id}')
        try:
            rec = run_fastccc_for_case(case_id, adata)
        except Exception as e:
            rec = {'case': str(case_id), 'age': np.nan, 'n_cells': np.nan, 'status': f'error: {repr(e)}'}
        run_records.append(rec)
        print(rec)

    run_summary = pd.DataFrame(run_records)
    run_summary.to_csv(RUN_SUMMARY_FILE, sep='\t', index=False)
    print('Saved:', RUN_SUMMARY_FILE)

display(run_summary.head(20))



In [ ]:
# Keep successful donor ids only
ok_cases = run_summary.loc[run_summary['status'].astype(str).isin(['ok', 'exists_skip']), 'case'].astype(str).tolist()
print('Successful/cached donor ids:', len(ok_cases))
print(ok_cases[:20])


In [ ]:
def parse_cellpair(cellpair):
    s = str(cellpair)
    for sep in ['|', ' -> ', '->', '__', ',']:
        if sep in s:
            parts = [x.strip() for x in s.split(sep)]
            if len(parts) == 2:
                return parts[0], parts[1]
    return s, np.nan


def load_case_fastccc(case_id, age_val=None, auto_rerun_missing=AUTO_RERUN_MISSING_CASE_RESULTS):
    paths = case_result_paths(case_id)
    if not case_result_files_exist(case_id):
        print(f'Missing FastCCC result files for donor_id={case_id}. Skipping cached-only extraction.')
        empty = pd.DataFrame()
        return empty, empty, empty

    strength = pd.read_csv(paths['strength_file'], sep='\t', index_col=0)
    pvals = pd.read_csv(paths['pval_file'], sep='\t', index_col=0)
    percent = pd.read_csv(paths['percent_file'], sep='\t', index_col=0)

    # Robust alignment: some FastCCC outputs may differ slightly in row/col sets
    # across files. Always intersect + reorder before indexing by np.where positions.
    strength.index = strength.index.astype(str)
    pvals.index = pvals.index.astype(str)
    percent.index = percent.index.astype(str)
    strength.columns = strength.columns.astype(str)
    pvals.columns = pvals.columns.astype(str)
    percent.columns = percent.columns.astype(str)

    common_rows = strength.index.intersection(pvals.index).intersection(percent.index)
    common_cols = strength.columns.intersection(pvals.columns).intersection(percent.columns)

    if len(common_rows) == 0 or len(common_cols) == 0:
        empty = pd.DataFrame()
        return empty, empty, empty

    strength = strength.loc[common_rows, common_cols]
    pvals = pvals.loc[common_rows, common_cols]
    percent = percent.loc[common_rows, common_cols]
    return strength, pvals, percent


def extract_significant_case_fastccc(case_id, age_val, pval_cutoff=0.05):
    # This is much faster than melting the full matrix. We first locate the
    # significant entries in the p-value table, then fetch only matching rows.
    strength, pvals, percent = load_case_fastccc(case_id, age_val=age_val)

    if pvals.empty:
        return pd.DataFrame(columns=[
            'cellpair', 'lr_pair', 'strength', 'pval', 'percent_ok',
            'sender', 'receiver', 'case', 'age', 'sig'
        ])

    pmat = pd.to_numeric(pvals.stack(), errors='coerce').unstack()
    pmat = pmat.reindex(index=pvals.index, columns=pvals.columns)
    sig_pos = np.where(pmat.values < pval_cutoff)
    row_idx = sig_pos[0]
    col_idx = sig_pos[1]

    if len(row_idx) == 0:
        return pd.DataFrame(columns=[
            'cellpair', 'lr_pair', 'strength', 'pval', 'percent_ok',
            'sender', 'receiver', 'case', 'age', 'sig'
        ])

    cellpairs = strength.index.to_numpy()[row_idx]
    lr_pairs = strength.columns.to_numpy()[col_idx]
    strengths = strength.values[row_idx, col_idx]
    pval_vals = pmat.values[row_idx, col_idx]
    percent_vals = percent.values[row_idx, col_idx]

    df = pd.DataFrame({
        'cellpair': cellpairs,
        'lr_pair': lr_pairs,
        'strength': strengths,
        'pval': pval_vals,
        'percent_ok': percent_vals,
    })
    df[['sender', 'receiver']] = df['cellpair'].apply(lambda x: pd.Series(parse_cellpair(x)))
    df['case'] = str(case_id)
    df['age'] = float(age_val) if pd.notna(age_val) else np.nan
    df['sig'] = True
    return df





In [ ]:
# Build or reuse the significant FastCCC long table.
# This does not rerun FastCCC; it only extracts significant matrix entries from cached per-donor outputs.
LONG_SIG_FILE = OUT / 'fastccc_long_sig_only.tsv'
REUSE_LONG_SIG_TABLE = False

if LONG_SIG_FILE.exists() and REUSE_LONG_SIG_TABLE and not FASTCCC_FORCE_RERUN:
    fastccc_long = pd.read_csv(LONG_SIG_FILE, sep='\t')
    print('Loaded cached significant long table:', LONG_SIG_FILE)
else:
    sig_long_files = []
    for i, (_, row) in enumerate(run_summary[run_summary['status'].astype(str).isin(['ok', 'exists_skip'])].iterrows(), 1):
        case_id = str(row['case'])
        age_val = row['age']
        print(f'[{i}] extract significant rows for donor_id={case_id}')
        df = extract_significant_case_fastccc(case_id, age_val, pval_cutoff=0.05)
        out_file = OUT / f'{case_id}_long_sig.tsv'
        df.to_csv(out_file, sep='\t', index=False)
        sig_long_files.append(out_file)
        print('saved:', out_file, '| rows =', len(df))
    all_long = [pd.read_csv(f, sep='\t') for f in sig_long_files if f.exists()]
    fastccc_long = pd.concat(all_long, ignore_index=True) if all_long else pd.DataFrame()
    fastccc_long.to_csv(LONG_SIG_FILE, sep='\t', index=False)

print('Significant long table shape:', fastccc_long.shape)
display(fastccc_long.head(20))


## Mainline CPDBv5 donor-level continuous aging CCC

The following cells replace the earlier observed-only Spearman workflow with zero-filled donor-level panels, CPDB annotation, nominal Spearman discovery, covariate sensitivity, and grouped companion visualization.

In [ ]:
# Local imports make this cell runnable even if the notebook is resumed from here.
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns

# Output folders and grouped-age definitions
TAB_DIR = OUT / 'tables'
FIG_DIR = Path('/home/aya/图片/14')
for p in [TAB_DIR, FIG_DIR]:
    p.mkdir(parents=True, exist_ok=True)

AGE_ORDER = ['Young adult', 'Adult', 'Elderly']
AGE_PAL = {'Young adult': '#3B74AF', 'Adult': '#D39C32', 'Elderly': '#C84E4E'}
SPEARMAN_P_DISCOVERY = 0.05
SPEARMAN_RHO_DISCOVERY = 0.30
MIN_NONZERO_DONORS = 5
MIN_DONOR_FRAC = 0.35

mpl.rcParams.update({'pdf.fonttype': 42, 'ps.fonttype': 42, 'svg.fonttype': 'none', 'savefig.dpi': 300})

def save_fig(fig, name, outdir=FIG_DIR, formats=('pdf','svg'), dpi=300, strip_titles=False):
    outdir = Path(outdir); outdir.mkdir(parents=True, exist_ok=True)
    hidden = []
    if strip_titles:
        st = getattr(fig, '_suptitle', None)
        if st is not None and st.get_visible():
            hidden.append((st, True)); st.set_visible(False)
        for ax in fig.axes:
            title = ax.title
            if title is not None and title.get_text() and title.get_visible():
                hidden.append((title, True)); title.set_visible(False)
    try:
        for fmt in formats:
            fig.savefig(outdir / f'{name}.{fmt}', format=fmt, dpi=dpi, bbox_inches='tight', pad_inches=0.05)
    finally:
        for artist, visible in hidden:
            artist.set_visible(visible)

def assign_age_group(age):
    if age < 40:
        return 'Young adult'
    if age < 70:
        return 'Adult'
    return 'Elderly'

def first_existing_col(df, candidates, fallback=None):
    for c in candidates:
        if c in df.columns:
            return c
    return fallback

# Donor-level metadata and cell fractions.
donor_base = eligible_case_info.rename(columns={CASE_KEY: 'donor_id', AGE_KEY: 'age'}).copy()
donor_base['donor_id'] = donor_base['donor_id'].astype(str)
donor_base['case'] = donor_base['donor_id']
donor_base['age_group'] = pd.Categorical(donor_base['age'].map(assign_age_group), categories=AGE_ORDER, ordered=True)
donor_base['log_total_cells'] = np.log1p(donor_base['n_cells'])

obs2 = adata.obs[[CASE_KEY, META_KEY, BROAD_META_KEY]].copy()
obs2[CASE_KEY] = obs2[CASE_KEY].astype(str)
obs2['celltype'] = obs2[META_KEY].astype(str)
obs2['broad_celltype'] = obs2[BROAD_META_KEY].astype(str)
ct_counts = obs2.groupby([CASE_KEY, 'celltype']).size().rename('n').reset_index()
ct_counts = ct_counts.merge(ct_counts.groupby(CASE_KEY)['n'].sum().rename('total'), on=CASE_KEY)
ct_counts['fraction'] = ct_counts['n'] / ct_counts['total'].replace(0, np.nan)
ct_frac_df = ct_counts.pivot(index=CASE_KEY, columns='celltype', values='fraction').fillna(0.0)
ct_frac_df.index = ct_frac_df.index.astype(str)

# Intermediate annotation table kept for optional display checks only.
# Main FastCCC, continuous analyses, and Figure 1 composition use META_KEY = new_clusters.
if 'new_clusters2' in adata.obs.columns:
    obs_mid = adata.obs[[CASE_KEY, 'new_clusters2']].copy()
    obs_mid[CASE_KEY] = obs_mid[CASE_KEY].astype(str)
    obs_mid['celltype_mid'] = obs_mid['new_clusters2'].astype(str)
    mid_counts = obs_mid.groupby([CASE_KEY, 'celltype_mid']).size().rename('n').reset_index()
    mid_counts = mid_counts.merge(mid_counts.groupby(CASE_KEY)['n'].sum().rename('total'), on=CASE_KEY)
    mid_counts['fraction'] = mid_counts['n'] / mid_counts['total'].replace(0, np.nan)
    ct_frac_mid_df = mid_counts.pivot(index=CASE_KEY, columns='celltype_mid', values='fraction').fillna(0.0)
    ct_frac_mid_df.index = ct_frac_mid_df.index.astype(str)
else:
    ct_frac_mid_df = ct_frac_df.copy()

donor_base.to_csv(TAB_DIR / 'S1_donor_metadata.tsv', sep='\t', index=False)
ct_frac_df.to_csv(TAB_DIR / 'S2_celltype_fraction_per_donor.tsv', sep='\t')
ct_frac_mid_df.to_csv(TAB_DIR / 'S2_new_clusters2_fraction_per_donor.tsv', sep='\t')
display(donor_base.groupby('age_group', observed=False).agg(n=('donor_id','nunique'), min_age=('age','min'), max_age=('age','max')))






In [ ]:
# Broad/fine annotation map from Jeffries h5ad.
# Fine annotation: new_clusters; broad annotation: new_clusters3. Fig.4h grouped display maps to new_clusters2 later.
assert BROAD_META_KEY in adata.obs.columns, f'Missing broad annotation column: {BROAD_META_KEY}'

celltype_map = (adata.obs[[META_KEY, BROAD_META_KEY]]
                .drop_duplicates()
                .rename(columns={META_KEY: 'celltype', BROAD_META_KEY: 'broad_celltype'})
                .sort_values(['broad_celltype', 'celltype'])
                .reset_index(drop=True))
celltype_map['celltype'] = celltype_map['celltype'].astype(str)
celltype_map['broad_celltype'] = celltype_map['broad_celltype'].astype(str)
# L2/3-2 is infant-enriched in this atlas and should not drive adult/aging interpretation.
# Keep this as a single analysis switch so all downstream fine-pair, LR-driver,
# network-role, and grouped-LRI analyses inherit the same exclusion.
EXCLUDED_FINE_CELLTYPES = {'L2/3-2'}

celltype_map['excluded_from_main_analysis'] = celltype_map['celltype'].isin(EXCLUDED_FINE_CELLTYPES)
celltype_map['note'] = np.select(
    [
        celltype_map['celltype'].isin(EXCLUDED_FINE_CELLTYPES),
        celltype_map['celltype'].isin(['AST-3']),
    ],
    [
        'excluded from main analysis; infant-enriched cluster',
        'infant-enriched cluster; interpret cautiously',
    ],
    default='',
)

# Lookup used later for FastCCC cellpair aggregation.
broad_lookup = dict(zip(celltype_map['celltype'], celltype_map['broad_celltype']))

def broad_celltype(celltype):
    return broad_lookup.get(str(celltype), str(celltype))

celltype_map.to_csv(TAB_DIR / 'S2_celltype_broad_map.tsv', sep='\t', index=False)
display(celltype_map)





In [ ]:

# CPDB annotation loader.
# CPI ids such as CPI-SS0C272BD3D are CPDB interaction IDs, not genes.
# This cell maps CPI interaction IDs to real ligand/receptor gene symbols using
# all CPDB annotation tables available in DB_PATH.
import re


def read_db_table(path):
    return pd.read_csv(path, sep='\t' if Path(path).suffix.lower() != '.csv' else ',')


def find_db_table(patterns):
    files = []
    for pat in patterns:
        files.extend(Path(DB_PATH).rglob(pat))
    return sorted(files, key=lambda x: str(x))[0] if files else None


def find_db_tables(patterns):
    files = []
    for pat in patterns:
        files.extend(Path(DB_PATH).rglob(pat))
    # Stable unique order.
    out = []
    seen = set()
    for f in sorted(files, key=lambda x: str(x)):
        if str(f) not in seen:
            out.append(f)
            seen.add(str(f))
    return out


def _first_existing_col(df, candidates, default=None):
    for c in candidates:
        if c in df.columns:
            return c
    low = {str(c).lower(): c for c in df.columns}
    for c in candidates:
        if str(c).lower() in low:
            return low[str(c).lower()]
    return default


def _looks_like_gene_symbol(x):
    if pd.isna(x):
        return False
    s = str(x).strip()
    if not s or s.lower() in {'nan', 'none', 'null', 'na'}:
        return False
    if re.match(r'^CPI[-_]', s, flags=re.IGNORECASE):
        return False
    # Avoid full protein descriptions; allow HGNC-style symbols including HLA-DRA, C1QA, GABRA1.
    if ' ' in s or len(s) > 35:
        return False
    return bool(re.match(r'^[A-Za-z][A-Za-z0-9_.-]*$', s))


def _clean_gene_token(x):
    if not _looks_like_gene_symbol(x):
        return None
    return str(x).strip()


def _split_gene_field(value):
    if pd.isna(value):
        return []
    text = str(value).strip()
    if not text:
        return []
    # If the whole field is a valid HGNC-like symbol, preserve it.
    if _looks_like_gene_symbol(text):
        return [text]
    toks = re.split(r'[+;&|,/\s]+', text)
    genes = []
    for tok in toks:
        g = _clean_gene_token(tok)
        if g and g not in genes:
            genes.append(g)
    return genes


def _join_genes(genes):
    out = []
    for g in genes:
        gg = _clean_gene_token(g)
        if gg and gg not in out:
            out.append(gg)
    return '+'.join(out) if out else np.nan


def _is_probable_id_col(col):
    name = str(col).lower()
    return any(k in name for k in ['id', 'multidata', 'uniprot', 'protein', 'complex', 'partner', 'interactor'])


def _is_probable_gene_col(col):
    name = str(col).lower()
    return any(k in name for k in ['hgnc', 'gene_name', 'genesymbol', 'gene_symbol', 'symbol']) or name in {'gene', 'name'}


def _read_all_cpdb_tables():
    tables = {}
    for path in find_db_tables(['*.csv', '*.tsv', '*.txt']):
        try:
            df = read_db_table(path)
        except Exception as exc:
            print(f'Could not read CPDB table {path}: {exc}')
            continue
        tables[path.name] = (path, df)
    print('CPDB tables found:')
    for name, (path, df) in tables.items():
        print(f'  {name}: {df.shape}; columns={list(df.columns)[:12]}')
    return tables


def _find_interaction_table(tables):
    for name, (path, df) in tables.items():
        cols = {str(c).lower() for c in df.columns}
        if 'id_cp_interaction' in cols or 'interacting_pair' in cols:
            return path, df
    # fallback by filename
    for name, (path, df) in tables.items():
        if 'interaction' in name.lower():
            return path, df
    return None, pd.DataFrame()


def _build_generic_cpdb_id_to_gene_map(tables, interaction_table_name=None):
    """Map protein/complex/multidata/uniprot ids to HGNC-like symbols using every CPDB table."""
    id_to_genes = {}

    def add_map(key, genes):
        if pd.isna(key):
            return
        key = str(key).strip()
        if not key or key.lower() in {'nan', 'none', 'null', 'na'}:
            return
        genes = [g for g in genes if _clean_gene_token(g)]
        if not genes:
            return
        cur = id_to_genes.get(key, [])
        for g in genes:
            if g not in cur:
                cur.append(g)
        id_to_genes[key] = cur

    # Multiple passes allow complex tables to resolve via protein ids found in protein/gene tables.
    for pass_i in range(3):
        for name, (path, df) in tables.items():
            # The interaction table maps interactions to partners, not directly to clean genes in most CPDB DBs.
            # Still allow explicit gene columns if present, but do not map id_cp_interaction to itself.
            id_cols = [c for c in df.columns if _is_probable_id_col(c)]
            gene_cols = [c for c in df.columns if _is_probable_gene_col(c)]
            if not id_cols or not gene_cols:
                continue
            for _, row in df.iterrows():
                genes = []
                for gc in gene_cols:
                    genes.extend(_split_gene_field(row.get(gc)))
                # If gene columns did not contain symbols, try resolving referenced ids in this row.
                if not genes:
                    for c in id_cols:
                        val = row.get(c)
                        if not pd.isna(val):
                            genes.extend(id_to_genes.get(str(val).strip(), []))
                if not genes:
                    continue
                for ic in id_cols:
                    # Avoid mapping an interaction CPI id to component genes here; we resolve interactions by A/B partners below.
                    if str(ic).lower() in {'id_cp_interaction', 'interaction_id'}:
                        continue
                    add_map(row.get(ic), genes)

    return id_to_genes


def _resolve_partner_genes(value, id_to_genes):
    if pd.isna(value):
        return []
    text = str(value).strip()
    if not text or text.lower() in {'nan', 'none', 'null', 'na'}:
        return []
    if text in id_to_genes:
        return id_to_genes[text]
    genes = []
    for tok in re.split(r'[+;&|,/\s]+', text):
        tok = tok.strip()
        if not tok:
            continue
        if tok in id_to_genes:
            genes.extend(id_to_genes[tok])
        else:
            genes.extend(_split_gene_field(tok))
    out = []
    for g in genes:
        gg = _clean_gene_token(g)
        if gg and gg not in out:
            out.append(gg)
    return out


def _resolve_interaction_row_genes(row, inter, id_to_genes, side):
    assert side in {'a', 'b'}
    if side == 'a':
        direct_cols = ['gene_name_a', 'hgnc_symbol_a', 'gene_a', 'ligand', 'protein_name_a']
        id_cols = ['partner_a', 'partner_a_id', 'multidata_1_id', 'protein_a_id', 'interactor_a', 'id_a']
    else:
        direct_cols = ['gene_name_b', 'hgnc_symbol_b', 'gene_b', 'receptor', 'protein_name_b']
        id_cols = ['partner_b', 'partner_b_id', 'multidata_2_id', 'protein_b_id', 'interactor_b', 'id_b']

    # First use explicit gene-symbol columns if they actually contain gene-like values.
    genes = []
    for c in direct_cols:
        cc = _first_existing_col(inter, [c], None)
        if cc:
            genes.extend(_split_gene_field(row.get(cc)))
    if genes:
        return genes

    # Then resolve CPDB partner ids through annotation tables.
    for c in id_cols:
        cc = _first_existing_col(inter, [c], None)
        if cc:
            genes.extend(_resolve_partner_genes(row.get(cc), id_to_genes))
    if genes:
        out = []
        for g in genes:
            if g not in out:
                out.append(g)
        return out

    # Last fallback: interacting_pair may be GENE_RECEPTOR in some DB exports.
    pair_col = _first_existing_col(inter, ['interacting_pair', 'lr_pair', 'name'], None)
    if pair_col and '_' in str(row.get(pair_col, '')) and not re.match(r'^CPI[-_]', str(row.get(pair_col, '')), flags=re.IGNORECASE):
        parts = str(row.get(pair_col)).split('_', 1)
        return _split_gene_field(parts[0] if side == 'a' else parts[1])
    return []


def load_cpdb_annotations():
    tables = _read_all_cpdb_tables()
    p, inter = _find_interaction_table(tables)
    if p is None or inter.empty:
        print('No CPDB interaction table found; using minimal LR annotation.')
        return pd.DataFrame(columns=['lr_pair','classification','directionality','interacting_pair','ligand','receptor'])

    id_to_genes = _build_generic_cpdb_id_to_gene_map(tables, interaction_table_name=p.name)
    print('Generic CPDB id -> gene mappings:', len(id_to_genes))

    id_col = _first_existing_col(inter, ['id_cp_interaction','interaction_id','id','lr_pair'], inter.columns[0])
    class_col = _first_existing_col(inter, ['classification','annotation_strategy','annotation','source'], None)
    direction_col = _first_existing_col(inter, ['directionality','is_directional'], None)
    pair_col = _first_existing_col(inter, ['interacting_pair','lr_pair','name'], None)

    records = []
    for _, row in inter.iterrows():
        ligands = _resolve_interaction_row_genes(row, inter, id_to_genes, 'a')
        receptors = _resolve_interaction_row_genes(row, inter, id_to_genes, 'b')
        records.append({
            'lr_pair': str(row.get(id_col)),
            'classification': str(row.get(class_col)) if class_col else 'CPDB interaction',
            'directionality': str(row.get(direction_col)) if direction_col else 'unknown',
            'interacting_pair': str(row.get(pair_col)) if pair_col else str(row.get(id_col)),
            'ligand': _join_genes(ligands),
            'receptor': _join_genes(receptors),
        })
    out = pd.DataFrame(records).drop_duplicates('lr_pair')
    out['has_gene_symbol_mapping'] = out['ligand'].notna() | out['receptor'].notna()
    out['ligand_is_cpi'] = out['ligand'].astype(str).str.contains(r'^CPI[-_]', case=False, na=False)
    out['receptor_is_cpi'] = out['receptor'].astype(str).str.contains(r'^CPI[-_]', case=False, na=False)
    print('Loaded CPDB interaction table:', p, inter.shape)
    print('Mapped CPI interactions to genes:', int(out['has_gene_symbol_mapping'].sum()), '/', len(out))
    print('Rows still CPI-like ligand/receptor:', int((out['ligand_is_cpi'] | out['receptor_is_cpi']).sum()))
    for q in ['CPI-SS0C272BD3D', 'CPI-CS058DED3AA']:
        hit = out[out['lr_pair'].astype(str).eq(q)]
        if not hit.empty:
            print('Example mapping', q, '=>', hit[['ligand','receptor','classification']].to_dict('records')[0])
    out.drop(columns=['ligand_is_cpi','receptor_is_cpi'], errors='ignore').to_csv(TAB_DIR / 'S3_cpdb_lr_annotation.tsv', sep='\t', index=False)
    unmapped = out[~out['has_gene_symbol_mapping']].copy()
    unmapped.to_csv(TAB_DIR / 'S3_cpdb_unmapped_interactions.tsv', sep='\t', index=False)
    return out.drop(columns=['ligand_is_cpi','receptor_is_cpi'], errors='ignore')


lr_anno = load_cpdb_annotations()
if not fastccc_long.empty and not lr_anno.empty:
    annot_cols = ['classification', 'directionality', 'interacting_pair', 'ligand', 'receptor', 'has_gene_symbol_mapping']
    fastccc_long = fastccc_long.drop(columns=[c for c in annot_cols if c in fastccc_long.columns], errors='ignore')
    fastccc_long = fastccc_long.merge(lr_anno, on='lr_pair', how='left')
if not fastccc_long.empty:
    fastccc_long['classification'] = fastccc_long.get('classification', pd.Series(index=fastccc_long.index, dtype=object)).fillna('Unannotated CPDB interaction')
    fastccc_long.to_csv(TAB_DIR / 'S3_fastccc_long_with_cpdb_annotation.tsv', sep='\t', index=False)
print('CPDB annotation preview:')
display(lr_anno.head(20))


In [ ]:
# Zero-filled donor-level panels.
# Broad uses new_clusters3 fractions; fine/LR use new_clusters fractions.

def _build_fraction_table(obs_df, donor_col, celltype_col):
    tmp = obs_df[[donor_col, celltype_col]].copy()
    tmp[donor_col] = tmp[donor_col].astype(str)
    tmp[celltype_col] = tmp[celltype_col].astype(str)
    counts = tmp.groupby([donor_col, celltype_col]).size().rename('n').reset_index()
    totals = counts.groupby(donor_col)['n'].sum().rename('total').reset_index()
    counts = counts.merge(totals, on=donor_col, how='left')
    counts['fraction'] = counts['n'] / counts['total'].replace(0, np.nan)
    frac = counts.pivot(index=donor_col, columns=celltype_col, values='fraction').fillna(0.0)
    frac.index = frac.index.astype(str)
    frac.columns = frac.columns.astype(str)
    return frac


excluded_fine = set(globals().get('EXCLUDED_FINE_CELLTYPES', set()))
analysis_obs = adata.obs.copy()
if excluded_fine:
    analysis_obs = analysis_obs[~analysis_obs[META_KEY].astype(str).isin(excluded_fine)].copy()

ct_frac_fine_df = _build_fraction_table(
    analysis_obs.rename(columns={CASE_KEY: 'donor_id'}),
    'donor_id',
    META_KEY,
)

ct_frac_broad_df = _build_fraction_table(
    analysis_obs.rename(columns={CASE_KEY: 'donor_id'}),
    'donor_id',
    BROAD_META_KEY,
)

ct_frac_fine_df.to_csv(TAB_DIR / 'S2_fine_celltype_fraction_per_donor.tsv', sep='\t')
ct_frac_broad_df.to_csv(TAB_DIR / 'S2_broad_celltype_fraction_per_donor.tsv', sep='\t')


def _lookup_fraction(frac_df, donor_id, celltype):
    donor_id = str(donor_id)
    celltype = str(celltype)
    if donor_id in frac_df.index and celltype in frac_df.columns:
        return float(frac_df.loc[donor_id, celltype])
    return 0.0


def add_cell_fractions(df, frac_df, sender_col='sender', receiver_col='receiver'):
    df = df.loc[:, ~df.columns.duplicated()].copy()
    df['sender_fraction'] = [
        _lookup_fraction(frac_df, d, s)
        for d, s in zip(df['donor_id'].astype(str), df[sender_col].astype(str))
    ]
    df['receiver_fraction'] = [
        _lookup_fraction(frac_df, d, r)
        for d, r in zip(df['donor_id'].astype(str), df[receiver_col].astype(str))
    ]
    return df


def zero_fill_panel(obs_df, entity_cols, aggs, meta_cols=None):
    if obs_df.empty:
        return pd.DataFrame()

    obs_df = obs_df.loc[:, ~obs_df.columns.duplicated()].copy()

    grouped = (
        obs_df.groupby(['donor_id'] + entity_cols, as_index=False)
        .agg(**aggs)
    )

    entities = grouped[entity_cols].drop_duplicates().copy()
    entities['key'] = 1

    base = donor_base[['donor_id', 'age', 'age_group', 'n_cells', 'log_total_cells']].copy()
    base['donor_id'] = base['donor_id'].astype(str)
    base['key'] = 1

    full = base.merge(entities, on='key', how='outer').drop(columns='key')
    full = full.merge(grouped, on=['donor_id'] + entity_cols, how='left')

    for col in aggs.keys():
        full[col] = full[col].fillna(0.0)

    if meta_cols:
        meta = obs_df[entity_cols + meta_cols].drop_duplicates(entity_cols)
        full = full.merge(meta, on=entity_cols, how='left')

    return full


fastccc_long = (
    fastccc_long.rename(columns={'case': 'donor_id'})
    if 'case' in fastccc_long.columns and 'donor_id' not in fastccc_long.columns
    else fastccc_long
)

fastccc_long['donor_id'] = fastccc_long['donor_id'].astype(str)
fastccc_long['sender'] = fastccc_long['sender'].astype(str)
fastccc_long['receiver'] = fastccc_long['receiver'].astype(str)

excluded_fine = set(globals().get('EXCLUDED_FINE_CELLTYPES', set()))
if excluded_fine:
    _before_rows = len(fastccc_long)
    excluded_mask = fastccc_long['sender'].isin(excluded_fine) | fastccc_long['receiver'].isin(excluded_fine)
    excluded_fastccc_long = fastccc_long.loc[excluded_mask].copy()
    fastccc_long = fastccc_long.loc[~excluded_mask].copy()
    excluded_summary = pd.DataFrame({
        'excluded_celltype': sorted(excluded_fine),
        'reason': ['infant-enriched cluster; excluded from adult aging analysis'] * len(excluded_fine),
    })
    excluded_summary['fastccc_rows_removed'] = [int(excluded_fastccc_long.shape[0])] * len(excluded_fine)
    excluded_summary['unique_cellpairs_removed'] = [int(excluded_fastccc_long['cellpair'].nunique())] * len(excluded_fine)
    excluded_summary['unique_lris_removed'] = [int(excluded_fastccc_long['lr_pair'].nunique())] * len(excluded_fine)
    excluded_summary.to_csv(TAB_DIR / 'S2_excluded_fine_celltypes.tsv', sep='	', index=False)
    excluded_fastccc_long.to_csv(TAB_DIR / 'S2_excluded_L2_3_2_fastccc_rows.tsv', sep='	', index=False)
    print(f"Excluded fine cell types from main analysis: {sorted(excluded_fine)}; removed {excluded_mask.sum():,}/{_before_rows:,} FastCCC rows.")
else:
    excluded_fastccc_long = pd.DataFrame()

fastccc_long['broad_sender'] = fastccc_long['sender'].map(broad_celltype)
fastccc_long['broad_receiver'] = fastccc_long['receiver'].map(broad_celltype)
fastccc_long['broad_pair'] = fastccc_long['broad_sender'] + '|' + fastccc_long['broad_receiver']
fastccc_long['cellpair_lr'] = fastccc_long['cellpair'].astype(str) + '::' + fastccc_long['lr_pair'].astype(str)

aggs = {
    'sum_strength': ('strength', 'sum'),
    'mean_strength': ('strength', 'mean'),
    'sig_lr_count': ('lr_pair', 'nunique'),
}

# Important: build a clean broad source to avoid duplicate sender/receiver columns.
broad_source = fastccc_long[
    ['donor_id', 'broad_pair', 'broad_sender', 'broad_receiver', 'strength', 'lr_pair']
].copy()
broad_source = broad_source.rename(
    columns={
        'broad_sender': 'sender',
        'broad_receiver': 'receiver',
    }
)

fine_source = fastccc_long[
    ['donor_id', 'cellpair', 'sender', 'receiver', 'strength', 'lr_pair']
].copy()

classification_source = fastccc_long[
    ['donor_id', 'classification', 'strength', 'lr_pair']
].copy()

lr_source_cols = [c for c in [
    'donor_id', 'cellpair_lr', 'cellpair', 'lr_pair', 'sender', 'receiver',
    'classification', 'directionality', 'interacting_pair', 'ligand', 'receptor',
    'has_gene_symbol_mapping', 'strength'
] if c in fastccc_long.columns]
lr_source = fastccc_long[lr_source_cols].copy()

broad_pair_full = zero_fill_panel(
    broad_source,
    ['broad_pair'],
    aggs,
    ['sender', 'receiver'],
)

fine_pair_full = zero_fill_panel(
    fine_source,
    ['cellpair'],
    aggs,
    ['sender', 'receiver'],
)

classification_full = zero_fill_panel(
    classification_source,
    ['classification'],
    aggs,
)

lr_meta_cols = [c for c in [
    'cellpair', 'lr_pair', 'sender', 'receiver', 'classification', 'directionality',
    'interacting_pair', 'ligand', 'receptor', 'has_gene_symbol_mapping'
] if c in lr_source.columns]
lr_agg = zero_fill_panel(
    lr_source,
    ['cellpair_lr'],
    {'sum_strength': ('strength', 'sum')},
    lr_meta_cols,
)

broad_pair_full = add_cell_fractions(broad_pair_full, ct_frac_broad_df) if not broad_pair_full.empty else broad_pair_full
fine_pair_full = add_cell_fractions(fine_pair_full, ct_frac_fine_df) if not fine_pair_full.empty else fine_pair_full
lr_agg = add_cell_fractions(lr_agg, ct_frac_fine_df) if not lr_agg.empty else lr_agg

if not classification_full.empty:
    classification_full['sender_fraction'] = 0.0
    classification_full['receiver_fraction'] = 0.0

broad_pair_full.to_csv(TAB_DIR / 'S4_broad_pair_donor_panel.tsv', sep='\t', index=False)
fine_pair_full.to_csv(TAB_DIR / 'S5_fine_pair_donor_panel.tsv', sep='\t', index=False)
classification_full.to_csv(TAB_DIR / 'S6_classification_donor_panel.tsv', sep='\t', index=False)
lr_agg.to_csv(TAB_DIR / 'S7_lr_driver_donor_panel.tsv', sep='\t', index=False)

print('Panels:')
print('  broad_pair_full:', broad_pair_full.shape)
print('  fine_pair_full:', fine_pair_full.shape)
print('  classification_full:', classification_full.shape)
print('  lr_agg:', lr_agg.shape)

print('Duplicate columns check:')
print('  broad:', broad_pair_full.columns[broad_pair_full.columns.duplicated()].tolist())
print('  fine:', fine_pair_full.columns[fine_pair_full.columns.duplicated()].tolist())
print('  lr:', lr_agg.columns[lr_agg.columns.duplicated()].tolist())

display(broad_pair_full.head())






## Donor-level nominal Spearman discovery and sensitivity annotation

In [ ]:
def spearman_age_trend(values, ages):
    tmp = pd.DataFrame({'value': values, 'age': ages}).replace([np.inf, -np.inf], np.nan).dropna()
    if len(tmp) < 5 or tmp['value'].nunique() < 2 or tmp['age'].nunique() < 2:
        return pd.Series({'spearman_rho': np.nan, 'spearman_p': np.nan})
    rho, p = st.spearmanr(tmp['age'], tmp['value'])
    return pd.Series({'spearman_rho': float(rho), 'spearman_p': float(p)})

def support_summary(values):
    vals = pd.Series(values, dtype='float64').replace([np.inf, -np.inf], np.nan).dropna()
    if vals.empty:
        return pd.Series({'n_donors_modeled': 0, 'n_nonzero_donors': 0, 'donor_frac': np.nan, 'value_sd': np.nan})
    n = len(vals); nz = int((vals.abs() > 0).sum())
    return pd.Series({'n_donors_modeled': int(n), 'n_nonzero_donors': nz, 'donor_frac': nz/n if n else np.nan, 'value_sd': float(vals.std(ddof=0))})

def ordered_group_pattern(values, groups, order=AGE_ORDER):
    tmp = pd.DataFrame({'value': values, 'age_group': groups}).dropna()
    means = tmp.groupby('age_group', observed=False)['value'].mean().reindex(order)
    if means.isna().any():
        return pd.Series({'ordered_curve_pattern': 'insufficient_groups'})
    y, m, o = means.iloc[0], means.iloc[1], means.iloc[2]
    rng = max(float(means.max() - means.min()), 1e-12); tol = 0.08 * rng
    e, l = m-y, o-m
    if e > tol and l > tol: pat = 'monotonic_up'
    elif e < -tol and l < -tol: pat = 'monotonic_down'
    elif e > tol and l < -tol: pat = 'middle_high'
    elif e < -tol and l > tol: pat = 'middle_low'
    elif abs(e) <= tol and abs(l) > tol: pat = 'late_onset'
    elif abs(e) > tol and abs(l) <= tol: pat = 'early_shift'
    else: pat = 'flat_or_uncertain'
    return pd.Series({'ordered_curve_pattern': pat, 'young_mean_ordered': y, 'adult_mean_ordered': m, 'elderly_mean_ordered': o})

def residual_spearman(model_grp):
    df = model_grp[['value','age','log_total_cells','sender_fraction','receiver_fraction']].replace([np.inf,-np.inf], np.nan).dropna()
    if len(df) < 6 or df['value'].nunique() < 2:
        return pd.Series({'residual_spearman_rho': np.nan, 'residual_spearman_p': np.nan})
    X = np.column_stack([np.ones(len(df)), df[['log_total_cells','sender_fraction','receiver_fraction']].to_numpy(float)])
    y = df['value'].to_numpy(float)
    try:
        b = np.linalg.lstsq(X, y, rcond=None)[0]
        resid = y - X.dot(b)
        rho, p = st.spearmanr(df['age'].to_numpy(float), resid)
        return pd.Series({'residual_spearman_rho': float(rho), 'residual_spearman_p': float(p)})
    except Exception:
        return pd.Series({'residual_spearman_rho': np.nan, 'residual_spearman_p': np.nan})

def fit_age_model(model_grp):
    df = model_grp[['value','age','log_total_cells','sender_fraction','receiver_fraction']].replace([np.inf,-np.inf], np.nan).dropna()
    if len(df) < 6 or df['value'].nunique() < 2:
        return pd.Series({'beta_age': np.nan, 'beta_10y': np.nan, 'p_age': np.nan})
    X = np.column_stack([np.ones(len(df)), df[['age','log_total_cells','sender_fraction','receiver_fraction']].to_numpy(float)])
    y = df['value'].to_numpy(float)
    try:
        b = np.linalg.lstsq(X, y, rcond=None)[0]
        resid = y - X.dot(b); dof = max(len(y)-X.shape[1], 1)
        cov = float((resid @ resid) / dof) * np.linalg.pinv(X.T @ X)
        se = np.sqrt(np.diag(cov)); t = b[1] / se[1] if se[1] > 0 else np.nan
        p = 2 * st.t.sf(abs(t), dof) if pd.notna(t) else np.nan
        return pd.Series({'beta_age': float(b[1]), 'beta_10y': float(10*b[1]), 'p_age': float(p) if pd.notna(p) else np.nan})
    except Exception:
        return pd.Series({'beta_age': np.nan, 'beta_10y': np.nan, 'p_age': np.nan})

def discovery_selected(row):
    return (pd.notna(row.get('spearman_p')) and row['spearman_p'] < SPEARMAN_P_DISCOVERY and
            pd.notna(row.get('spearman_rho')) and abs(row['spearman_rho']) > SPEARMAN_RHO_DISCOVERY and
            row.get('n_nonzero_donors', 0) >= MIN_NONZERO_DONORS and
            pd.notna(row.get('donor_frac')) and row['donor_frac'] >= MIN_DONOR_FRAC and row.get('value_sd', 0) > 0)

def assign_tier(row):
    if not row.get('discovery_selected', False): return 'not selected'
    rr, sr = row.get('residual_spearman_rho', np.nan), row.get('spearman_rho', np.nan)
    if pd.notna(rr) and pd.notna(sr) and np.sign(rr) == np.sign(sr): return 'Tier 1: main; covariate-consistent nominal trend'
    return 'Tier 2: main-caution; covariate-sensitive nominal trend'

def dual_track(panel, entity_col, metrics=('mean_strength',)):
    rows = []
    if panel is None or panel.empty: return pd.DataFrame()
    for metric in metrics:
        if metric not in panel.columns: continue
        for ent, sub in panel.groupby(entity_col, observed=False):
            mg = sub[['donor_id','age','age_group','log_total_cells','sender_fraction','receiver_fraction',metric]].rename(columns={metric:'value'}).copy()
            rec = {entity_col: ent, 'metric': metric}
            for ser in [spearman_age_trend(mg['value'], mg['age']), support_summary(mg['value']), ordered_group_pattern(mg['value'], mg['age_group']), residual_spearman(mg), fit_age_model(mg)]: rec.update(ser.to_dict())
            rec['discovery_selected'] = discovery_selected(rec)
            rec['aging_direction'] = 'aging-up' if rec.get('spearman_rho', 0) > 0 else ('aging-down' if rec.get('spearman_rho', 0) < 0 else 'flat')
            rec['interpretation_tier'] = assign_tier(rec)
            rec['recommended_location'] = 'main figure' if rec['interpretation_tier'].startswith('Tier 1') else ('main text caution' if rec['interpretation_tier'].startswith('Tier 2') else 'supplementary table')
            rows.append(rec)
    out = pd.DataFrame(rows)
    if not out.empty:
        out['abs_spearman_rho'] = out['spearman_rho'].abs()
        out = out.sort_values(['discovery_selected','spearman_p','abs_spearman_rho'], ascending=[False, True, False])
    return out



In [ ]:
broad_models = dual_track(broad_pair_full, 'broad_pair', ['mean_strength','sum_strength','sig_lr_count'])
fine_models = dual_track(fine_pair_full, 'cellpair', ['mean_strength','sum_strength','sig_lr_count'])
class_models = dual_track(classification_full, 'classification', ['mean_strength','sum_strength','sig_lr_count'])
for df, name in [(broad_models,'S4_broad_pair_models'), (fine_models,'S5_fine_pair_models'), (class_models,'S6_classification_models')]:
    df.to_csv(TAB_DIR / f'{name}.tsv', sep='\t', index=False)
print('Selected broad/fine/classification:', int(broad_models.discovery_selected.sum()) if not broad_models.empty else 0, int(fine_models.discovery_selected.sum()) if not fine_models.empty else 0, int(class_models.discovery_selected.sum()) if not class_models.empty else 0)
display(broad_models.head(10)); display(fine_models.head(10)); display(class_models.head(10))



In [ ]:

# Continuous LR driver discovery within the top aging-up/down fine communication pairs.
# Continuous analysis uses fine cell types (new_clusters) and mean_strength-selected communication pairs.


def select_top_directional_fine_pairs(model_df, top_each=2, metric='mean_strength'):
    if model_df is None or model_df.empty:
        return pd.DataFrame()
    df = model_df[(model_df['metric'].astype(str).eq(metric)) & (model_df['discovery_selected'].fillna(False))].copy()
    if df.empty:
        print(f'No discovery-selected fine pairs for metric={metric}')
        return pd.DataFrame()
    if 'abs_spearman_rho' not in df.columns:
        df['abs_spearman_rho'] = df['spearman_rho'].abs()
    df['aging_direction'] = np.where(df['spearman_rho'] > 0, 'aging-up', np.where(df['spearman_rho'] < 0, 'aging-down', 'flat'))
    # User-specified rule: choose the largest correlations first, not the smallest P values first.
    up = df[df['aging_direction'].eq('aging-up')].sort_values(
        ['spearman_rho', 'spearman_p', 'donor_frac'], ascending=[False, True, False]
    ).head(top_each)
    down = df[df['aging_direction'].eq('aging-down')].sort_values(
        ['abs_spearman_rho', 'spearman_p', 'donor_frac'], ascending=[False, True, False]
    ).head(top_each)
    notes = []
    if len(up) < top_each:
        notes.append(f'Only {len(up)} aging-up fine pairs passed strict mean-strength discovery; no extra pair was forced in.')
        print(notes[-1])
    if len(down) < top_each:
        notes.append(f'Only {len(down)} aging-down fine pairs passed strict mean-strength discovery; no extra pair was forced in.')
        print(notes[-1])
    out = pd.concat([up, down], ignore_index=True)
    if out.empty:
        return out
    out['top_pair_rank'] = np.arange(1, len(out) + 1)
    out['selection_note'] = '; '.join(notes) if notes else 'top 2 aging-up and top 2 aging-down strict mean-strength discovery'
    return out


# Exactly the four continuous communication pairs for downstream LR-driver interpretation:
# top 2 aging-up + top 2 aging-down at the finest new_clusters level.
top4_continuous_pairs = select_top_directional_fine_pairs(fine_models, top_each=2, metric='mean_strength')
top4_continuous_pairs.to_csv(TAB_DIR / 'S6_top2_up_top2_down_fine_pairs.tsv', sep='\t', index=False)
top4_continuous_pairs.to_csv(TAB_DIR / 'S7_top4_continuous_fine_pairs.tsv', sep='\t', index=False)
top4_pair_names = top4_continuous_pairs['cellpair'].astype(str).tolist() if not top4_continuous_pairs.empty else []

selected_fine_pairs = top4_pair_names.copy()
selected_fine_pair_table = top4_continuous_pairs.copy()
selected_fine_pair_table.to_csv(TAB_DIR / 'S5_selected_fine_pairs_for_LR.tsv', sep='\t', index=False)
selected_fine_pair_table.to_csv(TAB_DIR / 'S4_selected_fine_pairs_for_LR_continuous.tsv', sep='\t', index=False)

lr_source = lr_agg[lr_agg['cellpair'].astype(str).isin(selected_fine_pairs)].copy() if selected_fine_pairs else pd.DataFrame()
lr_models = dual_track(lr_source, 'cellpair_lr', ['sum_strength'])
if not lr_models.empty:
    lr_merge_cols = [c for c in [
        'cellpair_lr', 'cellpair', 'lr_pair', 'sender', 'receiver', 'classification',
        'directionality', 'interacting_pair', 'ligand', 'receptor', 'has_gene_symbol_mapping'
    ] if c in lr_source.columns]
    lr_models = lr_models.merge(
        lr_source[lr_merge_cols].drop_duplicates('cellpair_lr'),
        on='cellpair_lr', how='left'
    )
lr_models.to_csv(TAB_DIR / 'S7_lr_driver_models.tsv', sep='\t', index=False)
selected_lr = lr_models[lr_models['discovery_selected'].fillna(False)].copy() if not lr_models.empty else pd.DataFrame()
selected_lr.to_csv(TAB_DIR / 'S7_selected_lr_continuous.tsv', sep='\t', index=False)

print('Top continuous fine pairs for LR driver interpretation:', top4_pair_names)
if not top4_continuous_pairs.empty:
    print('Top fine-pair directions:', top4_continuous_pairs['aging_direction'].value_counts().to_dict())
print('LR model rows inside top pairs:', len(lr_models))
print('Discovery-selected LR drivers inside top pairs:', len(selected_lr))
if not selected_lr.empty:
    print('Selected LR directions:', selected_lr['aging_direction'].value_counts().to_dict())


# LR driver exploration and CPDB interaction-family enrichment among selected LR drivers.
def _lr_direction_table(lr_df):
    if lr_df is None or lr_df.empty:
        return pd.DataFrame()
    out = lr_df.copy()
    out['aging_direction'] = np.where(out['spearman_rho'] >= 0, 'aging-up', 'aging-down')
    cols = ['cellpair_lr','cellpair','lr_pair','sender','receiver','classification','metric','aging_direction',
            'spearman_rho','spearman_p','n_donors_modeled','n_nonzero_donors','donor_frac',
            'ordered_curve_pattern','residual_spearman_rho','residual_spearman_p','beta_10y','p_age',
            'discovery_selected','interpretation_tier','recommended_location']
    sort_cols = [c for c in ['cellpair','aging_direction','spearman_p','spearman_rho'] if c in out.columns]
    return out[[c for c in cols if c in out.columns]].sort_values(sort_cols)


def cpdb_lr_class_enrichment(selected_df, background_df, class_col='classification', entity_col='cellpair_lr'):
    if selected_df is None or selected_df.empty or background_df is None or background_df.empty:
        return pd.DataFrame(columns=['aging_direction','classification','selected_n','background_n','selected_total','background_total','fisher_p','odds_ratio','minus_log10_p'])
    bg = background_df[[entity_col, class_col]].drop_duplicates().copy()
    bg[class_col] = bg[class_col].fillna('Unannotated').astype(str)
    bg_entities = set(bg[entity_col].astype(str))
    rows = []
    sel = selected_df.copy()
    sel['aging_direction'] = np.where(sel['spearman_rho'] >= 0, 'aging-up', 'aging-down')
    sel = sel[[entity_col, class_col, 'aging_direction']].drop_duplicates().copy()
    sel[class_col] = sel[class_col].fillna('Unannotated').astype(str)
    for direction, sdf in sel.groupby('aging_direction'):
        selected_entities = set(sdf[entity_col].astype(str)) & bg_entities
        if not selected_entities:
            continue
        selected_total = len(selected_entities)
        background_total = len(bg_entities)
        for cls in sorted(bg[class_col].dropna().unique()):
            cls_bg = set(bg.loc[bg[class_col].eq(cls), entity_col].astype(str))
            a = len(selected_entities & cls_bg)
            if a == 0:
                continue
            b = selected_total - a
            c = len(cls_bg - selected_entities)
            d = max(background_total - selected_total - c, 0)
            try:
                odds, pval = st.fisher_exact([[a, b], [c, d]], alternative='greater')
            except Exception:
                odds, pval = np.nan, np.nan
            rows.append({'aging_direction': direction, 'classification': cls,
                         'selected_n': a, 'background_n': len(cls_bg),
                         'selected_total': selected_total, 'background_total': background_total,
                         'fisher_p': pval, 'odds_ratio': odds})
    out = pd.DataFrame(rows)
    if out.empty:
        return pd.DataFrame(columns=['aging_direction','classification','selected_n','background_n','selected_total','background_total','fisher_p','odds_ratio','minus_log10_p'])
    out['minus_log10_p'] = -np.log10(out['fisher_p'].clip(lower=1e-300))
    return out.sort_values(['aging_direction','fisher_p','selected_n'], ascending=[True, True, False])


selected_lr_exploration = _lr_direction_table(selected_lr)
selected_lr_exploration.to_csv(TAB_DIR / 'S7_selected_lr_continuous_exploration.tsv', sep='\t', index=False)

lr_class_enrichment = cpdb_lr_class_enrichment(selected_lr, lr_source if not lr_source.empty else lr_agg)
lr_class_enrichment.to_csv(TAB_DIR / 'S7_selected_lr_cpdb_classification_enrichment.tsv', sep='\t', index=False)

# All LR rows inside the four top pairs are kept for source traceability; discovery LR rows are highlighted.
top4_pair_lr_drivers = lr_models[lr_models['cellpair'].astype(str).isin(top4_pair_names)].copy() if (not lr_models.empty and top4_pair_names) else pd.DataFrame()
if not top4_pair_lr_drivers.empty:
    if 'abs_spearman_rho' not in top4_pair_lr_drivers.columns:
        top4_pair_lr_drivers['abs_spearman_rho'] = top4_pair_lr_drivers['spearman_rho'].abs()
    top4_pair_lr_drivers['aging_direction'] = np.where(top4_pair_lr_drivers['spearman_rho'] > 0, 'aging-up', np.where(top4_pair_lr_drivers['spearman_rho'] < 0, 'aging-down', 'flat'))
    pair_rank = top4_continuous_pairs[['cellpair', 'top_pair_rank', 'aging_direction', 'spearman_rho', 'spearman_p', 'donor_frac']].rename(
        columns={'aging_direction': 'pair_aging_direction', 'spearman_rho': 'pair_spearman_rho', 'spearman_p': 'pair_spearman_p', 'donor_frac': 'pair_donor_frac'})
    top4_pair_lr_drivers = top4_pair_lr_drivers.merge(pair_rank, on='cellpair', how='left')
    top4_pair_lr_drivers = top4_pair_lr_drivers.sort_values(['top_pair_rank', 'discovery_selected', 'spearman_p', 'abs_spearman_rho'], ascending=[True, False, True, False])
    top4_pair_lr_drivers['lr_rank_within_pair'] = top4_pair_lr_drivers.groupby('cellpair').cumcount() + 1
    top4_pair_lr_drivers['top_driver_for_plot'] = top4_pair_lr_drivers['lr_rank_within_pair'] <= 8
else:
    top4_pair_lr_drivers = pd.DataFrame()

top4_pair_lr_drivers.to_csv(TAB_DIR / 'S7_top4_pair_LR_drivers.tsv', sep='\t', index=False)
print('LR exploration direction counts:', selected_lr_exploration['aging_direction'].value_counts().to_dict() if not selected_lr_exploration.empty else {})
print('LR CPDB classification enrichment rows:', len(lr_class_enrichment))
print('Top4-pair LR drivers:', top4_pair_lr_drivers.shape if not top4_pair_lr_drivers.empty else (0, 0))
if not lr_class_enrichment.empty:
    display(lr_class_enrichment.head(20))


## Continuous LR driver functional enrichment

This section is the functional interpretation of continuous-age LR drivers. It is separate from CPDB communication-category summaries and separate from grouped FastCCC-style application plots.

In [ ]:

# Continuous LR driver functional enrichment.
# This cell keeps two layers separate:
# 1) an optional global selected-LR enrichment summary;
# 2) the main pair-specific enrichment for the top 2 up + top 2 down communication pairs.

import re

CONTINUOUS_ENRICHMENT_LIBRARIES = ['GO_Biological_Process_2023', 'Reactome_2022']


def _annotation_gene_list(value):
    if pd.isna(value):
        return []
    text = str(value).strip()
    if not text or text.lower() in {'nan', 'none', 'null', 'na'}:
        return []
    genes = []
    for tok in re.split(r'[+;&|,/\s]+', text):
        tok = tok.strip()
        if not tok or tok.lower() in {'nan', 'none', 'null', 'na'}:
            continue
        # Do not pass CPDB interaction IDs to GO/Reactome.
        if re.match(r'^CPI[-_]', tok, flags=re.IGNORECASE):
            continue
        if tok not in genes:
            genes.append(tok)
    return genes


def split_lr_genes_continuous(lr_pair):
    """Fallback parser for LR strings that already look like GENE_RECEPTOR.
    CPDB CPI ids cannot be split into genes and will return empty lists.
    """
    text = str(lr_pair)
    if re.match(r'^CPI[-_]', text, flags=re.IGNORECASE):
        return [], []
    if '_' not in text:
        return _annotation_gene_list(text), []
    ligand_part, receptor_part = text.split('_', 1)
    return _annotation_gene_list(ligand_part), _annotation_gene_list(receptor_part)


def resolve_lr_genes_from_row(row):
    ligands = _annotation_gene_list(row.get('ligand', np.nan))
    receptors = _annotation_gene_list(row.get('receptor', np.nan))
    if not ligands and not receptors:
        ligands, receptors = split_lr_genes_continuous(row.get('lr_pair', ''))
    return ligands, receptors


def _safe_gene_set_label(text):
    return re.sub(r'[^A-Za-z0-9]+', '_', str(text)).strip('_')


def build_continuous_lr_gene_sets(lr_df, rho_col='spearman_rho'):
    if lr_df is None or lr_df.empty:
        return pd.DataFrame(columns=['gene_set', 'aging_direction', 'side', 'gene', 'cellpair_lr', 'cellpair', 'lr_pair', 'sender', 'receiver', 'spearman_rho', 'spearman_p'])
    records = []
    unresolved = []
    for _, row in lr_df.iterrows():
        rho = row.get(rho_col, np.nan)
        if pd.isna(rho):
            continue
        direction = 'aging_up' if float(rho) > 0 else 'aging_down'
        ligands, receptors = resolve_lr_genes_from_row(row)
        if not ligands and not receptors:
            unresolved.append(row.get('lr_pair', np.nan))
        for side, genes in [('ligand', ligands), ('receptor', receptors)]:
            for gene in genes:
                records.append({
                    'gene_set': f'{direction}_{side}s',
                    'aging_direction': direction,
                    'side': side,
                    'gene': gene,
                    'cellpair_lr': row.get('cellpair_lr', np.nan),
                    'cellpair': row.get('cellpair', np.nan),
                    'lr_pair': row.get('lr_pair', np.nan),
                    'ligand': row.get('ligand', np.nan),
                    'receptor': row.get('receptor', np.nan),
                    'sender': row.get('sender', np.nan),
                    'receiver': row.get('receiver', np.nan),
                    'classification': row.get('classification', np.nan),
                    'spearman_rho': rho,
                    'spearman_p': row.get('spearman_p', np.nan),
                })
    if unresolved:
        pd.DataFrame({'unresolved_lr_pair': sorted(set(map(str, unresolved)))}).to_csv(
            TAB_DIR / 'S7_unresolved_CPI_ids_without_gene_mapping.tsv', sep='\t', index=False
        )
        print('Unresolved CPI/LR IDs without gene mapping:', len(set(map(str, unresolved))))
    return pd.DataFrame(records).drop_duplicates() if records else pd.DataFrame(columns=['gene_set', 'aging_direction', 'side', 'gene'])


def build_pair_specific_lr_gene_sets(lr_df, rho_col='spearman_rho'):
    if lr_df is None or lr_df.empty:
        return pd.DataFrame(columns=['gene_set', 'cellpair', 'pair_aging_direction', 'aging_direction', 'side', 'gene'])
    records = []
    unresolved = []
    for _, row in lr_df.iterrows():
        rho = row.get(rho_col, np.nan)
        if pd.isna(rho):
            continue
        cellpair = str(row.get('cellpair', 'unknown_pair'))
        cellpair_slug = _safe_gene_set_label(cellpair)
        direction = 'aging_up' if float(rho) > 0 else 'aging_down'
        pair_direction = row.get('pair_aging_direction', np.nan)
        ligands, receptors = resolve_lr_genes_from_row(row)
        if not ligands and not receptors:
            unresolved.append(row.get('lr_pair', np.nan))
        for side, genes in [('ligand', ligands), ('receptor', receptors)]:
            for gene in genes:
                records.append({
                    'gene_set': f'{cellpair_slug}__{direction}_{side}s',
                    'cellpair': cellpair,
                    'pair_aging_direction': pair_direction,
                    'aging_direction': direction,
                    'side': side,
                    'gene': gene,
                    'cellpair_lr': row.get('cellpair_lr', np.nan),
                    'lr_pair': row.get('lr_pair', np.nan),
                    'ligand': row.get('ligand', np.nan),
                    'receptor': row.get('receptor', np.nan),
                    'sender': row.get('sender', np.nan),
                    'receiver': row.get('receiver', np.nan),
                    'classification': row.get('classification', np.nan),
                    'spearman_rho': rho,
                    'spearman_p': row.get('spearman_p', np.nan),
                    'pair_spearman_rho': row.get('pair_spearman_rho', np.nan),
                    'pair_spearman_p': row.get('pair_spearman_p', np.nan),
                })
    if unresolved:
        pd.DataFrame({'unresolved_lr_pair': sorted(set(map(str, unresolved)))}).to_csv(
            TAB_DIR / 'S7_top4_pair_unresolved_CPI_ids_without_gene_mapping.tsv', sep='\t', index=False
        )
        print('Top-pair unresolved CPI/LR IDs without gene mapping:', len(set(map(str, unresolved))))
    return pd.DataFrame(records).drop_duplicates() if records else pd.DataFrame(columns=['gene_set', 'cellpair', 'aging_direction', 'side', 'gene'])


def run_continuous_lr_enrichr(gene_sets, libraries=CONTINUOUS_ENRICHMENT_LIBRARIES, min_genes=3):
    if gene_sets is None or gene_sets.empty:
        return pd.DataFrame()
    try:
        import gseapy as gp
    except Exception as exc:
        print('gseapy is not available; saved gene-set table only:', exc)
        return pd.DataFrame()

    results = []
    skipped = []
    for gene_set, sub in gene_sets.groupby('gene_set'):
        genes = sorted(set(sub['gene'].dropna().astype(str)))
        meta = {c: sub[c].dropna().astype(str).iloc[0] for c in ['cellpair', 'pair_aging_direction', 'aging_direction', 'side'] if c in sub.columns and sub[c].notna().any()}
        if len(genes) < min_genes:
            skipped.append({'gene_set': gene_set, 'reason': f'only {len(genes)} genes', **meta})
            continue
        for lib in libraries:
            try:
                enr = gp.enrichr(gene_list=genes, gene_sets=lib, organism='human', outdir=None, cutoff=1.0, no_plot=True)
                res = enr.results.copy() if hasattr(enr, 'results') and enr.results is not None else pd.DataFrame()
            except Exception as exc:
                print(f'Enrichr failed for {gene_set} / {lib}: {exc}')
                continue
            if res.empty:
                continue
            res['gene_set'] = gene_set
            res['library'] = lib
            res['input_gene_n'] = len(genes)
            for k, v in meta.items():
                res[k] = v
            results.append(res)
    if skipped:
        pd.DataFrame(skipped).to_csv(TAB_DIR / 'S7_GO_Reactome_enrichment_skipped_gene_sets.tsv', sep='\t', index=False)
    if not results:
        return pd.DataFrame()
    out = pd.concat(results, ignore_index=True)
    rename = {
        'Term': 'term',
        'P-value': 'enrichment_p',
        'Adjusted P-value': 'adjusted_p',
        'Overlap': 'overlap',
        'Genes': 'overlap_genes',
        'Combined Score': 'combined_score',
        'Odds Ratio': 'odds_ratio',
    }
    out = out.rename(columns={k: v for k, v in rename.items() if k in out.columns})
    if 'adjusted_p' in out.columns:
        out['adjusted_p'] = pd.to_numeric(out['adjusted_p'], errors='coerce')
    elif 'enrichment_p' in out.columns:
        # Fallback only for older enrichment outputs lacking adjusted p. New GO/Reactome plots use adjusted_p.
        out['adjusted_p'] = pd.to_numeric(out['enrichment_p'], errors='coerce')
    if 'adjusted_p' in out.columns:
        out['minus_log10_adjusted_p'] = -np.log10(out['adjusted_p'].clip(lower=1e-300))
        out['minus_log10_enrichment_p'] = out['minus_log10_adjusted_p']
    if 'overlap' in out.columns:
        out['overlap_n'] = out['overlap'].astype(str).str.extract(r'^(\d+)')[0].astype(float)
    elif 'overlap_genes' in out.columns:
        out['overlap_n'] = out['overlap_genes'].fillna('').astype(str).map(lambda x: len([g for g in re.split(r'[;,/\s]+', x) if g]))
    else:
        out['overlap_n'] = np.nan
    return out


def plot_continuous_lr_gene_enrichment(enrich_df, outfile='Figure5_continuous_LR_GO_Reactome_enrichment', top_n=18):
    if enrich_df is None or enrich_df.empty or 'term' not in enrich_df.columns:
        print('No continuous LR GO/Reactome enrichment results to plot')
        return None
    df = enrich_df.copy()
    p_col = 'adjusted_p' if 'adjusted_p' in df.columns else 'enrichment_p'
    df[p_col] = pd.to_numeric(df[p_col], errors='coerce')
    df = df.dropna(subset=[p_col])
    df['minus_log10_adjusted_p'] = -np.log10(df[p_col].clip(lower=1e-300))
    if df.empty:
        print('No valid adjusted enrichment P values to plot')
        return None
    df = (df.sort_values([p_col, 'overlap_n'], ascending=[True, False])
            .groupby(['aging_direction', 'side'], group_keys=False)
            .head(max(2, top_n // 4)))
    df = df.sort_values(['aging_direction', p_col], ascending=[True, False]).tail(top_n).copy()
    df['label'] = df['term'].astype(str).str.replace(r' \(GO:\d+\)$', '', regex=True).str.replace(r' R-HSA-\d+$', '', regex=True)
    df['label'] = df['label'].str.slice(0, 70)
    colors = np.where(df['aging_direction'].astype(str).eq('aging_up'), '#C84E4E', '#3B74AF')
    markers = {'ligand': 'o', 'receptor': '^'}
    fig, ax = plt.subplots(figsize=(6.7, max(3.2, 0.28 * len(df) + 1.2)))
    y = np.arange(len(df))
    for side, marker in markers.items():
        mask = df['side'].astype(str).eq(side)
        if mask.any():
            ax.scatter(df.loc[mask, 'minus_log10_adjusted_p'], y[mask],
                       s=np.clip(df.loc[mask, 'overlap_n'].fillna(1) * 34, 55, 300),
                       c=np.array(colors)[mask], marker=marker, edgecolor='white', linewidth=0.85, alpha=0.9, label=side)
    ax.set_yticks(y)
    ax.set_yticklabels(df['label'], fontsize=12)
    ax.set_xlabel('-log10 adjusted enrichment P')
    ax.set_ylabel('GO / Reactome term')
    ax.grid(axis='x', color='#E8E8E8', lw=1.05)
    ax.spines[['top', 'right']].set_visible(False)
    ax.legend(frameon=False, title='LR side')
    plt.tight_layout()
    save_fig(fig, outfile)
    df.to_csv(TAB_DIR / f'{outfile}_source.tsv', sep='\t', index=False)
    plt.show()
    return fig


# Optional global selected-LR enrichment companion.
continuous_lr_gene_sets = build_continuous_lr_gene_sets(selected_lr)
continuous_lr_gene_sets.to_csv(TAB_DIR / 'S7_selected_lr_continuous_aging_direction_gene_sets.tsv', sep='\t', index=False)
continuous_lr_functional_enrichment = run_continuous_lr_enrichr(continuous_lr_gene_sets)
continuous_lr_functional_enrichment.to_csv(TAB_DIR / 'S7_selected_lr_continuous_GO_Reactome_enrichment.tsv', sep='\t', index=False)
print('Global selected-LR GO/Reactome enrichment rows:', len(continuous_lr_functional_enrichment))

# Main pair-specific enrichment for Figure 5.
if 'top4_pair_lr_drivers' in globals() and top4_pair_lr_drivers is not None and not top4_pair_lr_drivers.empty:
    if 'discovery_selected' in top4_pair_lr_drivers.columns and top4_pair_lr_drivers['discovery_selected'].fillna(False).any():
        top4_pair_lr_drivers_for_enrichment = top4_pair_lr_drivers[top4_pair_lr_drivers['discovery_selected'].fillna(False)].copy()
    else:
        top4_pair_lr_drivers_for_enrichment = pd.DataFrame()
else:
    top4_pair_lr_drivers_for_enrichment = pd.DataFrame()

top4_pair_lr_gene_sets = build_pair_specific_lr_gene_sets(top4_pair_lr_drivers_for_enrichment)
top4_pair_lr_gene_sets.to_csv(TAB_DIR / 'S7_top4_pair_LR_gene_sets.tsv', sep='\t', index=False)
print('Pair-specific LR gene sets:', top4_pair_lr_gene_sets.groupby(['cellpair','gene_set'])['gene'].nunique().head(20).to_dict() if not top4_pair_lr_gene_sets.empty else {})

top4_pair_lr_functional_enrichment = run_continuous_lr_enrichr(top4_pair_lr_gene_sets)
top4_pair_lr_functional_enrichment.to_csv(TAB_DIR / 'S7_top4_pair_GO_Reactome_enrichment.tsv', sep='\t', index=False)
print('Pair-specific GO/Reactome enrichment rows:', len(top4_pair_lr_functional_enrichment))


## Figures, network roles, grouped companion, and evidence summary

In [ ]:

# Network role metrics from donor-level fine-cell communication networks.
# This is a continuous-age analysis layer, not part of grouped FastCCC Fig.4 companion plots.
try:
    display
except NameError:
    try:
        from IPython.display import display
    except Exception:
        def display(x):
            print(x)


NETWORK_ROLE_VALUE_COL = 'mean_strength'
NETWORK_ROLE_METRICS = [
    'sender_strength',
    'receiver_strength',
    'pagerank',
    'hub_score',
    'authority_score',
    'mediator',
    'net_flow',
]


def compute_network_panel(pair_panel, value_col=NETWORK_ROLE_VALUE_COL):
    if pair_panel is None or pair_panel.empty:
        return pd.DataFrame()
    try:
        import networkx as nx
    except Exception:
        print('networkx unavailable; skipping network roles')
        return pd.DataFrame()

    rows = []
    panel = pair_panel.copy()
    panel['donor_id'] = panel['donor_id'].astype(str)
    all_ct = sorted(set(panel['sender'].dropna().astype(str)).union(set(panel['receiver'].dropna().astype(str))))
    donor_meta = donor_base.assign(donor_id=lambda x: x['donor_id'].astype(str)).drop_duplicates('donor_id').set_index('donor_id')

    for donor_id, sub in panel.groupby('donor_id', observed=True):
        if str(donor_id) not in donor_meta.index:
            continue
        G = nx.DiGraph()
        G.add_nodes_from(all_ct)
        for _, r in sub.iterrows():
            try:
                w = float(r[value_col])
            except Exception:
                w = 0.0
            if np.isfinite(w) and w > 0:
                G.add_edge(str(r['sender']), str(r['receiver']), weight=w, distance=1.0 / (w + 1e-12))

        out_s = dict(G.out_degree(weight='weight'))
        in_s = dict(G.in_degree(weight='weight'))
        if G.number_of_edges() > 0:
            try:
                pr = nx.pagerank(G, weight='weight')
            except Exception:
                pr = {n: 0.0 for n in all_ct}
            try:
                hub, auth = nx.hits(G, max_iter=1000, normalized=True)
            except Exception:
                hub = {n: 0.0 for n in all_ct}; auth = {n: 0.0 for n in all_ct}
            try:
                btw = nx.betweenness_centrality(G, weight='distance', normalized=True)
            except Exception:
                btw = {n: 0.0 for n in all_ct}
        else:
            pr = {n: 0.0 for n in all_ct}; hub = {n: 0.0 for n in all_ct}; auth = {n: 0.0 for n in all_ct}; btw = {n: 0.0 for n in all_ct}

        meta = donor_meta.loc[str(donor_id)]
        for ct in all_ct:
            sender_strength = float(out_s.get(ct, 0.0))
            receiver_strength = float(in_s.get(ct, 0.0))
            rows.append({
                'donor_id': str(donor_id),
                'age': meta['age'],
                'age_group': meta['age_group'],
                'celltype': ct,
                'sender_strength': sender_strength,
                'receiver_strength': receiver_strength,
                'pagerank': float(pr.get(ct, 0.0)),
                'hub_score': float(hub.get(ct, 0.0)),
                'authority_score': float(auth.get(ct, 0.0)),
                'mediator': float(btw.get(ct, 0.0)),
                'net_flow': sender_strength - receiver_strength,
            })
    return pd.DataFrame(rows)


# Fine-level network roles preserve the main continuous new_clusters annotation.
network_panel = compute_network_panel(fine_pair_full, NETWORK_ROLE_VALUE_COL)
if not network_panel.empty:
    role_long = network_panel.melt(
        id_vars=['donor_id', 'age', 'age_group', 'celltype'],
        value_vars=NETWORK_ROLE_METRICS,
        var_name='metric',
        value_name='value',
    )
    role_long['role_entity'] = role_long['celltype'].astype(str) + '::' + role_long['metric'].astype(str)
    role_long['sender_fraction'] = 0.0
    role_long['receiver_fraction'] = 0.0
    role_long['log_total_cells'] = role_long['donor_id'].map(donor_base.set_index('donor_id')['log_total_cells'])

    role_model_panel = network_panel.copy()
    role_model_panel['sender_fraction'] = 0.0
    role_model_panel['receiver_fraction'] = 0.0
    role_model_panel['log_total_cells'] = role_model_panel['donor_id'].map(donor_base.set_index('donor_id')['log_total_cells'])
    role_models = dual_track(role_model_panel, 'celltype', NETWORK_ROLE_METRICS)
    if not role_models.empty:
        role_models['role_entity'] = role_models['celltype'].astype(str) + '::' + role_models['metric'].astype(str)
        role_models['role_metric'] = role_models['metric']
else:
    role_long = pd.DataFrame()
    role_models = pd.DataFrame()

network_panel.to_csv(TAB_DIR / 'S8_network_role_panel.tsv', sep='\t', index=False)
role_long.to_csv(TAB_DIR / 'S8_network_role_long.tsv', sep='\t', index=False)
role_models.to_csv(TAB_DIR / 'S8_network_role_models.tsv', sep='\t', index=False)

microglia_role_models = role_models[role_models['celltype'].astype(str).str.contains('micro', case=False, na=False)].copy() if not role_models.empty else pd.DataFrame()
microglia_role_models.to_csv(TAB_DIR / 'S8_microglia_network_role_models.tsv', sep='\t', index=False)

print('Network role panel:', network_panel.shape)
print('Network role long:', role_long.shape)
print('Network role models:', role_models.shape)
print('Microglia network role models:', microglia_role_models.shape)
if not role_models.empty:
    display(role_models.sort_values(['discovery_selected', 'spearman_p', 'abs_spearman_rho'], ascending=[False, True, False]).head(20))


In [ ]:
# Publication-style Figure 1-6 panels adapted to the Jeffries case-level notebook.
import matplotlib.patches as mpatches
# These figures mirror the test-(1) visual logic: overview -> broad/fine continuous trends
# -> communication categories -> LR drivers -> network roles. Grouped Fig.4h companion is in the next cell.

UP_COLOR = '#C44E52'
DOWN_COLOR = '#3B74AF'
NEUTRAL_COLOR = '#7A7A7A'
GRID_COLOR = '#E8E8E8'

mpl.rcParams.update({
    'font.family': 'sans-serif',
    'font.sans-serif': ['Arial', 'Helvetica', 'DejaVu Sans'],
    'font.size': 12,
    'axes.linewidth': 0.9,
    'axes.labelsize': 12,
    'axes.titlesize': 13,
    'axes.titleweight': 'regular',
    'axes.spines.top': False,
    'axes.spines.right': False,
    'xtick.major.width': 0.9,
    'ytick.major.width': 0.9,
    'xtick.major.size': 4.2,
    'ytick.major.size': 4.2,
    'xtick.direction': 'out',
    'ytick.direction': 'out',
    'legend.frameon': False,
    'pdf.fonttype': 42,
    'ps.fonttype': 42,
    'svg.fonttype': 'none',
})


def _panel_label(ax, label):
    ax.text(-0.12, 1.06, label, transform=ax.transAxes, fontsize=12,
            fontweight='bold', va='top', ha='right')


def _entity_label(text, max_len=52):
    text = str(text)
    return text


def _metric_label(metric):
    return {
        'mean_strength': 'Mean interaction strength',
        'sum_strength': 'Communication burden',
        'sig_lr_count': 'Detected LRI diversity',
        'mean_strength_lr': 'LR-level donor strength',
    }.get(metric, str(metric).replace('_', ' '))


def _model_metric_df(model_df, entity_col, metric='mean_strength', only_discovery=False):
    if model_df is None or model_df.empty:
        return pd.DataFrame()
    df = model_df[model_df['metric'].astype(str).eq(str(metric))].copy()
    if df.empty:
        return df
    if only_discovery and 'discovery_selected' in df.columns:
        df = df[df['discovery_selected'].fillna(False)].copy()
    if 'abs_spearman_rho' not in df.columns and 'spearman_rho' in df.columns:
        df['abs_spearman_rho'] = df['spearman_rho'].abs()
    if 'aging_direction' not in df.columns and 'spearman_rho' in df.columns:
        df['aging_direction'] = np.where(df['spearman_rho'] >= 0, 'aging-up', 'aging-down')
    sort_cols = []
    asc = []
    if 'discovery_selected' in df.columns:
        sort_cols.append('discovery_selected'); asc.append(False)
    if 'spearman_p' in df.columns:
        sort_cols.append('spearman_p'); asc.append(True)
    if 'abs_spearman_rho' in df.columns:
        sort_cols.append('abs_spearman_rho'); asc.append(False)
    if 'donor_frac' in df.columns:
        sort_cols.append('donor_frac'); asc.append(False)
    return df.sort_values(sort_cols, ascending=asc) if sort_cols else df


def select_entities(model_df, entity_col, metric='mean_strength', top_n=12, require_discovery=False, balance=True):
    df = _model_metric_df(model_df, entity_col, metric, only_discovery=require_discovery)
    if df.empty and require_discovery:
        df = _model_metric_df(model_df, entity_col, metric, only_discovery=False)
    if df.empty:
        return []
    if balance and 'spearman_rho' in df.columns:
        up = df[df['spearman_rho'] > 0].head(max(1, top_n // 2))
        down = df[df['spearman_rho'] < 0].head(max(1, top_n // 2))
        chosen = pd.concat([up, down], ignore_index=False)
        if len(chosen) < top_n:
            chosen = pd.concat([chosen, df[~df.index.isin(chosen.index)].head(top_n - len(chosen))])
        df = chosen.sort_values(['spearman_p', 'abs_spearman_rho'], ascending=[True, False]) if 'spearman_p' in chosen.columns else chosen
    else:
        df = df.head(top_n)
    return df[entity_col].astype(str).drop_duplicates().head(top_n).tolist()


def _case_order():
    cols = ['donor_id', 'age', 'age_group']
    return donor_base[cols].drop_duplicates('donor_id').sort_values(['age', 'donor_id']).copy()


def _source_save(df, name):
    if df is not None and not df.empty:
        df.to_csv(TAB_DIR / f'{name}_source.tsv', sep='\t', index=False)


def _annotate_trend(ax, model_df, entity_col, entity, metric):
    if model_df is None or model_df.empty:
        return
    row = model_df[(model_df[entity_col].astype(str).eq(str(entity))) & (model_df['metric'].astype(str).eq(str(metric)))]
    if row.empty:
        return
    r = row.iloc[0]
    label = f"rho={r.get('spearman_rho', np.nan):.2f}; P={r.get('spearman_p', np.nan):.2g}"
    if pd.notna(r.get('n_nonzero_donors', np.nan)):
        label += f"; nonzero={int(r.get('n_nonzero_donors'))}/{int(r.get('n_donors_modeled'))}"
    ax.text(0.02, 0.98, label, transform=ax.transAxes, va='top', ha='left', fontsize=12,
            bbox=dict(boxstyle='round,pad=0.12', fc='white', ec='#DDDDDD', lw=1.05, alpha=0.86))


def _plot_lollipop_ax(ax, model_df, entity_col, metric, top_n=14, title=None):
    df = _model_metric_df(model_df, entity_col, metric).head(top_n).copy()
    if df.empty:
        ax.text(0.5, 0.5, 'No trend table', ha='center', va='center')
        ax.axis('off')
        return df
    df = df.sort_values('spearman_rho')
    y = np.arange(len(df))
    colors = np.where(df['spearman_rho'] >= 0, UP_COLOR, DOWN_COLOR)
    sizes = np.clip(-np.log10(df['spearman_p'].clip(lower=1e-300)) * 28, 28, 220)
    ax.hlines(y, 0, df['spearman_rho'], color='#CFCFCF', lw=1.0, zorder=1)
    ax.scatter(df['spearman_rho'], y, s=sizes, c=colors, edgecolors='white', linewidths=0.755, zorder=3)
    ax.axvline(0, color='#333333', lw=1.0, ls='--')
    ax.set_yticks(y)
    ax.set_yticklabels([_entity_label(x, 38) for x in df[entity_col]], fontsize=12)
    ax.set_xlabel('Spearman rho with age')
    ax.grid(axis='x', color=GRID_COLOR, lw=1.05)
    return df


def _plot_trajectory_axes(parent_spec, fig, panel, model_df, entity_col, metric, entities, max_panels=4):
    import matplotlib.gridspec as gridspec
    entities = list(entities)[:max_panels]
    if not entities:
        ax = fig.add_subplot(parent_spec)
        ax.text(0.5, 0.5, 'No selected entities', ha='center', va='center')
        ax.axis('off')
        return []
    subgs = gridspec.GridSpecFromSubplotSpec(2, 2, subplot_spec=parent_spec, wspace=0.55, hspace=0.72)
    axes = []
    for k in range(4):
        ax = fig.add_subplot(subgs[k // 2, k % 2])
        axes.append(ax)
        if k >= len(entities):
            ax.axis('off')
            continue
        ent = entities[k]
        sub = panel[panel[entity_col].astype(str).eq(str(ent))].copy()
        if sub.empty:
            ax.text(0.5, 0.5, 'No data', ha='center', va='center')
            ax.axis('off')
            continue
        ax.scatter(sub['age'], sub[metric], c=sub['age_group'].map(AGE_PAL), s=60,
                   alpha=0.85, edgecolors='white', linewidths=0.655)
        if len(sub) >= 4 and sub[metric].nunique() > 1:
            sns.regplot(data=sub, x='age', y=metric, lowess=True, scatter=False,
                        color='#2F2F2F', line_kws={'lw': 1.2}, ax=ax)
        ax.set_title(_entity_label(ent, 44), loc='left', fontsize=12, pad=10)
        _annotate_trend(ax, model_df, entity_col, ent, metric)
        ax.set_xlabel('Age')
        ax.set_ylabel(_metric_label(metric))
        ax.grid(color=GRID_COLOR, lw=1.05)
    return axes


def _plot_ordered_heatmap_ax(ax, panel, entities, entity_col, value_col, title=None, clip=(-2, 3)):
    if panel is None or panel.empty or not entities:
        ax.text(0.5, 0.5, 'No heatmap data', ha='center', va='center')
        ax.axis('off')
        return pd.DataFrame()
    order = _case_order()
    mat = (panel[panel[entity_col].astype(str).isin([str(e) for e in entities])]
           .pivot_table(index=entity_col, columns='donor_id', values=value_col, aggfunc='mean', fill_value=0.0))
    ordered_cols = [x for x in order['donor_id'].astype(str).tolist() if x in mat.columns]
    mat = mat.reindex(index=[e for e in entities if e in mat.index], columns=ordered_cols)
    z = mat.sub(mat.mean(axis=1), axis=0).div(mat.std(axis=1).replace(0, np.nan), axis=0).fillna(0.0)
    z = z.clip(clip[0], clip[1])
    sns.heatmap(z, ax=ax, cmap='RdBu_r', center=0, vmin=clip[0], vmax=clip[1],
                cbar_kws={'label': 'row z-score', 'shrink': 0.65}, linewidths=0.15, linecolor='#F0F0F0')
    ax.set_xlabel('Cases ordered by age')
    ax.set_ylabel('')
    ax.set_yticklabels([_entity_label(t.get_text(), 36) for t in ax.get_yticklabels()], fontsize=12)
    ax.set_xticks([])
    return z.reset_index()


def plot_layer_figure(panel, model_df, entity_col, metric, title, outfile, top_n=12, trend_n=4):
    entities = select_entities(model_df, entity_col, metric, top_n=top_n, require_discovery=False, balance=True)
    fig = plt.figure(figsize=(15.8, 10.2))
    gs = fig.add_gridspec(2, 3, width_ratios=[1.05, 1.15, 1.15], height_ratios=[1.05, 1.0], wspace=0.58, hspace=0.58)
    ax_a = fig.add_subplot(gs[:, 0])
    source_a = _plot_lollipop_ax(ax_a, model_df, entity_col, metric, top_n=top_n)
    _panel_label(ax_a, 'a')
    trend_axes = _plot_trajectory_axes(gs[0, 1:], fig, panel, model_df, entity_col, metric, entities, max_panels=trend_n)
    if trend_axes:
        _panel_label(trend_axes[0], 'b')
    ax_c = fig.add_subplot(gs[1, 1:])
    source_c = _plot_ordered_heatmap_ax(ax_c, panel, entities[:min(top_n, 14)], entity_col, metric)
    _panel_label(ax_c, 'c')
    if title:
        fig.suptitle(title, fontsize=12, fontweight='bold', y=0.995)
    plt.tight_layout(rect=[0.02, 0.02, 0.98, 0.92])
    save_fig(fig, outfile)
    _source_save(source_a, outfile + '_lollipop')
    _source_save(source_c, outfile + '_heatmap')
    plt.show()
    return fig


def plot_figure1_overview():
    db = donor_base.copy().sort_values(['age', 'donor_id'])
    fig = plt.figure(figsize=(11.5, 7.4))
    gs = fig.add_gridspec(2, 2, width_ratios=[1.0, 1.55], height_ratios=[1.0, 1.15], wspace=0.34, hspace=0.38)

    ax1 = fig.add_subplot(gs[0, 0])
    for g in AGE_ORDER:
        sub = db[db['age_group'].astype(str).eq(g)]
        ax1.scatter(sub['age'], np.zeros(len(sub)) + AGE_ORDER.index(g), s=55,
                    color=AGE_PAL[g], label=g, edgecolors='white', linewidths=0.75)
    ax1.set_yticks(range(len(AGE_ORDER)))
    ax1.set_yticklabels(AGE_ORDER)
    ax1.set_xlabel('Age')
    ax1.grid(axis='x', color=GRID_COLOR)
    _panel_label(ax1, 'a')

    ax2 = fig.add_subplot(gs[1, 0])
    counts = db['age_group'].value_counts().reindex(AGE_ORDER).fillna(0)
    ax2.bar(counts.index.astype(str), counts.values, color=[AGE_PAL[g] for g in counts.index])
    ax2.set_ylabel('Cases')
    ax2.tick_params(axis='x', rotation=30)
    _panel_label(ax2, 'b')

    ax3 = fig.add_subplot(gs[:, 1])
    if 'ct_frac_fine_df' in globals() and not ct_frac_fine_df.empty:
        comp = ct_frac_fine_df.reindex(db['donor_id'].astype(str)).fillna(0.0)
        keep = comp.mean().sort_values(ascending=False).head(12).index.tolist()
        comp = comp[keep]
        bottom = np.zeros(comp.shape[0])
        palette = sns.color_palette('tab20', n_colors=len(keep))
        x = np.arange(comp.shape[0])
        for color, ct in zip(palette, keep):
            vals = comp[ct].to_numpy(float)
            ax3.bar(x, vals, bottom=bottom, width=0.86, color=color, label=ct, linewidth=0)
            bottom += vals
        ax3.set_xticks(x)
        ax3.set_xticklabels([f'{a:g}' for a in db['age'].astype(float)], rotation=90, fontsize=12)
        ax3.set_ylabel('new_clusters fine-cluster fraction')
        ax3.set_xlabel('Cases ordered by age')
        ax3.legend(bbox_to_anchor=(1.01, 1), loc='upper left', fontsize=12, title='new_clusters fine cluster')
    else:
        ax3.text(0.5, 0.5, 'No broad composition table', ha='center', va='center')
        ax3.axis('off')
    _panel_label(ax3, 'c')
    fig.suptitle('Figure 1. Donor age distribution and fine-cluster composition', fontsize=12, fontweight='bold', y=0.995)
    plt.tight_layout(rect=[0.02, 0.02, 0.98, 0.92])
    save_fig(fig, 'Figure1_dataset_overview')
    db.to_csv(TAB_DIR / 'Figure1_dataset_overview_source.tsv', sep='\t', index=False)
    plt.show()
    return fig


def plot_lr_figure():
    if lr_models is None or lr_models.empty:
        print('No LR model table available')
        return None
    plot_df = _model_metric_df(lr_models, 'cellpair_lr', 'sum_strength').head(18).copy()
    if plot_df.empty:
        print('No LR candidates available')
        return None
    entities = plot_df['cellpair_lr'].astype(str).tolist()[:8]
    fig = plt.figure(figsize=(15.8, 10.0))
    gs = fig.add_gridspec(2, 3, width_ratios=[1.15, 1.1, 1.1], height_ratios=[1.0, 1.0], wspace=0.58, hspace=0.62)
    ax_a = fig.add_subplot(gs[:, 0])
    source_a = _plot_lollipop_ax(ax_a, lr_models, 'cellpair_lr', 'sum_strength', top_n=18)
    _panel_label(ax_a, 'a')
    trend_axes = _plot_trajectory_axes(gs[0, 1:], fig, lr_agg, lr_models, 'cellpair_lr', 'sum_strength', entities, max_panels=4)
    if trend_axes:
        _panel_label(trend_axes[0], 'b')
    ax_c = fig.add_subplot(gs[1, 1:])
    source_c = _plot_ordered_heatmap_ax(ax_c, lr_agg, entities, 'cellpair_lr', 'sum_strength')
    _panel_label(ax_c, 'c')
    fig.suptitle('Figure 5. Continuous LR driver discovery', fontsize=12, fontweight='bold', y=0.995)
    plt.tight_layout(rect=[0.02, 0.02, 0.98, 0.92])
    save_fig(fig, 'Figure5_LR_driver_continuous_age')
    _source_save(source_a, 'Figure5_LR_driver_continuous_age_lollipop')
    _source_save(source_c, 'Figure5_LR_driver_continuous_age_heatmap')
    plt.show()
    return fig


def plot_network_figure():
    if role_models is None or role_models.empty or role_long is None or role_long.empty:
        print('No network role table available')
        return None
    plot_df = _model_metric_df(role_models, 'role_entity', 'mean_strength').head(14).copy()
    entities = plot_df['role_entity'].astype(str).tolist()[:8]
    fig = plt.figure(figsize=(15.8, 10.0))
    gs = fig.add_gridspec(2, 3, width_ratios=[1.1, 1.1, 1.1], height_ratios=[1.0, 1.0], wspace=0.55, hspace=0.62)
    ax_a = fig.add_subplot(gs[:, 0])
    source_a = _plot_lollipop_ax(ax_a, role_models, 'role_entity', 'mean_strength', top_n=14)
    _panel_label(ax_a, 'a')
    axes = _plot_trajectory_axes(gs[0, 1:], fig, role_long, role_models, 'role_entity', 'mean_strength', entities, max_panels=4)
    if axes:
        _panel_label(axes[0], 'b')
    ax_c = fig.add_subplot(gs[1, 1:])
    source_c = _plot_ordered_heatmap_ax(ax_c, role_long, entities, 'role_entity', 'mean_strength')
    _panel_label(ax_c, 'c')
    plt.tight_layout(rect=[0.02, 0.02, 0.98, 0.92])
    save_fig(fig, 'Figure6_network_role_continuous_age')
    _source_save(source_a, 'Figure6_network_role_continuous_age_lollipop')
    _source_save(source_c, 'Figure6_network_role_continuous_age_heatmap')
    plt.show()
    return fig




# ---- Overrides/additional paper-style panels requested for fine composition, LR enrichment, and network role shifts ----
def plot_figure1_overview():
    db = donor_base.copy().sort_values(['age', 'donor_id'])
    fig = plt.figure(figsize=(12.8, 8.0))
    gs = fig.add_gridspec(2, 2, width_ratios=[1.0, 1.75], height_ratios=[1.0, 1.15], wspace=0.34, hspace=0.38)

    ax1 = fig.add_subplot(gs[0, 0])
    for g in AGE_ORDER:
        sub = db[db['age_group'].astype(str).eq(g)]
        ax1.scatter(sub['age'], np.zeros(len(sub)) + AGE_ORDER.index(g), s=58,
                    color=AGE_PAL[g], label=g, edgecolors='white', linewidths=0.75)
    ax1.set_yticks(range(len(AGE_ORDER)))
    ax1.set_yticklabels(AGE_ORDER)
    ax1.set_xlabel('Age')
    ax1.grid(axis='x', color=GRID_COLOR)
    _panel_label(ax1, 'a')

    ax2 = fig.add_subplot(gs[1, 0])
    counts = db['age_group'].value_counts().reindex(AGE_ORDER).fillna(0)
    ax2.bar(counts.index.astype(str), counts.values, color=[AGE_PAL[g] for g in counts.index])
    ax2.set_ylabel('Cases')
    ax2.tick_params(axis='x', rotation=30)
    _panel_label(ax2, 'b')

    ax3 = fig.add_subplot(gs[:, 1])
    # Figure 1 composition uses the finest Jeffries annotation (new_clusters).
    # Grouped LRI display can use new_clusters2 later, but this overview stays fine-grained.
    if 'ct_frac_fine_df' in globals() and not ct_frac_fine_df.empty:
        comp = ct_frac_fine_df.reindex(db['donor_id'].astype(str)).fillna(0.0)
        comp_label = 'new_clusters fine-cluster fraction'
        legend_title = 'new_clusters fine cluster'
        source_name = 'Figure1C_fine_cluster_composition_source.tsv'
    else:
        comp = pd.DataFrame()
    if not comp.empty:
        keep = comp.mean().sort_values(ascending=False).index.tolist()
        comp = comp[keep]
        bottom = np.zeros(comp.shape[0])
        palette = sns.color_palette('tab20', n_colors=min(20, len(keep))) + sns.color_palette('Set3', n_colors=max(0, len(keep)-20))
        x = np.arange(comp.shape[0])
        for color, ct in zip(palette, keep):
            vals = comp[ct].to_numpy(float)
            ax3.bar(x, vals, bottom=bottom, width=0.86, color=color, label=ct, linewidth=0)
            bottom += vals
        ax3.set_xticks(x)
        ax3.set_xticklabels([f'{a:g}' for a in db['age'].astype(float)], rotation=90, fontsize=12)
        ax3.set_ylabel(comp_label)
        ax3.set_xlabel('Cases ordered by age')
        ax3.legend(bbox_to_anchor=(1.01, 1), loc='upper left', fontsize=12, title=legend_title, ncol=1)
        comp.reset_index(names='donor_id').to_csv(TAB_DIR / source_name, sep='\t', index=False)
    else:
        ax3.text(0.5, 0.5, 'No composition table', ha='center', va='center')
        ax3.axis('off')
    _panel_label(ax3, 'c')
    fig.suptitle('Figure 1. Donor age distribution and fine-cluster composition', fontsize=12, fontweight='bold', y=0.995)
    plt.tight_layout(rect=[0.02, 0.02, 0.98, 0.92])
    save_fig(fig, 'Figure1_dataset_overview')
    db.to_csv(TAB_DIR / 'Figure1_dataset_overview_source.tsv', sep='\t', index=False)
    plt.show()
    return fig


def plot_lr_class_enrichment_figure(enrich_df=None, outfile='Figure5D_LR_driver_CPDB_classification_enrichment', top_n=10):
    if enrich_df is None:
        enrich_df = globals().get('lr_class_enrichment', pd.DataFrame())
    if enrich_df is None or enrich_df.empty:
        print('No LR CPDB classification enrichment table available')
        return None
    df = enrich_df.copy()
    df['direction_label'] = df['aging_direction'].map({'aging-up': 'aging-up', 'aging-down': 'aging-down'}).fillna(df['aging_direction'].astype(str))
    parts = []
    for direction in ['aging-up', 'aging-down']:
        sub = df[df['direction_label'].eq(direction)].sort_values(['fisher_p','selected_n']).head(top_n).copy()
        parts.append(sub)
    plot_df = pd.concat(parts, ignore_index=True) if parts else pd.DataFrame()
    if plot_df.empty:
        return None
    plot_df['label'] = plot_df['classification'].astype(str)
    plot_df['minus_log10_p'] = plot_df['minus_log10_p'].replace([np.inf, -np.inf], np.nan).fillna(0.0)
    plot_df = plot_df.sort_values(['direction_label','minus_log10_p'], ascending=[True, True])
    y = np.arange(len(plot_df))
    colors = np.where(plot_df['direction_label'].eq('aging-up'), UP_COLOR, DOWN_COLOR)
    sizes = np.clip(plot_df['selected_n'].fillna(1).astype(float) * 38, 45, 320)
    fig, ax = plt.subplots(figsize=(7.6, max(4.2, 0.34 * len(plot_df) + 1.0)))
    ax.scatter(plot_df['minus_log10_p'], y, s=sizes, c=colors, edgecolors='white', linewidths=0.6)
    ax.set_yticks(y)
    ax.set_yticklabels([_entity_label(x, 55) for x in plot_df['label']], fontsize=12)
    ax.set_xlabel('-log10 nominal Fisher P')
    ax.grid(axis='x', color=GRID_COLOR, lw=1.05)
    handles = [mpatches.Patch(color=UP_COLOR, label='aging-up LR'), mpatches.Patch(color=DOWN_COLOR, label='aging-down LR')]
    ax.legend(handles=handles, loc='lower right')
    fig.suptitle('Figure 5D. CPDB interaction-family enrichment among LR drivers', fontsize=12, fontweight='bold', y=0.995)
    plt.tight_layout(rect=[0.02, 0.02, 0.98, 0.92])
    save_fig(fig, outfile)
    plot_df.to_csv(TAB_DIR / f'{outfile}_source.tsv', sep='\t', index=False)
    plt.show()
    return fig




def _network_model_rank_df(top_n=16):
    if role_models is None or role_models.empty:
        return pd.DataFrame()
    df = role_models.copy()
    if 'abs_spearman_rho' not in df.columns:
        df['abs_spearman_rho'] = df['spearman_rho'].abs()
    if 'role_entity' not in df.columns:
        df['role_entity'] = df['celltype'].astype(str) + '::' + df['metric'].astype(str)
    sort_cols = ['discovery_selected', 'spearman_p', 'abs_spearman_rho']
    return df.sort_values(sort_cols, ascending=[False, True, False]).head(top_n).copy()


def _plot_network_lollipop(ax, model_df, top_n=16):
    df = _network_model_rank_df(top_n)
    if df.empty:
        ax.text(0.5, 0.5, 'No network role model table', ha='center', va='center')
        ax.axis('off')
        return df
    df = df.sort_values('spearman_rho')
    y = np.arange(len(df))
    colors = np.where(df['spearman_rho'] >= 0, UP_COLOR, DOWN_COLOR)
    sizes = np.clip(-np.log10(df['spearman_p'].clip(lower=1e-300)) * 28, 28, 220)
    ax.hlines(y, 0, df['spearman_rho'], color='#CFCFCF', lw=1.0, zorder=1)
    ax.scatter(df['spearman_rho'], y, s=sizes, c=colors, edgecolors='white', linewidths=0.755, zorder=3)
    ax.axvline(0, color='#333333', lw=1.0, ls='--')
    ax.set_yticks(y)
    ax.set_yticklabels([f"{r.celltype} | {r.metric}" for r in df.itertuples()], fontsize=12)
    ax.set_xlabel('Spearman rho with age')
    ax.grid(axis='x', color=GRID_COLOR, lw=1.05)
    return df


def _plot_role_agebin_sem(ax, entity, ylabel=None):
    if role_long is None or role_long.empty:
        ax.axis('off')
        return pd.DataFrame()
    sub = role_long[role_long['role_entity'].astype(str).eq(str(entity))].copy()
    if sub.empty:
        ax.axis('off')
        return pd.DataFrame()
    metric = sub['metric'].iloc[0]
    celltype = sub['celltype'].iloc[0]
    xmap = {g: i for i, g in enumerate(AGE_ORDER)}
    # donor points jittered by age bin, so multiple donors in one age group remain visible
    rng = np.random.default_rng(7)
    x = sub['age_group'].astype(str).map(xmap).astype(float).to_numpy()
    jitter = rng.normal(0, 0.045, size=len(sub))
    ax.scatter(x + jitter, sub['value'], s=55, c=[AGE_PAL.get(g, '#777777') for g in sub['age_group'].astype(str)], alpha=0.55, edgecolors='none')
    stats = (sub.groupby('age_group', observed=False)['value']
             .agg(mean='mean', sem=lambda v: np.nanstd(v, ddof=1) / np.sqrt(np.sum(pd.notna(v))) if np.sum(pd.notna(v)) > 1 else 0.0)
             .reindex(AGE_ORDER).reset_index())
    xs = np.arange(len(AGE_ORDER))
    ax.errorbar(xs, stats['mean'], yerr=stats['sem'], color='#222222', lw=1.0, marker='o', ms=6, capsize=3.5, zorder=4)
    ax.set_xticks(xs)
    ax.set_xticklabels(AGE_ORDER, rotation=35, ha='right', fontsize=12)
    ax.set_title(f'{celltype}\n{metric}', fontsize=12, pad=2)
    if ylabel:
        ax.set_ylabel(ylabel)
    ax.grid(axis='y', color=GRID_COLOR, lw=1.05)
    ax.spines[['top', 'right']].set_visible(False)
    return stats.assign(role_entity=entity, celltype=celltype, metric=metric)


def _network_rank_transition_source(metrics=('sender_strength', 'receiver_strength', 'pagerank'), top_n=3):
    if network_panel is None or network_panel.empty:
        return pd.DataFrame()
    rows = []
    for metric in metrics:
        if metric not in network_panel.columns:
            continue
        gm = (network_panel.groupby(['age_group', 'celltype'], observed=False)[metric]
              .mean().reset_index().sort_values(['age_group', metric], ascending=[True, False]))
        for group in AGE_ORDER:
            sub = gm[gm['age_group'].astype(str).eq(group)].head(top_n)
            for rank, (_, r) in enumerate(sub.iterrows(), start=1):
                rows.append({'metric': metric, 'age_group': group, 'rank': rank, 'celltype': r['celltype'], 'value': r[metric]})
    return pd.DataFrame(rows)


def _plot_network_rank_transition(ax, metrics=('sender_strength', 'receiver_strength', 'pagerank'), top_n=3):
    src = _network_rank_transition_source(metrics, top_n)
    if src.empty:
        ax.text(0.5, 0.5, 'No rank transition data', ha='center', va='center')
        ax.axis('off')
        return src
    ax.set_xlim(-0.5, len(AGE_ORDER) - 0.5)
    ax.set_ylim(-0.5, len(metrics) - 0.5)
    ax.set_xticks(range(len(AGE_ORDER)))
    ax.set_xticklabels(AGE_ORDER, rotation=25, ha='right')
    ax.set_yticks(range(len(metrics)))
    ax.set_yticklabels([m.replace('_', ' ') for m in metrics])
    for mi, metric in enumerate(metrics):
        for gi, group in enumerate(AGE_ORDER):
            sub = src[(src['metric'].eq(metric)) & (src['age_group'].astype(str).eq(group))].sort_values('rank')
            label = '\n'.join([f"{int(r.rank)}. {r.celltype}" for r in sub.itertuples()])
            ax.text(gi, mi, label, ha='center', va='center', fontsize=12,
                    bbox=dict(boxstyle='round,pad=0.24', fc='white', ec='#DDDDDD', lw=1.05))
    ax.grid(color=GRID_COLOR, lw=1.05)
    ax.spines[['top', 'right']].set_visible(False)
    src.to_csv(TAB_DIR / 'Figure6_network_role_rank_transition_source.tsv', sep='\t', index=False)
    return src


def _plot_microglia_role_ax(ax):
    if microglia_role_models is None or microglia_role_models.empty:
        ax.text(0.5, 0.5, 'No microglia role trend passed modeling', ha='center', va='center')
        ax.axis('off')
        print('Microglia did not appear in network role model table.')
        return pd.DataFrame()
    df = microglia_role_models.copy()
    if 'abs_spearman_rho' not in df.columns:
        df['abs_spearman_rho'] = df['spearman_rho'].abs()
    df = df.sort_values(['spearman_p', 'abs_spearman_rho'], ascending=[True, False])
    y = np.arange(len(df))
    colors = np.where(df['spearman_rho'] >= 0, UP_COLOR, DOWN_COLOR)
    ax.axvline(0, color='#333333', lw=1.0, ls='--')
    ax.scatter(df['spearman_rho'], y, s=np.clip(-np.log10(df['spearman_p'].clip(lower=1e-300)) * 35, 35, 240),
               c=colors, edgecolors='white', linewidths=0.755)
    ax.set_yticks(y)
    ax.set_yticklabels(df['metric'].str.replace('_', ' '), fontsize=12)
    ax.set_xlabel('Spearman rho with age')
    ax.grid(axis='x', color=GRID_COLOR, lw=1.05)
    df.to_csv(TAB_DIR / 'Figure6_microglia_network_role_source.tsv', sep='\t', index=False)
    print('Microglia network role trends:', df[['metric', 'spearman_rho', 'spearman_p', 'donor_frac']].to_dict('records'))
    return df


def plot_network_figure():
    if role_models is None or role_models.empty or role_long is None or role_long.empty:
        print('No network role table available')
        return None
    fig = plt.figure(figsize=(16.5, 11.2))
    gs = fig.add_gridspec(2, 2, width_ratios=[1.0, 1.25], height_ratios=[1.0, 1.0], wspace=0.55, hspace=0.72)
    ax_a = fig.add_subplot(gs[0, 0])
    source_a = _plot_network_lollipop(ax_a, role_models, top_n=16)
    _panel_label(ax_a, 'a')

    top_entities = source_a.sort_values(['spearman_p', 'abs_spearman_rho'], ascending=[True, False])['role_entity'].astype(str).head(6).tolist() if not source_a.empty else []
    subgs = gs[0, 1].subgridspec(2, 3, wspace=0.62, hspace=0.92)
    sem_sources = []
    for i, entity in enumerate(top_entities):
        ax = fig.add_subplot(subgs[i // 3, i % 3])
        ss = _plot_role_agebin_sem(ax, entity, ylabel='Role value' if i % 3 == 0 else None)
        if not ss.empty:
            sem_sources.append(ss)
        if i == 0:
            _panel_label(ax, 'b')
    for j in range(len(top_entities), 6):
        ax = fig.add_subplot(subgs[j // 3, j % 3]); ax.axis('off')

    ax_c = fig.add_subplot(gs[1, 0])
    rank_src = _plot_network_rank_transition(ax_c)
    _panel_label(ax_c, 'c')

    ax_d = fig.add_subplot(gs[1, 1])
    micro_src = _plot_microglia_role_ax(ax_d)
    _panel_label(ax_d, 'd')

    plt.tight_layout(rect=[0.02, 0.02, 0.98, 0.92])
    save_fig(fig, 'Figure6_network_role_continuous_age')
    _source_save(source_a, 'Figure6_network_role_continuous_age_lollipop')
    if sem_sources:
        pd.concat(sem_sources, ignore_index=True).to_csv(TAB_DIR / 'Figure6_network_role_agebin_sem_source.tsv', sep='\t', index=False)
    if not rank_src.empty:
        rank_src.to_csv(TAB_DIR / 'Figure6_network_role_rank_transition_source.tsv', sep='\t', index=False)
    if not micro_src.empty:
        micro_src.to_csv(TAB_DIR / 'Figure6_microglia_network_role_source.tsv', sep='\t', index=False)
    plt.show()
    return fig


def plot_network_sender_receiver_shift(outfile='Figure6B_sender_receiver_role_shift'):
    if network_panel is None or network_panel.empty:
        print('No network_panel available for sender/receiver role shift')
        return None
    df = network_panel[network_panel['age_group'].isin(AGE_ORDER)].copy()
    if df.empty:
        return None
    group_mean = (df.groupby(['age_group', 'celltype'], observed=False)[['sender_strength', 'receiver_strength', 'pagerank', 'net_flow']]
                    .mean().reset_index())
    y = group_mean[group_mean['age_group'].astype(str).eq('Young adult')].set_index('celltype')
    o = group_mean[group_mean['age_group'].astype(str).eq('Elderly')].set_index('celltype')
    common = sorted(set(y.index) & set(o.index))
    if not common:
        return None
    shift = pd.DataFrame({
        'celltype': common,
        'young_sender': y.loc[common, 'sender_strength'].to_numpy(float),
        'young_receiver': y.loc[common, 'receiver_strength'].to_numpy(float),
        'elderly_sender': o.loc[common, 'sender_strength'].to_numpy(float),
        'elderly_receiver': o.loc[common, 'receiver_strength'].to_numpy(float),
        'young_pagerank': y.loc[common, 'pagerank'].to_numpy(float),
        'elderly_pagerank': o.loc[common, 'pagerank'].to_numpy(float),
        'young_net_flow': y.loc[common, 'net_flow'].to_numpy(float),
        'elderly_net_flow': o.loc[common, 'net_flow'].to_numpy(float),
    })
    shift['delta_sender'] = shift['elderly_sender'] - shift['young_sender']
    shift['delta_receiver'] = shift['elderly_receiver'] - shift['young_receiver']
    shift['shift_norm'] = np.sqrt(shift['delta_sender']**2 + shift['delta_receiver']**2)
    top_label = set(shift.sort_values('shift_norm', ascending=False).head(10)['celltype'])
    fig, ax = plt.subplots(figsize=(6.6, 5.8))
    ax.scatter(shift['young_sender'], shift['young_receiver'], s=52, c=AGE_PAL['Young adult'], alpha=0.85, label='Young adult')
    ax.scatter(shift['elderly_sender'], shift['elderly_receiver'], s=58, c=AGE_PAL['Elderly'], alpha=0.9, label='Elderly')
    for _, r in shift.iterrows():
        ax.annotate('', xy=(r['elderly_sender'], r['elderly_receiver']), xytext=(r['young_sender'], r['young_receiver']),
                    arrowprops=dict(arrowstyle='->', color='#777777', lw=1.0, alpha=0.65))
        if r['celltype'] in top_label:
            ax.text(r['elderly_sender'], r['elderly_receiver'], str(r['celltype']), fontsize=12, ha='left', va='bottom')
    ax.set_xlabel('Sender role (outgoing strength)')
    ax.set_ylabel('Receiver role (incoming strength)')
    ax.grid(color=GRID_COLOR, lw=1.05)
    ax.legend(loc='best')
    plt.tight_layout()
    save_fig(fig, outfile)
    shift.to_csv(TAB_DIR / f'{outfile}_source.tsv', sep='\t', index=False)
    plt.show()
    return fig




def _top_network_role_entities(top_n=12, balanced=True):
    df = _network_model_rank_df(max(top_n * 2, 18))
    if df.empty:
        return []
    if balanced:
        up = df[df['spearman_rho'] > 0].head(max(1, top_n // 2))
        down = df[df['spearman_rho'] < 0].head(max(1, top_n // 2))
        chosen = pd.concat([up, down], ignore_index=False)
        if len(chosen) < top_n:
            chosen = pd.concat([chosen, df[~df.index.isin(chosen.index)].head(top_n - len(chosen))])
        df = chosen.sort_values(['spearman_p', 'abs_spearman_rho'], ascending=[True, False])
    return df['role_entity'].astype(str).drop_duplicates().head(top_n).tolist()


def _annotate_network_role(ax, entity):
    if role_models is None or role_models.empty:
        return
    if 'role_entity' not in role_models.columns:
        return
    row = role_models[role_models['role_entity'].astype(str).eq(str(entity))]
    if row.empty:
        return
    r = row.iloc[0]
    txt = f"rho={r.get('spearman_rho', np.nan):.2f}; P={r.get('spearman_p', np.nan):.2g}"
    if pd.notna(r.get('n_nonzero_donors', np.nan)):
        txt += f"; nonzero={int(r.get('n_nonzero_donors'))}/{int(r.get('n_donors_modeled'))}"
    ax.text(0.02, 0.96, txt, transform=ax.transAxes, va='top', ha='left', fontsize=12,
            bbox=dict(boxstyle='round,pad=0.15', fc='white', ec='none', alpha=0.72))


def plot_network_role_trajectories(top_n=8, outfile='Figure6A_network_role_donor_level_trajectories'):
    if role_long is None or role_long.empty:
        print('No network role long table available')
        return None
    entities = _top_network_role_entities(top_n=top_n, balanced=True)
    ncols = 4
    nrows = int(np.ceil(max(1, len(entities)) / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(4.4 * ncols, 3.65 * nrows), squeeze=False)
    src_rows = []
    for i, ax in enumerate(axes.flat):
        if i >= len(entities):
            ax.axis('off'); continue
        ent = entities[i]
        sub = role_long[role_long['role_entity'].astype(str).eq(ent)].copy()
        if sub.empty:
            ax.axis('off'); continue
        ax.scatter(sub['age'], sub['value'], c=sub['age_group'].map(AGE_PAL), s=58, alpha=0.82, edgecolors='white', linewidths=0.655)
        if len(sub) >= 4 and sub['value'].nunique() > 1:
            sns.regplot(data=sub, x='age', y='value', lowess=True, scatter=False, color='#2F2F2F', line_kws={'lw': 1.2}, ax=ax)
        _annotate_network_role(ax, ent)
        ax.set_title(ent.replace('::', ' | '), fontsize=12)
        ax.set_xlabel('Age')
        ax.set_ylabel('Role value')
        ax.grid(color=GRID_COLOR, lw=1.05)
        src_rows.append(sub)
    fig.suptitle('Figure 6A. Network role donor-level trajectories', fontsize=12, fontweight='bold', y=0.995)
    plt.tight_layout(rect=[0.02, 0.02, 0.98, 0.92])
    save_fig(fig, outfile)
    if src_rows:
        pd.concat(src_rows, ignore_index=True).to_csv(TAB_DIR / f'{outfile}_source.tsv', sep='\t', index=False)
    plt.show()
    return fig


def plot_network_role_agebin_errorbars(top_n=8, outfile='Figure6B_network_role_agebin_errorbars'):
    entities = _top_network_role_entities(top_n=top_n, balanced=True)
    if not entities:
        print('No network role entities for age-bin plot')
        return None
    ncols = 4
    nrows = int(np.ceil(len(entities) / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(4.3 * ncols, 3.7 * nrows), squeeze=False)
    sources = []
    for i, ax in enumerate(axes.flat):
        if i >= len(entities):
            ax.axis('off'); continue
        ss = _plot_role_agebin_sem(ax, entities[i], ylabel='Role value' if i % ncols == 0 else None)
        if not ss.empty:
            sources.append(ss)
    fig.suptitle('Figure 6B. Network role age-bin mean +/- SEM', fontsize=12, fontweight='bold', y=0.995)
    plt.tight_layout(rect=[0.02, 0.02, 0.98, 0.92])
    save_fig(fig, outfile)
    if sources:
        pd.concat(sources, ignore_index=True).to_csv(TAB_DIR / f'{outfile}_source.tsv', sep='\t', index=False)
    plt.show()
    return fig



def plot_network_role_spearman_scatter(outfile='Figure6D_sender_receiver_spearman_scatter'):
    if role_models is None or role_models.empty:
        print('No network role models available')
        return None
    s = role_models[role_models['metric'].eq('sender_strength')].set_index('celltype')['spearman_rho'].rename('sender_rho')
    r = role_models[role_models['metric'].eq('receiver_strength')].set_index('celltype')['spearman_rho'].rename('receiver_rho')
    df = pd.concat([s, r], axis=1).dropna()
    if df.empty:
        print('No sender/receiver Spearman scatter data')
        return None
    df = df.reset_index()
    df['abs_norm'] = np.sqrt(df['sender_rho']**2 + df['receiver_rho']**2)
    label_set = set(df.sort_values('abs_norm', ascending=False).head(10)['celltype'])
    label_set |= set(df[df['celltype'].astype(str).str.contains('micro', case=False, na=False)]['celltype'])
    fig, ax = plt.subplots(figsize=(5.6, 5.2))
    colors = np.where(df['celltype'].astype(str).str.contains('micro', case=False, na=False), '#D71958', '#4C78A8')
    ax.scatter(df['sender_rho'], df['receiver_rho'], s=54, c=colors, alpha=0.86, edgecolors='white', linewidths=0.85)
    ax.axhline(0, color='#777777', lw=1.0, ls='--')
    ax.axvline(0, color='#777777', lw=1.0, ls='--')
    for _, rr in df.iterrows():
        if rr['celltype'] in label_set:
            ax.text(rr['sender_rho'], rr['receiver_rho'], str(rr['celltype']), fontsize=12, ha='left', va='bottom')
    ax.set_xlabel('Sender role rho with age')
    ax.set_ylabel('Receiver role rho with age')
    ax.grid(color=GRID_COLOR, lw=1.05)
    fig.suptitle('Figure 6D. Sender versus receiver role age associations', fontsize=12, fontweight='bold', y=0.995)
    plt.tight_layout(rect=[0.02, 0.02, 0.98, 0.92])
    save_fig(fig, outfile)
    df.to_csv(TAB_DIR / f'{outfile}_source.tsv', sep='\t', index=False)
    plt.show()
    return fig


def plot_microglia_network_role_focus(outfile='Figure6E_microglia_network_role_focus'):
    fig, ax = plt.subplots(figsize=(5.4, 3.4))
    df = _plot_microglia_role_ax(ax)
    fig.suptitle('Figure 6E. Microglia network role trends', fontsize=12, fontweight='bold', y=0.995)
    plt.tight_layout(rect=[0.02, 0.02, 0.98, 0.92])
    save_fig(fig, outfile)
    if not df.empty:
        df.to_csv(TAB_DIR / f'{outfile}_source.tsv', sep='\t', index=False)
    plt.show()
    return fig




# ---- Figure 5 focused continuous LR driver panels: top 2 aging-up and top 2 aging-down fine pairs ----
def _top4_pair_entities_for_plot(max_lr_per_pair=6):
    if 'top4_pair_lr_drivers' not in globals() or top4_pair_lr_drivers is None or top4_pair_lr_drivers.empty:
        return []
    df = top4_pair_lr_drivers.copy()
    if 'abs_spearman_rho' not in df.columns:
        df['abs_spearman_rho'] = df['spearman_rho'].abs()
    df = df.sort_values(['top_pair_rank', 'discovery_selected', 'spearman_p', 'abs_spearman_rho'], ascending=[True, False, True, False])
    out = []
    for _, sub in df.groupby('cellpair', sort=False):
        out.extend(sub.head(max_lr_per_pair)['cellpair_lr'].astype(str).tolist())
    return out


def plot_top4_pair_lr_driver_figure(max_lr_per_pair=6, outfile='Figure5_top4_pair_LR_drivers'):
    if 'top4_pair_lr_drivers' not in globals() or top4_pair_lr_drivers is None or top4_pair_lr_drivers.empty:
        print('No top4-pair LR drivers available')
        return None
    df = top4_pair_lr_drivers.copy()
    if 'abs_spearman_rho' not in df.columns:
        df['abs_spearman_rho'] = df['spearman_rho'].abs()
    plot_df = (df.sort_values(['top_pair_rank', 'discovery_selected', 'spearman_p', 'abs_spearman_rho'], ascending=[True, False, True, False])
                 .groupby('cellpair', sort=False, group_keys=False)
                 .head(max_lr_per_pair)
                 .copy())
    plot_df.to_csv(TAB_DIR / f'{outfile}_source.tsv', sep='\t', index=False)
    pairs = plot_df[['cellpair', 'top_pair_rank', 'pair_aging_direction', 'pair_spearman_rho', 'pair_spearman_p']].drop_duplicates().sort_values('top_pair_rank')
    n_pairs = len(pairs)
    fig, axes = plt.subplots(max(1, n_pairs), 3, figsize=(17.2, max(4.4, 4.2 * max(1, n_pairs))), squeeze=False,
                             gridspec_kw={'width_ratios': [1.2, 1.2, 1.35], 'wspace': 0.62, 'hspace': 0.92})
    for row_i, pair_row in enumerate(pairs.itertuples()):
        pair = pair_row.cellpair
        sub = plot_df[plot_df['cellpair'].astype(str).eq(str(pair))].copy()
        sub = sub.sort_values('spearman_rho')
        ax = axes[row_i, 0]
        y = np.arange(len(sub))
        colors = np.where(sub['spearman_rho'] >= 0, UP_COLOR, DOWN_COLOR)
        sizes = np.clip(-np.log10(sub['spearman_p'].clip(lower=1e-300)) * 26, 24, 180)
        ax.hlines(y, 0, sub['spearman_rho'], color='#CFCFCF', lw=0.9)
        ax.scatter(sub['spearman_rho'], y, s=sizes, c=colors, edgecolors='white', linewidths=0.755, zorder=3)
        ax.axvline(0, color='#333333', lw=1.0, ls='--')
        ax.set_yticks(y)
        ax.set_yticklabels([str(x).split('::')[-1] for x in sub['cellpair_lr']], fontsize=12)
        ax.set_xlabel('LR Spearman rho')
        ax.set_title(f"{pair}\n{pair_row.pair_aging_direction}; pair rho={pair_row.pair_spearman_rho:.2f}, P={pair_row.pair_spearman_p:.2g}", fontsize=12)
        ax.grid(axis='x', color=GRID_COLOR, lw=1.05)

        ax_traj = axes[row_i, 1]
        top_ent = sub.sort_values(['discovery_selected', 'spearman_p', 'abs_spearman_rho'], ascending=[False, True, False])['cellpair_lr'].astype(str).iloc[0]
        tr = lr_agg[lr_agg['cellpair_lr'].astype(str).eq(top_ent)].copy()
        if not tr.empty:
            ax_traj.scatter(tr['age'], tr['sum_strength'], c=tr['age_group'].map(AGE_PAL), s=58, alpha=0.85, edgecolors='white', linewidths=0.655)
            if len(tr) >= 4 and tr['sum_strength'].nunique() > 1:
                sns.regplot(data=tr, x='age', y='sum_strength', lowess=True, scatter=False, color='#2F2F2F', line_kws={'lw':1.2}, ax=ax_traj)
            _annotate_trend(ax_traj, lr_models, 'cellpair_lr', top_ent, 'sum_strength')
            ax_traj.set_title(str(top_ent).split('::')[-1], fontsize=12)
            ax_traj.set_xlabel('Age')
            ax_traj.set_ylabel('LR-level donor strength')
            ax_traj.grid(color=GRID_COLOR, lw=1.05)
        else:
            ax_traj.axis('off')

        ax_hm = axes[row_i, 2]
        source_hm = _plot_ordered_heatmap_ax(ax_hm, lr_agg, sub['cellpair_lr'].astype(str).tolist(), 'cellpair_lr', 'sum_strength')
        ax_hm.set_title('Donor-ordered LR strength', fontsize=12)
    fig.suptitle('Figure 5. Top aging-up/down communication pairs and LR drivers', fontsize=12, fontweight='bold', y=0.995)
    plt.tight_layout(rect=[0.02, 0.02, 0.98, 0.92])
    save_fig(fig, outfile)
    plt.show()
    return fig


def plot_top4_pair_functional_enrichment(outfile='Figure5B_top4_pair_GO_Reactome_enrichment'):
    enrich_df = globals().get('top4_pair_lr_functional_enrichment', pd.DataFrame())
    return plot_continuous_lr_gene_enrichment(enrich_df, outfile=outfile)


def plot_cpdb_interaction_family_summary(outfile='Figure5E_CPDB_interaction_family_summary'):
    return plot_lr_class_enrichment_figure(enrich_df=globals().get('lr_class_enrichment', pd.DataFrame()), outfile=outfile, top_n=12)


# ---- Figure 6 template-style age-bin role transition heatmap ----
def make_network_age_bins(panel, n_bins=5):
    if panel is None or panel.empty:
        return pd.DataFrame(), pd.DataFrame()
    df = panel.copy()
    donor_age = donor_base[['donor_id', 'age']].drop_duplicates().assign(donor_id=lambda x: x['donor_id'].astype(str))
    if 'age' not in df.columns:
        df = df.merge(donor_age, on='donor_id', how='left')
    donors = donor_age[donor_age['donor_id'].isin(df['donor_id'].astype(str).unique())].copy().sort_values('age')
    q = min(n_bins, max(1, donors['age'].nunique()))
    try:
        donors['age_bin'] = pd.qcut(donors['age'], q=q, duplicates='drop')
    except Exception:
        donors['age_bin'] = pd.cut(donors['age'], bins=q)
    donors['age_bin'] = donors['age_bin'].astype(str)
    bin_stats = (donors.groupby('age_bin', observed=True)
                 .agg(age_mid=('age','mean'), age_min=('age','min'), age_max=('age','max'), n_donors=('donor_id','nunique'))
                 .reset_index()
                 .sort_values('age_mid'))
    bin_stats['age_bin_label'] = bin_stats.apply(lambda r: f"{r.age_min:.0f}-{r.age_max:.0f}y", axis=1)
    df['donor_id'] = df['donor_id'].astype(str)
    df = df.merge(donors[['donor_id','age_bin']], on='donor_id', how='left')
    df = df.merge(bin_stats[['age_bin','age_bin_label','age_mid']], on='age_bin', how='left')
    return df, bin_stats


def plot_network_role_rank_transition(outfile='Figure6C_network_role_rank_transition_tiles', metrics=('pagerank','sender_strength','receiver_strength','hub_score'), top_k=3):
    if network_panel is None or network_panel.empty:
        print('No network_panel available for rank transition')
        return None
    df, bin_stats = make_network_age_bins(network_panel, n_bins=5)
    if df.empty or bin_stats.empty:
        return None
    role_df = (df.groupby(['age_bin','age_bin_label','age_mid','celltype'], observed=True)[list(metrics)]
               .mean().reset_index())
    long = role_df.melt(id_vars=['age_bin','age_bin_label','age_mid','celltype'], value_vars=list(metrics), var_name='metric', value_name='metric_value')
    long['rank'] = long.groupby(['metric','age_bin'], observed=True)['metric_value'].rank(ascending=False, method='first')
    top = long[long['rank'] <= top_k].copy()
    top['rank'] = top['rank'].astype(int)
    top = top.sort_values(['metric','age_mid','rank'])
    top.to_csv(TAB_DIR / f'{outfile}_source.tsv', sep='\t', index=False)
    metric_order = [m for m in metrics if m in top['metric'].unique()]
    fig, axes = plt.subplots(len(metric_order), 1, figsize=(11, 2.15 * max(1, len(metric_order))), squeeze=False)
    axes = axes.flatten()
    bin_order = bin_stats['age_bin'].tolist()
    bin_labels = dict(zip(bin_stats['age_bin'], bin_stats['age_bin_label']))
    for ax, metric in zip(axes, metric_order):
        sub = top[top['metric'].eq(metric)].copy()
        mat = np.full((top_k, len(bin_order)), np.nan)
        labels = [['' for _ in bin_order] for __ in range(top_k)]
        for _, row in sub.iterrows():
            if row['age_bin'] in bin_order:
                x = bin_order.index(row['age_bin'])
                y = int(row['rank']) - 1
                mat[y, x] = row['metric_value']
                labels[y][x] = str(row['celltype'])[:18]
        sns.heatmap(mat, ax=ax, cmap='YlGnBu', cbar=True, linewidths=0.85, linecolor='white',
                    xticklabels=[bin_labels.get(b, str(b)) for b in bin_order],
                    yticklabels=[f'Top {i}' for i in range(1, top_k + 1)])
        for y in range(top_k):
            for x in range(len(bin_order)):
                ax.text(x + 0.5, y + 0.5, labels[y][x], ha='center', va='center', fontsize=12, color='#1f1f1f')
        ax.set_title(f'{metric}: top cell-type role transition', fontsize=12, fontweight='bold')
        ax.set_xlabel('Age bin')
        ax.set_ylabel('Rank')
    fig.suptitle('Figure 6C. Network core role transition across age bins', fontsize=12, fontweight='bold', y=0.995)
    plt.tight_layout(rect=[0.02, 0.02, 0.98, 0.92])
    save_fig(fig, outfile)
    plt.show()
    return fig



# ---- Figure 4: top 2 aging-up and top 2 aging-down fine communication pairs ----
def plot_top4_communication_pairs_figure(outfile='Figure4_top2_up_top2_down_fine_pairs'):
    if 'top4_continuous_pairs' not in globals() or top4_continuous_pairs is None or top4_continuous_pairs.empty:
        print('No top aging-up/down fine communication pairs available')
        return None
    pairs = top4_continuous_pairs.copy().sort_values('top_pair_rank')
    pair_names = pairs['cellpair'].astype(str).tolist()
    pairs.to_csv(TAB_DIR / f'{outfile}_source.tsv', sep='\t', index=False)

    fig = plt.figure(figsize=(15.8, 9.8))
    gs = fig.add_gridspec(2, 3, width_ratios=[1.0, 1.25, 1.25], height_ratios=[1.0, 1.0], wspace=0.60, hspace=0.72)

    ax_a = fig.add_subplot(gs[:, 0])
    plot_df = pairs.sort_values('spearman_rho')
    y = np.arange(len(plot_df))
    colors = np.where(plot_df['spearman_rho'] >= 0, UP_COLOR, DOWN_COLOR)
    sizes = np.clip(-np.log10(plot_df['spearman_p'].clip(lower=1e-300)) * 30, 35, 190)
    ax_a.hlines(y, 0, plot_df['spearman_rho'], color='#CFCFCF', lw=1.0)
    ax_a.scatter(plot_df['spearman_rho'], y, s=sizes, c=colors, edgecolor='white', linewidth=0.85, zorder=3)
    ax_a.axvline(0, color='#333333', lw=1.0, ls='--')
    ax_a.set_yticks(y)
    ax_a.set_yticklabels([_entity_label(x, 34) for x in plot_df['cellpair']], fontsize=12)
    ax_a.set_xlabel('Fine-pair Spearman rho with age')
    ax_a.set_ylabel('Fine communication pair')
    ax_a.grid(axis='x', color=GRID_COLOR, lw=1.05)
    _panel_label(ax_a, 'a')

    trend_axes = _plot_trajectory_axes(gs[0, 1:], fig, fine_pair_full, fine_models, 'cellpair', 'mean_strength', pair_names, max_panels=4)
    if trend_axes:
        _panel_label(trend_axes[0], 'b')
        for ax in trend_axes:
            ax.set_ylabel('Mean communication strength')

    ax_c = fig.add_subplot(gs[1, 1:])
    heat = _plot_ordered_heatmap_ax(ax_c, fine_pair_full, pair_names, 'cellpair', 'mean_strength')
    ax_c.set_title('Donor-ordered mean strength', fontsize=12)
    _panel_label(ax_c, 'c')

    fig.suptitle('Figure 4. Top aging-up and aging-down fine communication pairs', fontsize=12, fontweight='bold', y=0.995)
    plt.tight_layout(rect=[0.02, 0.02, 0.98, 0.92])
    save_fig(fig, outfile)
    if isinstance(heat, pd.DataFrame) and not heat.empty:
        heat.to_csv(TAB_DIR / f'{outfile}_heatmap_source.tsv', sep='\t', index=False)
    plt.show()
    return fig


# ---- Figure 5 overrides: pair-specific LR drivers and pair-specific enrichment ----
def plot_top4_pair_lr_driver_figure(max_lr_per_pair=6, outfile='Figure5_top4_pair_LR_drivers'):
    if 'top4_pair_lr_drivers' not in globals() or top4_pair_lr_drivers is None or top4_pair_lr_drivers.empty:
        print('No top-pair LR drivers available')
        return None
    df = top4_pair_lr_drivers.copy()
    if 'abs_spearman_rho' not in df.columns:
        df['abs_spearman_rho'] = df['spearman_rho'].abs()
    sort_cols = ['top_pair_rank', 'discovery_selected', 'abs_spearman_rho', 'spearman_p']
    plot_df = (df.sort_values(sort_cols, ascending=[True, False, False, True])
                 .groupby('cellpair', sort=False, group_keys=False)
                 .head(max_lr_per_pair)
                 .copy())
    plot_df.to_csv(TAB_DIR / f'{outfile}_source.tsv', sep='\t', index=False)
    pairs = plot_df[['cellpair', 'top_pair_rank', 'pair_aging_direction', 'pair_spearman_rho', 'pair_spearman_p', 'pair_donor_frac']].drop_duplicates().sort_values('top_pair_rank')
    n_pairs = len(pairs)
    fig, axes = plt.subplots(max(1, n_pairs), 3, figsize=(17.4, max(4.4, 4.25 * max(1, n_pairs))), squeeze=False,
                             gridspec_kw={'width_ratios': [1.2, 1.2, 1.35], 'wspace': 0.64, 'hspace': 0.95})
    for row_i, pair_row in enumerate(pairs.itertuples()):
        pair = pair_row.cellpair
        sub = plot_df[plot_df['cellpair'].astype(str).eq(str(pair))].copy()
        sub = sub.sort_values('spearman_rho')

        ax = axes[row_i, 0]
        y = np.arange(len(sub))
        colors = np.where(sub['spearman_rho'] >= 0, UP_COLOR, DOWN_COLOR)
        sizes = np.clip(-np.log10(sub['spearman_p'].clip(lower=1e-300)) * 28, 24, 180)
        ax.hlines(y, 0, sub['spearman_rho'], color='#CFCFCF', lw=0.9)
        ax.scatter(sub['spearman_rho'], y, s=sizes, c=colors, edgecolors='white', linewidths=0.755, zorder=3)
        ax.axvline(0, color='#333333', lw=1.0, ls='--')
        ax.set_yticks(y)
        ax.set_yticklabels([_entity_label(str(x).split('::')[-1], 36) for x in sub['cellpair_lr']], fontsize=12)
        ax.set_xlabel('LR Spearman rho')
        ax.set_ylabel('LR driver')
        title = f"{pair}\n{pair_row.pair_aging_direction}; pair rho={pair_row.pair_spearman_rho:.2f}, P={pair_row.pair_spearman_p:.2g}, donors={pair_row.pair_donor_frac:.2f}"
        ax.set_title(title, fontsize=12)
        ax.grid(axis='x', color=GRID_COLOR, lw=1.05)
        if row_i == 0:
            _panel_label(ax, 'a')

        ax_traj = axes[row_i, 1]
        if not sub.empty:
            if 'discovery_selected' in sub.columns and sub['discovery_selected'].fillna(False).any():
                top_ent = sub[sub['discovery_selected'].fillna(False)].sort_values(['abs_spearman_rho', 'spearman_p'], ascending=[False, True])['cellpair_lr'].astype(str).iloc[0]
            else:
                top_ent = sub.sort_values(['abs_spearman_rho', 'spearman_p'], ascending=[False, True])['cellpair_lr'].astype(str).iloc[0]
            tr = lr_agg[lr_agg['cellpair_lr'].astype(str).eq(top_ent)].copy()
            if not tr.empty:
                ax_traj.scatter(tr['age'], tr['sum_strength'], c=tr['age_group'].map(AGE_PAL), s=58, alpha=0.85, edgecolors='white', linewidths=0.655)
                if len(tr) >= 4 and tr['sum_strength'].nunique() > 1:
                    sns.regplot(data=tr, x='age', y='sum_strength', lowess=True, scatter=False, color='#2F2F2F', line_kws={'lw':1.2}, ax=ax_traj)
                _annotate_trend(ax_traj, lr_models, 'cellpair_lr', top_ent, 'sum_strength')
                ax_traj.set_title(_entity_label(str(top_ent).split('::')[-1], 42), fontsize=12)
                ax_traj.set_xlabel('Age')
                ax_traj.set_ylabel('LR-level donor strength')
                ax_traj.grid(color=GRID_COLOR, lw=1.05)
            else:
                ax_traj.axis('off')
        else:
            ax_traj.axis('off')
        if row_i == 0:
            _panel_label(ax_traj, 'b')

        ax_hm = axes[row_i, 2]
        source_hm = _plot_ordered_heatmap_ax(ax_hm, lr_agg, sub['cellpair_lr'].astype(str).tolist(), 'cellpair_lr', 'sum_strength')
        ax_hm.set_title('Donor-ordered LR strength', fontsize=12)
        if row_i == 0:
            _panel_label(ax_hm, 'c')
    fig.suptitle('Figure 5. LR drivers for the top aging-up/down communication pairs', fontsize=12, fontweight='bold', y=0.995)
    plt.tight_layout(rect=[0.02, 0.02, 0.98, 0.92])
    save_fig(fig, outfile)
    plt.show()
    return fig


def plot_top4_pair_functional_enrichment(outfile='Figure5E_top4_pair_GO_Reactome_enrichment', top_n_per_pair=5):
    enrich_df = globals().get('top4_pair_lr_functional_enrichment', pd.DataFrame())
    if enrich_df is None or enrich_df.empty or 'term' not in enrich_df.columns:
        print('No pair-specific GO/Reactome enrichment results to plot')
        return None
    df = enrich_df.copy()
    p_col = 'adjusted_p' if 'adjusted_p' in df.columns else 'enrichment_p'
    df[p_col] = pd.to_numeric(df[p_col], errors='coerce')
    df = df.dropna(subset=[p_col])
    if df.empty or 'cellpair' not in df.columns:
        print('Pair-specific enrichment table is empty or lacks cellpair')
        return None
    df['minus_log10_adjusted_p'] = -np.log10(df[p_col].clip(lower=1e-300))
    df['minus_log10_enrichment_p'] = df['minus_log10_adjusted_p']
    df['overlap_n'] = pd.to_numeric(df.get('overlap_n', np.nan), errors='coerce').fillna(1)
    df['label'] = df['term'].astype(str).str.replace(r' \(GO:\d+\)$', '', regex=True).str.replace(r' R-HSA-\d+$', '', regex=True).str.slice(0, 58)
    pair_order = top4_continuous_pairs.sort_values('top_pair_rank')['cellpair'].astype(str).tolist() if 'top4_continuous_pairs' in globals() and not top4_continuous_pairs.empty else sorted(df['cellpair'].astype(str).unique())
    pair_order = [p for p in pair_order if p in set(df['cellpair'].astype(str))]
    if not pair_order:
        print('No selected pair has plottable enrichment terms')
        return None
    fig, axes = plt.subplots(len(pair_order), 1, figsize=(8.2, max(2.4, 2.2 * len(pair_order))), squeeze=False)
    axes = axes.flatten()
    source_rows = []
    for ax, pair in zip(axes, pair_order):
        sub = (df[df['cellpair'].astype(str).eq(pair)]
               .sort_values([p_col, 'overlap_n'], ascending=[True, False])
               .head(top_n_per_pair)
               .sort_values('minus_log10_adjusted_p'))
        if sub.empty:
            ax.text(0.5, 0.5, f'No enrichment terms for {pair}', ha='center', va='center')
            ax.axis('off')
            continue
        y = np.arange(len(sub))
        colors = np.where(sub['aging_direction'].astype(str).eq('aging_up'), UP_COLOR, DOWN_COLOR)
        markers = np.where(sub['side'].astype(str).eq('ligand'), 'o', '^')
        for marker in ['o', '^']:
            mask = markers == marker
            if mask.any():
                ax.scatter(sub.loc[mask, 'minus_log10_adjusted_p'], y[mask],
                           s=np.clip(sub.loc[mask, 'overlap_n'] * 34, 55, 300),
                           c=np.array(colors)[mask], marker=marker, edgecolor='white', linewidth=0.85, alpha=0.9)
        ax.set_yticks(y)
        ax.set_yticklabels(sub['label'], fontsize=12)
        ax.set_xlabel('-log10 adjusted enrichment P')
        ax.set_title(pair, fontsize=12, loc='left')
        ax.grid(axis='x', color=GRID_COLOR, lw=1.05)
        ax.spines[['top', 'right']].set_visible(False)
        source_rows.append(sub)
    if source_rows:
        pd.concat(source_rows, ignore_index=True).to_csv(TAB_DIR / f'{outfile}_source.tsv', sep='\t', index=False)
    fig.suptitle('Figure 5E. Pair-specific GO/Reactome enrichment of LR drivers', fontsize=12, fontweight='bold', y=0.995)
    plt.tight_layout(rect=[0.02, 0.02, 0.98, 0.92])
    save_fig(fig, outfile)
    plt.show()
    return fig


## Figure 1 - Dataset overview

In [ ]:
fig1 = plot_figure1_overview()

## Figure 2 - Broad cell-pair continuous-age trends

In [ ]:
fig2 = plot_layer_figure(broad_pair_full, broad_models, 'broad_pair', 'mean_strength',
                         'Figure 2. Broad cell-pair continuous-age communication trends',
                         'Figure2_broad_cellpair_continuous_age', top_n=14, trend_n=4)

## Figure 3 - Fine cell-pair continuous-age trends

In [ ]:
fig3 = plot_layer_figure(fine_pair_full, fine_models, 'cellpair', 'mean_strength',
                         'Figure 3. Fine cell-pair continuous-age communication trends',
                         'Figure3_fine_cellpair_continuous_age', top_n=14, trend_n=4)

## Figure 4 - Top continuous fine communication pairs

In [ ]:
fig4 = plot_top4_communication_pairs_figure()

## Figure 5 - LR drivers and GO/Reactome enrichment for top communication pairs

In [ ]:
fig5 = plot_top4_pair_lr_driver_figure(max_lr_per_pair=6)
fig5e = plot_top4_pair_functional_enrichment()

## Figure 6 - Network role changes

In [ ]:
fig6a = plot_network_role_trajectories(top_n=8)
fig6b = plot_network_role_agebin_errorbars(top_n=8)
fig6c = plot_network_role_rank_transition()
fig6d = plot_network_role_spearman_scatter()
fig6e = plot_microglia_network_role_focus()
fig6_shift = plot_network_sender_receiver_shift()

## CPDB communication-category analysis

In [ ]:

# CPDB-style official downstream analysis.
# FastCCC is the inference engine; CPDBv5 provides the LR database annotation.
# This section follows the usual CellPhoneDB/CPDB result logic: significant interaction table,
# cell-pair interaction counts, cellpair x LRI dotplot, and selected-LRI bubble views.
# CPDB classification is used only as a color/annotation field.


def build_cpdb_style_annotated_interactions(sig_long):
    if sig_long is None or sig_long.empty:
        return pd.DataFrame()
    df = sig_long.copy()
    if 'donor_id' not in df.columns and 'case' in df.columns:
        df = df.rename(columns={'case': 'donor_id'})
    df['donor_id'] = df['donor_id'].astype(str)
    if 'cellpair_lr' not in df.columns:
        df['cellpair_lr'] = df['cellpair'].astype(str) + '::' + df['lr_pair'].astype(str)
    if 'classification' not in df.columns:
        df['classification'] = 'Unannotated'
    meta = donor_base[['donor_id', 'age', 'age_group']].copy()
    meta['donor_id'] = meta['donor_id'].astype(str)
    df = df.drop(columns=['age', 'age_group'], errors='ignore').merge(meta, on='donor_id', how='left')
    if 'ligand' not in df.columns or 'receptor' not in df.columns:
        parsed = df['lr_pair'].map(split_lr_genes_continuous)
        df['ligand'] = ['+'.join(x[0]) if x and x[0] else np.nan for x in parsed]
        df['receptor'] = ['+'.join(x[1]) if x and len(x) > 1 and x[1] else np.nan for x in parsed]
    else:
        df['ligand'] = df['ligand'].replace('', np.nan)
        df['receptor'] = df['receptor'].replace('', np.nan)
    keep = ['donor_id','age','age_group','sender','receiver','cellpair','lr_pair','ligand','receptor',
            'cellpair_lr','classification','strength','pval']
    out = df[[c for c in keep if c in df.columns]].copy()
    return out.sort_values(['cellpair','lr_pair','donor_id'])


def summarize_cpdb_cellpair_interactions(annotated_df):
    if annotated_df is None or annotated_df.empty:
        return pd.DataFrame()
    out = (annotated_df.groupby(['cellpair','sender','receiver'], observed=True)
           .agg(significant_lri_n=('lr_pair','nunique'),
                interaction_rows=('cellpair_lr','size'),
                donor_n=('donor_id','nunique'),
                mean_strength=('strength','mean'),
                min_p=('pval','min'))
           .reset_index()
           .sort_values(['significant_lri_n','mean_strength'], ascending=[False, False]))
    return out


def build_cpdb_lri_dotplot_table(annotated_df, top_cellpair_n=18, top_lri_n=35):
    if annotated_df is None or annotated_df.empty:
        return pd.DataFrame()
    top_cellpairs = (annotated_df.groupby('cellpair')['lr_pair'].nunique()
                     .sort_values(ascending=False).head(top_cellpair_n).index.tolist())
    top_lris = (annotated_df.groupby('lr_pair')
                .agg(cellpair_n=('cellpair','nunique'), donor_n=('donor_id','nunique'), mean_strength=('strength','mean'))
                .sort_values(['cellpair_n','donor_n','mean_strength'], ascending=[False, False, False])
                .head(top_lri_n).index.tolist())
    sub = annotated_df[annotated_df['cellpair'].isin(top_cellpairs) & annotated_df['lr_pair'].isin(top_lris)].copy()
    out = (sub.groupby(['cellpair','lr_pair'], observed=True)
           .agg(mean_strength=('strength','mean'),
                min_p=('pval','min'),
                donor_n=('donor_id','nunique'),
                classification=('classification', lambda x: x.dropna().astype(str).iloc[0] if len(x.dropna()) else 'Unannotated'))
           .reset_index())
    out['neglog10_p'] = -np.log10(out['min_p'].clip(lower=1e-300))
    return out


def build_cpdb_selected_lri_bubble_table(annotated_df, selected_lr_df=None, top_n=40):
    if annotated_df is None or annotated_df.empty:
        return pd.DataFrame()
    if selected_lr_df is not None and not selected_lr_df.empty and 'lr_pair' in selected_lr_df.columns:
        selected_lris = selected_lr_df['lr_pair'].astype(str).drop_duplicates().tolist()
    else:
        selected_lris = (annotated_df.groupby('lr_pair')
                         .agg(donor_n=('donor_id','nunique'), cellpair_n=('cellpair','nunique'), mean_strength=('strength','mean'))
                         .sort_values(['donor_n','cellpair_n','mean_strength'], ascending=[False, False, False])
                         .head(top_n).index.astype(str).tolist())
    sub = annotated_df[annotated_df['lr_pair'].astype(str).isin(selected_lris)].copy()
    out = (sub.groupby(['cellpair','lr_pair'], observed=True)
           .agg(mean_strength=('strength','mean'),
                min_p=('pval','min'),
                donor_n=('donor_id','nunique'),
                classification=('classification', lambda x: x.dropna().astype(str).iloc[0] if len(x.dropna()) else 'Unannotated'))
           .reset_index())
    out['neglog10_p'] = -np.log10(out['min_p'].clip(lower=1e-300))
    return out


def _classification_palette(values):
    vals = sorted(pd.Series(values).dropna().astype(str).unique())
    pal = sns.color_palette('tab20', n_colors=max(1, len(vals)))
    return {v: pal[i % len(pal)] for i, v in enumerate(vals)}


def plot_cpdb_cellpair_count_bar(summary_df, outfile='Figure_CPDB_A_cellpair_significant_LRI_count', top_n=20):
    if summary_df is None or summary_df.empty:
        print('No CPDB cellpair interaction summary available')
        return None
    df = summary_df.head(top_n).sort_values('significant_lri_n')
    fig, ax = plt.subplots(figsize=(6.8, max(3.2, 0.27 * len(df) + 1.0)))
    ax.barh(np.arange(len(df)), df['significant_lri_n'], color='#4C78A8', alpha=0.9)
    ax.set_yticks(np.arange(len(df)))
    ax.set_yticklabels([_entity_label(x, 36) for x in df['cellpair']], fontsize=12)
    ax.set_xlabel('Significant LRIs')
    ax.set_ylabel('Sender-receiver cell pair')
    ax.grid(axis='x', color=GRID_COLOR, lw=1.05)
    ax.spines[['top','right']].set_visible(False)
    fig.suptitle('CPDB-style significant LRI count by cell pair', fontsize=12, fontweight='bold', y=0.995)
    plt.tight_layout(rect=[0,0,1,0.95])
    save_fig(fig, outfile)
    df.to_csv(TAB_DIR / f'{outfile}_source.tsv', sep='\t', index=False)
    plt.show()
    return fig


def plot_cpdb_cellpair_lri_dotplot(dot_df, outfile='Figure_CPDB_B_cellpair_LRI_dotplot', top_lri_n=35):
    if dot_df is None or dot_df.empty:
        print('No CPDB cellpair x LRI dotplot table available')
        return None
    df = dot_df.copy()
    lr_order = (df.groupby('lr_pair')['mean_strength'].max().sort_values(ascending=True).tail(top_lri_n).index.tolist())
    cp_order = (df.groupby('cellpair')['mean_strength'].max().sort_values(ascending=False).index.tolist())
    df = df[df['lr_pair'].isin(lr_order) & df['cellpair'].isin(cp_order)].copy()
    class_color = _classification_palette(df['classification'])
    x_map = {cp: i for i, cp in enumerate(cp_order)}
    y_map = {lr: i for i, lr in enumerate(lr_order)}
    fig, ax = plt.subplots(figsize=(max(8.5, 0.34 * len(cp_order) + 3.0), max(5.5, 0.22 * len(lr_order) + 2.0)))
    for cls, sub in df.groupby('classification'):
        ax.scatter(sub['cellpair'].map(x_map), sub['lr_pair'].map(y_map),
                   s=np.clip(sub['neglog10_p'] * 18, 14, 150),
                   c=[class_color.get(str(cls), '#777777')], alpha=0.86,
                   edgecolors='white', linewidths=1.4, label=str(cls))
    ax.set_xticks(range(len(cp_order)))
    ax.set_xticklabels([_entity_label(x, 24) for x in cp_order], rotation=45, ha='right', fontsize=12)
    ax.set_yticks(range(len(lr_order)))
    ax.set_yticklabels([_entity_label(x, 36) for x in lr_order], fontsize=12)
    ax.set_xlabel('Sender-receiver cell pair')
    ax.set_ylabel('Ligand-receptor interaction')
    ax.grid(color='#EFEFEF', lw=0.955)
    handles, labels = ax.get_legend_handles_labels()
    if len(handles) <= 14:
        ax.legend(handles, labels, frameon=False, bbox_to_anchor=(1.01, 1), loc='upper left', fontsize=12, title='CPDB classification')
    fig.suptitle('CPDB-style cellpair x LRI dotplot', fontsize=12, fontweight='bold', y=0.995)
    plt.tight_layout(rect=[0,0,0.92 if len(handles) <= 14 else 1,0.95])
    save_fig(fig, outfile)
    df.to_csv(TAB_DIR / f'{outfile}_source.tsv', sep='\t', index=False)
    plt.show()
    return fig


def plot_cpdb_selected_lri_bubble(bubble_df, outfile='Figure_CPDB_C_selected_LRI_bubble', top_lri_n=30):
    if bubble_df is None or bubble_df.empty:
        print('No CPDB selected-LRI bubble table available')
        return None
    df = bubble_df.copy()
    top_lris = df.groupby('lr_pair')['mean_strength'].max().sort_values(ascending=False).head(top_lri_n).index.tolist()
    df = df[df['lr_pair'].isin(top_lris)].copy()
    if df.empty:
        return None
    cp_order = df.groupby('cellpair')['mean_strength'].max().sort_values(ascending=False).index.tolist()
    lr_order = df.groupby('lr_pair')['mean_strength'].max().sort_values(ascending=True).index.tolist()
    class_color = _classification_palette(df['classification'])
    fig, ax = plt.subplots(figsize=(max(7.5, 0.32 * len(cp_order) + 3), max(4.8, 0.24 * len(lr_order) + 1.5)))
    for cls, sub in df.groupby('classification'):
        ax.scatter(sub['cellpair'].map({x:i for i,x in enumerate(cp_order)}),
                   sub['lr_pair'].map({x:i for i,x in enumerate(lr_order)}),
                   s=np.clip(sub['mean_strength'] * 55, 18, 180),
                   c=[class_color.get(str(cls), '#777777')], alpha=0.88,
                   edgecolors='white', linewidths=1.4, label=str(cls))
    ax.set_xticks(range(len(cp_order)))
    ax.set_xticklabels([_entity_label(x, 24) for x in cp_order], rotation=45, ha='right', fontsize=12)
    ax.set_yticks(range(len(lr_order)))
    ax.set_yticklabels([_entity_label(x, 34) for x in lr_order], fontsize=12)
    ax.set_xlabel('Sender-receiver cell pair')
    ax.set_ylabel('Selected LRI')
    ax.grid(color='#EFEFEF', lw=0.955)
    handles, labels = ax.get_legend_handles_labels()
    if len(handles) <= 14:
        ax.legend(handles, labels, frameon=False, bbox_to_anchor=(1.01, 1), loc='upper left', fontsize=12, title='CPDB classification')
    fig.suptitle('CPDB-style bubble plot for selected LRIs', fontsize=12, fontweight='bold', y=0.995)
    plt.tight_layout(rect=[0,0,0.92 if len(handles) <= 14 else 1,0.95])
    save_fig(fig, outfile)
    df.to_csv(TAB_DIR / f'{outfile}_source.tsv', sep='\t', index=False)
    plt.show()
    return fig


cpdb_style_interactions = build_cpdb_style_annotated_interactions(fastccc_long)
cpdb_style_interactions.to_csv(TAB_DIR / 'S8_CPDB_style_significant_interactions_annotated.tsv', sep='\t', index=False)

cpdb_cellpair_summary = summarize_cpdb_cellpair_interactions(cpdb_style_interactions)
cpdb_cellpair_summary.to_csv(TAB_DIR / 'S8_CPDB_cellpair_significant_LRI_summary.tsv', sep='\t', index=False)

cpdb_lri_dotplot_table = build_cpdb_lri_dotplot_table(cpdb_style_interactions)
cpdb_lri_dotplot_table.to_csv(TAB_DIR / 'S8_CPDB_cellpair_LRI_dotplot_table.tsv', sep='\t', index=False)

cpdb_selected_lri_bubble_table = build_cpdb_selected_lri_bubble_table(cpdb_style_interactions, selected_lr if 'selected_lr' in globals() else None)
cpdb_selected_lri_bubble_table.to_csv(TAB_DIR / 'S8_CPDB_selected_LRI_bubble_table.tsv', sep='\t', index=False)

fig_cpdb_a = plot_cpdb_cellpair_count_bar(cpdb_cellpair_summary)
fig_cpdb_b = plot_cpdb_cellpair_lri_dotplot(cpdb_lri_dotplot_table)
fig_cpdb_c = plot_cpdb_selected_lri_bubble(cpdb_selected_lri_bubble_table)

print('CPDB-style annotated significant interactions:', cpdb_style_interactions.shape)
print('CPDB cellpair significant-LRI summary:', cpdb_cellpair_summary.shape)
print('CPDB cellpair x LRI dotplot table:', cpdb_lri_dotplot_table.shape)


## Grouped FastCCC-style application

This section follows the FastCCC paper application logic. It uses group-level significant LRIs and `new_clusters2` display labels for readability. It is a companion group-level display and does not control the continuous mainline.

In [ ]:

# Grouped FastCCC-style application helpers.
# This section follows the FastCCC Fig.4 application logic: group-level significant LRIs,
# shared/unique LRI sets, chord plots, Fig.4h-style circle plots, and category summaries.
# It is independent of the continuous-age mainline.

import re
import matplotlib.patches as mpatches
import matplotlib.colors as mcolors
from matplotlib.path import Path as MplPath
from matplotlib.patches import PathPatch, FancyArrowPatch
from matplotlib.transforms import blended_transform_factory
from matplotlib.offsetbox import TextArea, HPacker, AnnotationBbox
from mpl_toolkits.axes_grid1.inset_locator import inset_axes
try:
    import scipy.stats as st
except Exception:
    st = None

GROUPED_DISPLAY_META_KEY = globals().get('GROUPED_DISPLAY_META_KEY', 'new_clusters2')
GROUPED_MIN_FRAC = 0.05
MICROGLIA_PATTERN = re.compile(r'micro', re.IGNORECASE)
FIG4H_GLOBAL_VMAX = 3.0
COLOR_G1 = np.array(mcolors.to_rgb('#253494'))
COLOR_G2 = np.array(mcolors.to_rgb('#F4A53A'))
COLOR_BOTH = np.array(mcolors.to_rgb('#D71958'))
COLOR_WEAK = np.array(mcolors.to_rgb('#F6E8C3'))
LIGAND_COLOR = '#1F4E79'
RECEPTOR_COLOR = '#D71958'


def _safe_slug(text):
    return re.sub(r'[^A-Za-z0-9]+', '_', str(text)).strip('_')


def _parse_cellpair_fig4h(cellpair):
    text = str(cellpair)
    if '|' in text:
        a, b = text.split('|', 1)
    elif '->' in text:
        a, b = text.split('->', 1)
    else:
        a = b = text
    return a.strip(), b.strip()


def _lr_label_fig4h(lr_pair):
    text = str(lr_pair)
    if '_' in text:
        lig, rec = text.split('_', 1)
        return f'{lig} -> {rec}'
    return text


def _annotation_gene_list_grouped(value):
    if pd.isna(value):
        return []
    text = str(value).strip()
    if not text or text.lower() in {'nan', 'none', 'null', 'na'}:
        return []
    genes = []
    for tok in re.split(r'[+;&|,/\s]+', text):
        tok = tok.strip()
        if not tok or tok.lower() in {'nan', 'none', 'null', 'na'}:
            continue
        if re.match(r'^CPI[-_]', tok, flags=re.IGNORECASE):
            continue
        if tok not in genes:
            genes.append(tok)
    return genes


def split_lr_genes(lr_pair):
    text = str(lr_pair)
    if re.match(r'^CPI[-_]', text, flags=re.IGNORECASE):
        return [], []
    if '_' not in text:
        return _annotation_gene_list_grouped(text), []
    lig, rec = text.split('_', 1)
    return _annotation_gene_list_grouped(lig), _annotation_gene_list_grouped(rec)


def resolve_grouped_lr_genes_from_row(row):
    ligands = _annotation_gene_list_grouped(row.get('ligand', np.nan))
    receptors = _annotation_gene_list_grouped(row.get('receptor', np.nan))
    if not ligands and not receptors:
        ligands, receptors = split_lr_genes(row.get('lr_pair', ''))
    return ligands, receptors


def _build_grouped_display_lookup(display_key=GROUPED_DISPLAY_META_KEY):
    lookup = {}
    if 'adata' not in globals() or display_key not in adata.obs.columns:
        return lookup
    obs = adata.obs.copy()
    display = obs[display_key].astype(str)
    lookup.update({x: x for x in display.dropna().unique()})
    for col in ['new_clusters', 'new_clusters2', 'new_clusters3', META_KEY if 'META_KEY' in globals() else None, BROAD_META_KEY if 'BROAD_META_KEY' in globals() else None]:
        if col and col in obs.columns:
            tmp = pd.DataFrame({'raw': obs[col].astype(str), 'display': display}).dropna().drop_duplicates()
            mode_map = (tmp.groupby(['raw', 'display']).size().rename('n').reset_index()
                          .sort_values(['raw', 'n'], ascending=[True, False])
                          .drop_duplicates('raw'))
            lookup.update(dict(zip(mode_map['raw'], mode_map['display'])))
    return lookup


GROUPED_DISPLAY_LOOKUP = _build_grouped_display_lookup()


def _display_celltype(celltype):
    return GROUPED_DISPLAY_LOOKUP.get(str(celltype), str(celltype))


def _ensure_grouped_sig_long(sig_long):
    if sig_long is None or sig_long.empty:
        return pd.DataFrame()
    df = sig_long.copy()
    if 'donor_id' not in df.columns and 'case' in df.columns:
        df = df.rename(columns={'case': 'donor_id'})
    if 'donor_id' not in df.columns:
        raise KeyError('fastccc_long must contain donor_id or case before grouped LRI plotting')
    df['donor_id'] = df['donor_id'].astype(str)
    if 'cellpair_lr' not in df.columns:
        df['cellpair_lr'] = df['cellpair'].astype(str) + '::' + df['lr_pair'].astype(str)
    if 'classification' not in df.columns:
        df['classification'] = 'Unannotated'
    df['raw_cellpair'] = df['cellpair'].astype(str)
    df['raw_cellpair_lr'] = df['cellpair_lr'].astype(str)
    parsed = df['raw_cellpair'].map(_parse_cellpair_fig4h)
    df['raw_sender'] = [x[0] for x in parsed]
    df['raw_receiver'] = [x[1] for x in parsed]
    df['sender'] = df['raw_sender'].map(_display_celltype)
    df['receiver'] = df['raw_receiver'].map(_display_celltype)
    df['cellpair'] = df['sender'].astype(str) + '|' + df['receiver'].astype(str)
    df['cellpair_lr'] = df['cellpair'].astype(str) + '::' + df['lr_pair'].astype(str)
    meta = donor_base[['donor_id', 'age_group']].copy()
    meta['donor_id'] = meta['donor_id'].astype(str)
    df = df.drop(columns=['age_group'], errors='ignore').merge(meta.drop_duplicates('donor_id'), on='donor_id', how='left')
    df = df[df['age_group'].isin(AGE_ORDER)].copy()
    return df


def prepare_group_lr(sig_long):
    df = _ensure_grouped_sig_long(sig_long)
    if df.empty:
        return pd.DataFrame()
    group_n = (donor_base[donor_base['age_group'].isin(AGE_ORDER)]
               .assign(donor_id=lambda x: x['donor_id'].astype(str))
               .groupby('age_group', observed=True)['donor_id'].nunique().to_dict())
    agg_map = {
        'donor_n': ('donor_id', 'nunique'),
        'mean_strength': ('strength', 'mean'),
        'min_p': ('pval', 'min'),
        'classification': ('classification', lambda x: x.dropna().astype(str).iloc[0] if len(x.dropna()) else 'Unannotated'),
    }
    for col in ['ligand', 'receptor', 'interacting_pair', 'directionality', 'has_gene_symbol_mapping']:
        if col in df.columns:
            agg_map[col] = (col, lambda x: x.dropna().astype(str).iloc[0] if len(x.dropna()) else np.nan)
    out = (df.groupby(['age_group', 'cellpair', 'lr_pair'], observed=True)
           .agg(**agg_map)
           .reset_index()
           .rename(columns={'age_group': 'group'}))
    out['group_donor_n'] = out['group'].map(group_n)
    out['frac_sig'] = out['donor_n'] / out['group_donor_n'].replace(0, np.nan)
    out['neglog10_p'] = -np.log10(out['min_p'].clip(lower=1e-300))
    parsed = out['cellpair'].map(_parse_cellpair_fig4h)
    out['sender'] = [x[0] for x in parsed]
    out['receiver'] = [x[1] for x in parsed]
    out['lr_label'] = out['lr_pair'].map(_lr_label_fig4h)
    out['cellpair_lr'] = out['cellpair'].astype(str) + '::' + out['lr_pair'].astype(str)
    return out


def build_grouped_lri_membership(group_lr, min_frac=GROUPED_MIN_FRAC):
    if group_lr is None or group_lr.empty:
        return pd.DataFrame()
    base_cols = [c for c in ['cellpair_lr', 'cellpair', 'lr_pair', 'lr_label', 'sender', 'receiver',
                              'classification', 'ligand', 'receptor', 'interacting_pair', 'directionality',
                              'has_gene_symbol_mapping'] if c in group_lr.columns]
    base = group_lr[base_cols].drop_duplicates('cellpair_lr').copy()
    support = group_lr[group_lr['frac_sig'].fillna(0) >= min_frac].copy()
    present = support.pivot_table(index='cellpair_lr', columns='group', values='frac_sig', aggfunc='max', fill_value=0)
    for g in AGE_ORDER:
        if g not in present.columns:
            present[g] = 0.0
    present = present[AGE_ORDER]
    out = base.merge(present.reset_index(), on='cellpair_lr', how='left')
    for g in AGE_ORDER:
        out[g] = out[g].fillna(0.0)
        out[f'present_{_safe_slug(g)}'] = out[g] > 0
    def _classify(row):
        groups = [g for g in AGE_ORDER if bool(row[f'present_{_safe_slug(g)}'])]
        if len(groups) == len(AGE_ORDER):
            return 'shared_all'
        if len(groups) == 1:
            return f'{groups[0]}_specific'
        if len(groups) == 0:
            return 'not_supported'
        return 'shared_' + '_'.join(groups)
    out['membership_class'] = out.apply(_classify, axis=1)
    out['n_supported_groups'] = out[[f'present_{_safe_slug(g)}' for g in AGE_ORDER]].sum(axis=1)
    return out.sort_values(['n_supported_groups', 'membership_class', 'cellpair_lr'], ascending=[False, True, True])


def build_grouped_lri_gene_sets(membership_df, prefix):
    records = []
    if membership_df is None or membership_df.empty:
        out = pd.DataFrame()
    else:
        unresolved = []
        for _, row in membership_df.iterrows():
            ligands, receptors = resolve_grouped_lr_genes_from_row(row)
            if not ligands and not receptors:
                unresolved.append(row.get('lr_pair', np.nan))
            for side, genes in [('ligand', ligands), ('receptor', receptors)]:
                for gene in genes:
                    records.append({'gene_set': f"{row['membership_class']}_{side}s", 'membership_class': row['membership_class'],
                                    'side': side, 'gene': gene, 'cellpair_lr': row['cellpair_lr'], 'cellpair': row['cellpair'],
                                    'lr_pair': row['lr_pair'], 'ligand': row.get('ligand', np.nan),
                                    'receptor': row.get('receptor', np.nan), 'sender': row['sender'], 'receiver': row['receiver'],
                                    'classification': row.get('classification', 'Unannotated')})
        out = pd.DataFrame(records).drop_duplicates() if records else pd.DataFrame()
    out.to_csv(TAB_DIR / f'{prefix}_lr_gene_sets.tsv', sep='\t', index=False)
    return out


def summarize_grouped_categories(membership_df, group_lr, prefix):
    if membership_df is None or membership_df.empty or group_lr is None or group_lr.empty:
        out = pd.DataFrame()
        out.to_csv(TAB_DIR / f'{prefix}_cpdb_category_summary.tsv', sep='\t', index=False)
        return out
    rows = []
    classes = group_lr[['cellpair_lr', 'classification']].drop_duplicates().copy()
    background = set(classes['cellpair_lr'].astype(str))
    for label, sub in membership_df.groupby('membership_class'):
        selected = set(sub['cellpair_lr'].astype(str)) & background
        if not selected:
            continue
        for cls, cdf in classes.groupby(classes['classification'].fillna('Unannotated').astype(str)):
            cls_set = set(cdf['cellpair_lr'].astype(str))
            a = len(selected & cls_set)
            if a == 0:
                continue
            b = len(selected) - a
            c = len(cls_set - selected)
            d = max(len(background) - len(selected) - c, 0)
            if st is not None:
                odds, pval = st.fisher_exact([[a, b], [c, d]], alternative='greater')
            else:
                odds, pval = np.nan, np.nan
            rows.append({'membership_class': label, 'classification': cls, 'selected_n': a,
                         'background_n': len(cls_set), 'selected_total': len(selected),
                         'background_total': len(background), 'odds_ratio': odds, 'fisher_p': pval})
    out = pd.DataFrame(rows)
    if not out.empty:
        out['minus_log10_p'] = -np.log10(out['fisher_p'].clip(lower=1e-300))
        out = out.sort_values(['membership_class', 'fisher_p', 'selected_n'], ascending=[True, True, False])
    out.to_csv(TAB_DIR / f'{prefix}_cpdb_category_summary.tsv', sep='\t', index=False)
    return out


def _is_microglia_series(s):
    return s.astype(str).str.contains(MICROGLIA_PATTERN)


In [ ]:

# Build grouped FastCCC application tables from all significant LRIs.
# This mirrors the FastCCC paper application layer and is not restricted to continuous-age LR drivers.

grouped_sig_long = _ensure_grouped_sig_long(fastccc_long)
grouped_sig_long.to_csv(TAB_DIR / 'S9_grouped_significant_LRI_long_newclusters2.tsv', sep='\t', index=False)

group_lr = prepare_group_lr(fastccc_long)
group_lr.to_csv(TAB_DIR / 'S9_grouped_LRI_support_table.tsv', sep='\t', index=False)

membership = build_grouped_lri_membership(group_lr, min_frac=GROUPED_MIN_FRAC)
membership.to_csv(TAB_DIR / 'S9_grouped_LRI_membership.tsv', sep='\t', index=False)
grouped_lri_gene_sets = build_grouped_lri_gene_sets(membership, 'S9_grouped_LRI')
grouped_category_summary = summarize_grouped_categories(membership, group_lr, 'S9_grouped_LRI')
if 'run_continuous_lr_enrichr' in globals():
    grouped_lri_functional_enrichment = run_continuous_lr_enrichr(grouped_lri_gene_sets)
else:
    grouped_lri_functional_enrichment = pd.DataFrame()
grouped_lri_functional_enrichment.to_csv(TAB_DIR / 'S9_grouped_LRI_GO_Reactome_enrichment.tsv', sep='\t', index=False)

if group_lr.empty:
    micro_group_lr = pd.DataFrame()
    micro_membership = pd.DataFrame()
else:
    micro_mask = _is_microglia_series(group_lr['sender']) | _is_microglia_series(group_lr['receiver'])
    micro_sender_mask = _is_microglia_series(group_lr['sender'])
    micro_receiver_mask = _is_microglia_series(group_lr['receiver'])
    group_lr[micro_sender_mask].to_csv(TAB_DIR / 'S9_microglia_sender_grouped_LRI_support.tsv', sep='\t', index=False)
    group_lr[micro_receiver_mask].to_csv(TAB_DIR / 'S9_microglia_receiver_grouped_LRI_support.tsv', sep='\t', index=False)
    micro_group_lr = group_lr[micro_mask].copy()
    micro_group_lr.to_csv(TAB_DIR / 'S9_microglia_involved_grouped_LRI_support.tsv', sep='\t', index=False)
    micro_membership = build_grouped_lri_membership(micro_group_lr, min_frac=GROUPED_MIN_FRAC)
    micro_membership.to_csv(TAB_DIR / 'S9_microglia_involved_LRI_membership.tsv', sep='\t', index=False)
    micro_lri_gene_sets = build_grouped_lri_gene_sets(micro_membership, 'S9_microglia_involved_LRI')
    micro_category_summary = summarize_grouped_categories(micro_membership, micro_group_lr, 'S9_microglia_involved_LRI')
    if 'run_continuous_lr_enrichr' in globals():
        micro_lri_functional_enrichment = run_continuous_lr_enrichr(micro_lri_gene_sets)
    else:
        micro_lri_functional_enrichment = pd.DataFrame()
    micro_lri_functional_enrichment.to_csv(TAB_DIR / 'S9_microglia_involved_LRI_GO_Reactome_enrichment.tsv', sep='\t', index=False)

print('Grouped display annotation:', GROUPED_DISPLAY_META_KEY)
print('Grouped significant long:', grouped_sig_long.shape)
print('Grouped support table:', group_lr.shape)
print('Grouped membership:', membership['membership_class'].value_counts().to_dict() if not membership.empty else {})
print('Microglia-involved support:', micro_group_lr.shape if micro_group_lr is not None else (0, 0))


In [ ]:

# Grouped application overview: sample/group counts and LRI membership matrix.

def plot_grouped_application_overview(group_lr, membership, outfile='Figure9A_grouped_application_overview'):
    fig, axes = plt.subplots(1, 2, figsize=(7.2, 2.8), gridspec_kw={'width_ratios': [1.0, 1.4]})
    donor_counts = donor_base.groupby('age_group', observed=False)['donor_id'].nunique().reindex(AGE_ORDER).fillna(0)
    axes[0].bar(np.arange(len(AGE_ORDER)), donor_counts.values, color=[AGE_PAL[g] for g in AGE_ORDER], width=0.72)
    axes[0].set_xticks(np.arange(len(AGE_ORDER)))
    axes[0].set_xticklabels(AGE_ORDER, rotation=35, ha='right')
    axes[0].set_ylabel('Donors')
    axes[0].spines[['top', 'right']].set_visible(False)
    axes[0].text(-0.22, 1.05, 'a', transform=axes[0].transAxes, fontweight='bold', fontsize=12)

    if group_lr is not None and not group_lr.empty:
        lri_counts = (group_lr[group_lr['frac_sig'].fillna(0) >= GROUPED_MIN_FRAC]
                      .groupby('group', observed=False)['cellpair_lr'].nunique()
                      .reindex(AGE_ORDER).fillna(0))
    else:
        lri_counts = pd.Series(0, index=AGE_ORDER)
    axes[1].bar(np.arange(len(AGE_ORDER)), lri_counts.values, color=[AGE_PAL[g] for g in AGE_ORDER], width=0.72)
    axes[1].set_xticks(np.arange(len(AGE_ORDER)))
    axes[1].set_xticklabels(AGE_ORDER, rotation=35, ha='right')
    axes[1].set_ylabel('Supported LRIs')
    axes[1].spines[['top', 'right']].set_visible(False)
    axes[1].text(-0.16, 1.05, 'b', transform=axes[1].transAxes, fontweight='bold', fontsize=12)
    plt.tight_layout()
    save_fig(fig, outfile)
    plt.show()
    source = pd.DataFrame({'age_group': AGE_ORDER, 'donor_n': donor_counts.values, 'supported_lri_n': lri_counts.values})
    source.to_csv(TAB_DIR / f'{outfile}_source.tsv', sep='\t', index=False)
    return fig


def plot_lri_membership_matrix(membership, outfile='Figure9B_grouped_LRI_membership_matrix', top_n=12):
    if membership is None or membership.empty:
        print('No grouped membership table available')
        return None
    counts = membership['membership_class'].value_counts().reset_index()
    counts.columns = ['membership_class', 'n_lri']
    counts = counts[~counts['membership_class'].eq('not_supported')].head(top_n)
    if counts.empty:
        print('No supported membership classes to plot')
        return None
    fig, (ax_bar, ax_mat) = plt.subplots(2, 1, figsize=(max(4.2, 0.48 * len(counts)), 3.8),
                                         gridspec_kw={'height_ratios': [1.25, 1.0], 'hspace': 0.08})
    x = np.arange(len(counts))
    ax_bar.bar(x, counts['n_lri'], color='#555555', width=0.68)
    ax_bar.set_ylabel('LRIs')
    ax_bar.set_xticks([])
    ax_bar.spines[['top', 'right']].set_visible(False)
    ax_bar.text(-0.08, 1.06, 'a', transform=ax_bar.transAxes, fontweight='bold', fontsize=12)
    for yi, g in enumerate(AGE_ORDER):
        ax_mat.text(-0.6, yi, g, ha='right', va='center', fontsize=12)
    for xi, cls in enumerate(counts['membership_class']):
        present = []
        if cls == 'shared_all':
            present = AGE_ORDER
        elif cls.endswith('_specific'):
            present = [cls.replace('_specific', '')]
        elif cls.startswith('shared_'):
            present = cls.replace('shared_', '').split('_')
        yvals = []
        for g in AGE_ORDER:
            on = g in present
            y = AGE_ORDER.index(g)
            yvals.append(y) if on else None
            ax_mat.scatter(xi, y, s=68 if on else 32, color=AGE_PAL[g] if on else '#D9D9D9', zorder=3)
        if len(yvals) > 1:
            ax_mat.plot([xi, xi], [min(yvals), max(yvals)], color='#777777', lw=1.0, zorder=1)
    ax_mat.set_xlim(-0.6, len(counts)-0.4)
    ax_mat.set_ylim(-0.6, len(AGE_ORDER)-0.4)
    ax_mat.set_yticks([])
    ax_mat.set_xticks(x)
    ax_mat.set_xticklabels(counts['membership_class'], rotation=45, ha='right', fontsize=12)
    ax_mat.spines[['top', 'right', 'left']].set_visible(False)
    ax_mat.text(-0.08, 1.08, 'b', transform=ax_mat.transAxes, fontweight='bold', fontsize=12)
    plt.tight_layout()
    save_fig(fig, outfile)
    counts.to_csv(TAB_DIR / f'{outfile}_source.tsv', sep='\t', index=False)
    plt.show()
    return fig

plot_grouped_application_overview(group_lr, membership)
plot_lri_membership_matrix(membership, 'Figure9B_grouped_LRI_membership_matrix')
if micro_membership is not None and not micro_membership.empty:
    plot_lri_membership_matrix(micro_membership, 'Figure9B_microglia_LRI_membership_matrix')
else:
    print('No microglia-involved LRI passed the grouped display filter.')


In [ ]:

# Group-specific LRI chord plots, matching the FastCCC Fig.4 unique-LRI idea.

def plot_group_specific_chord(membership_df, group_name, outfile, top_n=80):
    if membership_df is None or membership_df.empty:
        return None
    cls = f'{group_name}_specific'
    sub = membership_df[membership_df['membership_class'].astype(str).eq(cls)].copy()
    if sub.empty:
        print(f'No {cls} LRIs for chord plot')
        return None
    edge = (sub.groupby(['sender', 'receiver'], observed=True)
            .agg(n_lri=('cellpair_lr', 'nunique'))
            .reset_index()
            .sort_values('n_lri', ascending=False)
            .head(top_n))
    edge.to_csv(TAB_DIR / f'{outfile}_source.tsv', sep='\t', index=False)
    nodes = sorted(set(edge['sender']).union(edge['receiver']))
    if len(nodes) < 2:
        return None
    angles = np.linspace(0, 2 * np.pi, len(nodes), endpoint=False)
    pos = {node: np.array([np.cos(a), np.sin(a)]) for node, a in zip(nodes, angles)}
    pal = sns.color_palette('tab20', n_colors=max(3, len(nodes)))
    color = {node: pal[i % len(pal)] for i, node in enumerate(nodes)}
    fig, ax = plt.subplots(figsize=(6.2, 6.2))
    ax.set_aspect('equal'); ax.axis('off')
    max_w = max(edge['n_lri'].max(), 1)
    for _, r in edge.iterrows():
        p0, p1 = pos[r['sender']], pos[r['receiver']]
        path = MplPath([p0, p0 * 0.35, p1 * 0.35, p1], [MplPath.MOVETO, MplPath.CURVE4, MplPath.CURVE4, MplPath.CURVE4])
        ax.add_patch(PathPatch(path, facecolor='none', edgecolor=color[r['sender']], lw=1.05 + 4.0 * r['n_lri'] / max_w, alpha=0.55))
        ax.add_patch(FancyArrowPatch(p1 * 0.92, p1 * 0.99, arrowstyle='-|>', mutation_scale=10, color=color[r['sender']], lw=0, alpha=0.7))
    for node in nodes:
        p = pos[node]
        ax.scatter(p[0], p[1], s=95, color=color[node], edgecolor='white', linewidth=0.7, zorder=3)
        ax.text(p[0] * 1.16, p[1] * 1.16, node, ha='center', va='center', fontsize=12)
    save_fig(fig, outfile)
    plt.show()
    return fig

for group_name in AGE_ORDER:
    specific = membership[membership['membership_class'].astype(str).eq(f'{group_name}_specific')] if membership is not None and not membership.empty else pd.DataFrame()
    specific.to_csv(TAB_DIR / f'S9_{_safe_slug(group_name)}_specific_LRIs.tsv', sep='\t', index=False)
    plot_group_specific_chord(membership, group_name, f'Figure9C_{_safe_slug(group_name)}_specific_LRI_chord')

if micro_membership is not None and not micro_membership.empty:
    for group_name in AGE_ORDER:
        plot_group_specific_chord(micro_membership, group_name, f'Figure9C_microglia_{_safe_slug(group_name)}_specific_LRI_chord')


In [ ]:

# FastCCC Fig.4h-style pairwise LRI circle plots.

def _norm_s(v, cap):
    return (np.clip(float(v), 0, cap) / cap) ** 0.55


def _norm_d(d, cap):
    return np.clip(float(d), -cap, cap) / cap


def _dot_style(nlp1, nlp2, vmax=FIG4H_GLOBAL_VMAX):
    g1, g2 = float(nlp1), float(nlp2)
    strength = _norm_s(g1 + g2, vmax * 2)
    diff = _norm_d(g1 - g2, vmax)
    size = 12 + 220 * strength
    if strength < 0.12:
        rgb = COLOR_WEAK
    else:
        if diff > 0.15:
            dom = min(abs(diff), 1.0); rgb = (1 - dom) * COLOR_BOTH + dom * COLOR_G1
        elif diff < -0.15:
            dom = min(abs(diff), 1.0); rgb = (1 - dom) * COLOR_BOTH + dom * COLOR_G2
        else:
            rgb = COLOR_BOTH
        rgb = COLOR_WEAK * (1 - strength) + rgb * strength
    rgb = np.clip(rgb, 0, 1)
    return (*rgb, 1.0), size


def _make_legend_img(vmax=FIG4H_GLOBAL_VMAX, n=50):
    img = np.zeros((n, n, 4)); vals = np.linspace(0, vmax, n)
    for ix in range(n):
        for iy in range(n):
            rgba, _ = _dot_style(vals[iy], vals[ix], vmax)
            img[n - 1 - iy, ix] = rgba
    return img


def plot_group_lri_contrast(group_lr, g1, g2, outfile, top_n=30, min_frac=GROUPED_MIN_FRAC, focus_pattern=None):
    if group_lr is None or group_lr.empty:
        print('No grouped LRI table available for plotting')
        return None
    plot_source = group_lr.copy()
    if focus_pattern is not None:
        mask = plot_source['sender'].astype(str).str.contains(focus_pattern) | plot_source['receiver'].astype(str).str.contains(focus_pattern)
        plot_source = plot_source[mask].copy()
    if plot_source.empty:
        print(f'No grouped LRI rows for {g1} vs {g2}')
        return None
    cols = ['cellpair', 'lr_pair', 'lr_label', 'sender', 'receiver', 'mean_strength', 'min_p', 'frac_sig', 'neglog10_p']
    d1 = plot_source.loc[plot_source['group'].astype(str).eq(g1), cols].copy()
    d2 = plot_source.loc[plot_source['group'].astype(str).eq(g2), ['cellpair', 'lr_pair', 'mean_strength', 'min_p', 'frac_sig', 'neglog10_p']].copy()
    mg = d1.merge(d2, on=['cellpair', 'lr_pair'], suffixes=(f'_{g1}', f'_{g2}'), how='outer')
    for g in [g1, g2]:
        mg[f'mean_strength_{g}'] = mg[f'mean_strength_{g}'].fillna(0.0)
        mg[f'min_p_{g}'] = mg[f'min_p_{g}'].fillna(1.0)
        mg[f'frac_sig_{g}'] = mg[f'frac_sig_{g}'].fillna(0.0)
        mg[f'neglog10_p_{g}'] = mg[f'neglog10_p_{g}'].fillna(0.0)
    if 'sender' not in mg.columns or mg['sender'].isna().any():
        parsed = mg['cellpair'].map(_parse_cellpair_fig4h)
        mg['sender'] = [x[0] for x in parsed]
        mg['receiver'] = [x[1] for x in parsed]
    if 'lr_label' not in mg.columns or mg['lr_label'].isna().any():
        mg['lr_label'] = mg['lr_pair'].map(_lr_label_fig4h)
    mg = mg[(mg[f'frac_sig_{g1}'] >= min_frac) | (mg[f'frac_sig_{g2}'] >= min_frac)].copy()
    if mg.empty:
        print(f'No grouped LRI passed min_frac={min_frac}: {g1} vs {g2}')
        return None
    mg['max_nlp'] = mg[[f'neglog10_p_{g1}', f'neglog10_p_{g2}']].max(axis=1)
    top_lr = mg.groupby('lr_label')['max_nlp'].max().nlargest(top_n).index.tolist()
    mg = mg[mg['lr_label'].isin(top_lr)].copy()
    lr_order = mg.groupby('lr_label')['max_nlp'].max().sort_values(ascending=True).index.tolist()
    cp_order = (mg[['cellpair', 'sender', 'receiver']].drop_duplicates().sort_values(['sender', 'receiver'])['cellpair'].tolist())
    n_y, n_x = len(lr_order), len(cp_order)
    if n_x == 0 or n_y == 0:
        return None
    fig, ax = plt.subplots(figsize=(max(10, n_x * max(0.30, min(0.55, 18 / n_x)) + 5), max(6, n_y * max(0.28, min(0.42, 14 / n_y)) + 3.8)))
    fig.patch.set_facecolor('white'); ax.set_facecolor('white')
    for _, r in mg.iterrows():
        if r['lr_label'] not in lr_order or r['cellpair'] not in cp_order:
            continue
        xi = cp_order.index(r['cellpair']); yi = lr_order.index(r['lr_label'])
        rgba, size = _dot_style(r[f'neglog10_p_{g1}'], r[f'neglog10_p_{g2}'])
        ax.scatter(xi, yi, s=size, c=[rgba[:3]], alpha=rgba[3], edgecolors='black' if r['max_nlp'] > 0.3 else 'none', linewidths=0.6, zorder=3)
    senders = [_parse_cellpair_fig4h(c)[0] for c in cp_order]
    for i in range(1, n_x):
        if senders[i] != senders[i - 1]: ax.axvline(i - 0.5, color='#AAAAAA', ls='--', lw=1.05, zorder=1)
    for yi in range(n_y): ax.axhline(yi, color='#EEEEEE', lw=0.95, zorder=0)
    ax.set_xlim(-0.5, n_x - 0.5); ax.set_ylim(-0.5, n_y - 0.5)
    ax.set_xticks(range(n_x)); ax.set_xticklabels([''] * n_x); ax.set_yticks(range(n_y)); ax.set_yticklabels([''] * n_y); ax.tick_params(length=0)
    for spine in ax.spines.values(): spine.set_visible(False)
    for i, lr in enumerate(lr_order):
        parts = [p.strip() for p in str(lr).split(' -> ')]
        children = [TextArea(str(lr), textprops=dict(color='black', size=10))]
        if len(parts) == 2:
            children = [TextArea(parts[0], textprops=dict(color=LIGAND_COLOR, size=10, weight='bold')), TextArea('-', textprops=dict(color='#555555', size=10)), TextArea(parts[1], textprops=dict(color=RECEPTOR_COLOR, size=10, weight='bold'))]
        ax.add_artist(AnnotationBbox(HPacker(children=children, pad=0, sep=1, align='center'), xy=(-0.005, i), xycoords=('axes fraction', 'data'), box_alignment=(1.0, 0.5), frameon=False))
    xt = blended_transform_factory(ax.transData, ax.transAxes)
    for i, cp in enumerate(cp_order):
        _, recv = _parse_cellpair_fig4h(cp)
        ax.text(i, -0.015, recv, transform=xt, ha='right', va='top', rotation=45, fontsize=12, color=RECEPTOR_COLOR, weight='bold', style='italic', clip_on=False)
    sender_groups = {}
    for i, cp in enumerate(cp_order): sender_groups.setdefault(_parse_cellpair_fig4h(cp)[0], []).append(i)
    for sender, idxs in sender_groups.items():
        mid = (idxs[0] + idxs[-1]) / 2
        ax.plot([idxs[0] - 0.35, idxs[-1] + 0.35], [-0.17, -0.17], color='#AFAFAF', lw=1.0, transform=xt, clip_on=False)
        ax.text(mid, -0.31, sender, transform=xt, ha='center', va='top', rotation=90, fontsize=12, color=LIGAND_COLOR, clip_on=False)
    # 2D inset legend removed to avoid covering the LRI dot matrix.
    handles = []
    for v in [1, 2, 3]:
        _, size = _dot_style(v, 0)
        handles.append(plt.scatter([], [], s=size, c=[[*COLOR_G1, 0.85]], edgecolors='black', linewidths=0.6, label=f'-log10(P)={v}'))
    handles += [mpatches.Patch(facecolor=COLOR_G1, label=f'{g1} dominant'), mpatches.Patch(facecolor=COLOR_G2, label=f'{g2} dominant'), mpatches.Patch(facecolor=COLOR_BOTH, label='Both high'), mpatches.Patch(facecolor=COLOR_WEAK, label='Both weak')]
    ax.legend(handles=handles, loc='upper left', bbox_to_anchor=(1.01, 0.50), frameon=True, framealpha=0.92, fontsize=12, title='Legend', title_fontsize=12, edgecolor='#CCCCCC', handletextpad=0.8)
    plt.subplots_adjust(bottom=0.26, left=0.18, right=0.87, top=0.93)
    mg.to_csv(TAB_DIR / f'{outfile}_source.tsv', sep='\t', index=False)
    save_fig(fig, outfile)
    plt.show()
    return fig

for g1, g2, tag in [('Young adult', 'Adult', 'early'), ('Adult', 'Elderly', 'late'), ('Young adult', 'Elderly', 'full_aging')]:
    plot_group_lri_contrast(group_lr, g1, g2, outfile=f'Figure9D_FastCCC_Fig4h_LRI_{tag}_{_safe_slug(g1)}_vs_{_safe_slug(g2)}', top_n=30, min_frac=GROUPED_MIN_FRAC)
    plot_group_lri_contrast(micro_group_lr, g1, g2, outfile=f'Figure9D_microglia_FastCCC_Fig4h_LRI_{tag}_{_safe_slug(g1)}_vs_{_safe_slug(g2)}', top_n=25, min_frac=GROUPED_MIN_FRAC, focus_pattern=MICROGLIA_PATTERN)


In [ ]:

# CPDB communication-category summaries for grouped LRIs.
# CPDB is used here only as an LRI annotation/category source, not as a cell-type label.

def plot_grouped_category_summary(cat_df, outfile, top_n=18):
    if cat_df is None or cat_df.empty:
        print(f'No category summary for {outfile}')
        return None
    df = cat_df[~cat_df['membership_class'].eq('not_supported')].copy()
    if df.empty:
        print(f'No supported category rows for {outfile}')
        return None
    df = df.sort_values(['fisher_p', 'selected_n'], ascending=[True, False]).head(top_n).copy()
    df['label'] = df['membership_class'].astype(str) + ' | ' + df['classification'].astype(str)
    df = df.sort_values('minus_log10_p')
    fig, ax = plt.subplots(figsize=(6.4, max(3.0, 0.24 * len(df) + 1.2)))
    colors = np.where(df['membership_class'].str.contains('Elderly'), '#C84E4E', np.where(df['membership_class'].str.contains('Young'), '#3B74AF', '#777777'))
    ax.scatter(df['minus_log10_p'], np.arange(len(df)), s=np.clip(df['selected_n'] * 22, 30, 240), c=colors, edgecolor='white', linewidth=0.85)
    ax.set_yticks(np.arange(len(df))); ax.set_yticklabels(df['label'], fontsize=12)
    ax.set_xlabel('-log10 nominal enrichment P')
    ax.set_ylabel('CPDB communication category')
    ax.grid(axis='x', color='#E8E8E8', lw=1.05)
    ax.spines[['top', 'right']].set_visible(False)
    plt.tight_layout()
    save_fig(fig, outfile)
    df.to_csv(TAB_DIR / f'{outfile}_source.tsv', sep='\t', index=False)
    plt.show()
    return fig

plot_grouped_category_summary(grouped_category_summary, 'Figure9E_grouped_LRI_CPDB_category_summary')
if 'micro_group_lr' in globals() and micro_group_lr is not None and not micro_group_lr.empty:
    micro_category_summary = summarize_grouped_categories(micro_membership, micro_group_lr, 'S9_microglia_involved_LRI')
    plot_grouped_category_summary(micro_category_summary, 'Figure9E_microglia_LRI_CPDB_category_summary')
else:
    print('No microglia-involved category summary to plot')



def plot_grouped_lri_functional_enrichment(enrich_df, outfile, top_n=18):
    if enrich_df is None or enrich_df.empty or 'term' not in enrich_df.columns:
        print(f'No grouped GO/Reactome enrichment for {outfile}')
        return None
    df = enrich_df.copy()
    p_col = 'adjusted_p' if 'adjusted_p' in df.columns else 'enrichment_p'
    df[p_col] = pd.to_numeric(df.get(p_col, np.nan), errors='coerce')
    df = df.dropna(subset=[p_col]).copy()
    if df.empty:
        print(f'No valid grouped adjusted enrichment P values for {outfile}')
        return None
    if 'membership_class' not in df.columns:
        df['membership_class'] = df['gene_set'].astype(str).str.replace(r'_(ligands|receptors)$', '', regex=True)
    df['minus_log10_adjusted_p'] = -np.log10(df[p_col].clip(lower=1e-300))
    df['minus_log10_adjusted_p'] = df['minus_log10_adjusted_p']
    df['overlap_n'] = pd.to_numeric(df.get('overlap_n', np.nan), errors='coerce').fillna(1)
    df = (df.sort_values([p_col, 'overlap_n'], ascending=[True, False])
            .groupby('membership_class', group_keys=False)
            .head(max(2, top_n // 4))
            .sort_values(p_col)
            .head(top_n)
            .copy())
    df['label'] = df['term'].astype(str).str.replace(r' \(GO:\d+\)$', '', regex=True).str.replace(r' R-HSA-\d+$', '', regex=True).str.slice(0, 72)
    def _class_color(label):
        text = str(label)
        if 'Elderly' in text:
            return '#C84E4E'
        if 'Young adult' in text:
            return '#3B74AF'
        if 'Adult' in text:
            return '#D39C32'
        return '#777777'
    colors = df['membership_class'].map(_class_color)
    fig, ax = plt.subplots(figsize=(6.8, max(3.0, 0.30 * len(df) + 1.1)))
    y = np.arange(len(df))[::-1]
    ax.scatter(df['minus_log10_adjusted_p'], y,
               s=np.clip(df['overlap_n'] * 26, 35, 240),
               c=colors, edgecolor='white', linewidth=0.85, alpha=0.9)
    ax.set_yticks(y)
    ax.set_yticklabels(df['label'], fontsize=12)
    ax.set_xlabel('-log10 adjusted enrichment P')
    ax.set_ylabel('GO / Reactome term')
    ax.grid(axis='x', color='#E8E8E8', lw=1.05)
    ax.spines[['top', 'right']].set_visible(False)
    handles = [mpatches.Patch(facecolor='#3B74AF', label='Young adult-linked'),
               mpatches.Patch(facecolor='#D39C32', label='Adult-linked'),
               mpatches.Patch(facecolor='#C84E4E', label='Elderly-linked'),
               mpatches.Patch(facecolor='#777777', label='Shared/other')]
    ax.legend(handles=handles, frameon=False, fontsize=12, loc='lower right')
    plt.tight_layout()
    save_fig(fig, outfile)
    df.to_csv(TAB_DIR / f'{outfile}_source.tsv', sep='\t', index=False)
    plt.show()
    return fig


plot_grouped_lri_functional_enrichment(globals().get('grouped_lri_functional_enrichment', pd.DataFrame()), 'Figure9F_grouped_LRI_GO_Reactome_enrichment')
plot_grouped_lri_functional_enrichment(globals().get('micro_lri_functional_enrichment', pd.DataFrame()), 'Figure9F_microglia_LRI_GO_Reactome_enrichment')


In [ ]:
def compact_summary(df, layer, entity_col):
    if df.empty: return pd.DataFrame()
    out = df.copy(); out['layer'] = layer; out['entity'] = out[entity_col].astype(str)
    keep = ['layer','entity','metric','aging_direction','spearman_rho','spearman_p','n_donors_modeled','n_nonzero_donors','donor_frac','ordered_curve_pattern','residual_spearman_rho','residual_spearman_p','beta_10y','p_age','discovery_selected','interpretation_tier','recommended_location']
    return out[[c for c in keep if c in out.columns]]
summary = pd.concat([
    compact_summary(broad_models, 'broad_pair', 'broad_pair'),
    compact_summary(fine_models, 'fine_pair', 'cellpair'),
    compact_summary(class_models, 'classification', 'classification'),
    compact_summary(lr_models, 'LR_driver', 'cellpair_lr'),
    compact_summary(role_models, 'network_role', 'role_entity'),
], ignore_index=True)
summary.to_csv(TAB_DIR / 'S10_evidence_summary.tsv', sep='\t', index=False)
print('Evidence summary rows:', len(summary)); display(summary.sort_values(['discovery_selected','spearman_p'], ascending=[False, True]).head(40))
print('Figures:', FIG_DIR); print('Tables:', TAB_DIR)

